In [1]:
import numpy as np
import pandas
import pm4py
from matplotlib import pyplot as plt
from sklearn.mixture import GaussianMixture
import scipy.stats as stats
import ot
import os
from tqdm import tqdm
import collections
import matplotlib.dates as md
import importlib
import pickle
import random
import math
import CRPS.CRPS as pscore
import datetime

import multiprocessing as mp
mp.set_start_method('spawn')


np.seterr(all='warn', over='ignore')
pandas.set_option('display.max_columns', None)
#pandas.set_option('display.max_rows', None)


import sys
sys.path.append('../../TaskExecutionTimeMining/')
from drbart_parser import *
from event_log_transformer import *

#sys.path.append('../../Evaluation')
sys.path.append('../../Evaluation/')
from conduct_evaluation import ConductEvaluation
from normal_evaluation.drbart_evaluation import *

get_pscores = lambda likelihoods : [pscore(likelihoods[1][i], likelihoods[2][k][3]).compute()[0] for i, k in enumerate(list(likelihoods[0].keys()))]

In [2]:
#model_name = 'bpic_2017_all_2'
model_name = 'bpic_2019'

n_processes = 128
batch_size = 25

log_name = 'test'

with open('../transformed_event_logs/BPIC_19_'+log_name+'.pickle', 'rb') as f:
    test_event_log = pickle.load(f)

test_event_log['case:concept:name'] = test_event_log['case:concept:name'].astype(str)
known_resources = ['NONE', 'batch_00', 'batch_01', 'batch_02', 'batch_03', 'batch_04', 'batch_05', 'batch_06', 'batch_07', 'batch_08', 'batch_09', 'batch_10', 'batch_11', 'batch_12', 'batch_13', 'batch_14', 'batch_15', 'batch_16', 'batch_17', 'batch_18', 'batch_19', 'user_000', 'user_001', 'user_002', 'user_003', 'user_004', 'user_005', 'user_006', 'user_007', 'user_008', 'user_009', 'user_010', 'user_011', 'user_012', 'user_013', 'user_014', 'user_015', 'user_016', 'user_017', 'user_018', 'user_019', 'user_020', 'user_021', 'user_022', 'user_023', 'user_024', 'user_025', 'user_026', 'user_027', 'user_028', 'user_029', 'user_030', 'user_031', 'user_032', 'user_033', 'user_034', 'user_035', 'user_036', 'user_037', 'user_038', 'user_039', 'user_040', 'user_041', 'user_042', 'user_043', 'user_044', 'user_045', 'user_046', 'user_047', 'user_048', 'user_049', 'user_050', 'user_051', 'user_052', 'user_053', 'user_054', 'user_055', 'user_056', 'user_057', 'user_058', 'user_059', 'user_060', 'user_061', 'user_062', 'user_063', 'user_064', 'user_065', 'user_066', 'user_067', 'user_068', 'user_069', 'user_070', 'user_071', 'user_072', 'user_073', 'user_074', 'user_075', 'user_076', 'user_077', 'user_078', 'user_079', 'user_080', 'user_081', 'user_082', 'user_083', 'user_084', 'user_085', 'user_086', 'user_087', 'user_088', 'user_089', 'user_090', 'user_091', 'user_092', 'user_093', 'user_094', 'user_095', 'user_096', 'user_097', 'user_098', 'user_099', 'user_100', 'user_101', 'user_102', 'user_103', 'user_104', 'user_105', 'user_106', 'user_107', 'user_108', 'user_109', 'user_110', 'user_111', 'user_112', 'user_113', 'user_114', 'user_115', 'user_116', 'user_117', 'user_118', 'user_119', 'user_120', 'user_121', 'user_122', 'user_123', 'user_124', 'user_125', 'user_126', 'user_127', 'user_128', 'user_129', 'user_130', 'user_131', 'user_132', 'user_133', 'user_134', 'user_135', 'user_136', 'user_137', 'user_138', 'user_139', 'user_140', 'user_141', 'user_142', 'user_143', 'user_144', 'user_145', 'user_146', 'user_147', 'user_148', 'user_149', 'user_150', 'user_151', 'user_152', 'user_153', 'user_154', 'user_155', 'user_156', 'user_157', 'user_158', 'user_159', 'user_160', 'user_161', 'user_162', 'user_163', 'user_164', 'user_165', 'user_166', 'user_167', 'user_168', 'user_169', 'user_170', 'user_171', 'user_172', 'user_173', 'user_174', 'user_175', 'user_176', 'user_177', 'user_178', 'user_179', 'user_180', 'user_181', 'user_182', 'user_183', 'user_184', 'user_185', 'user_186', 'user_187', 'user_188', 'user_189', 'user_190', 'user_191', 'user_192', 'user_193', 'user_194', 'user_195', 'user_196', 'user_197', 'user_198', 'user_199', 'user_200', 'user_201', 'user_202', 'user_203', 'user_204', 'user_205', 'user_206', 'user_207', 'user_208', 'user_209', 'user_210', 'user_211', 'user_212', 'user_213', 'user_214', 'user_215', 'user_216', 'user_217', 'user_218', 'user_219', 'user_220', 'user_221', 'user_222', 'user_223', 'user_224', 'user_225', 'user_226', 'user_227', 'user_228', 'user_229', 'user_230', 'user_231', 'user_232', 'user_233', 'user_234', 'user_235', 'user_236', 'user_237', 'user_238', 'user_239', 'user_240', 'user_241', 'user_242', 'user_243', 'user_244', 'user_245', 'user_246', 'user_247', 'user_248', 'user_249', 'user_250', 'user_251', 'user_252', 'user_253', 'user_254', 'user_255', 'user_256', 'user_257', 'user_258', 'user_259', 'user_260', 'user_261', 'user_262', 'user_263', 'user_264', 'user_265', 'user_266', 'user_267', 'user_268', 'user_269', 'user_270', 'user_271', 'user_272', 'user_273', 'user_274', 'user_275', 'user_277', 'user_278', 'user_279', 'user_280', 'user_281', 'user_282', 'user_283', 'user_284', 'user_285', 'user_286', 'user_287', 'user_288', 'user_289', 'user_290', 'user_291', 'user_292', 'user_293', 'user_294', 'user_295', 'user_296', 'user_297', 'user_298', 'user_299', 'user_300', 'user_301', 'user_302', 'user_303', 'user_304', 'user_305', 'user_306', 'user_307', 'user_308', 'user_309', 'user_310', 'user_311', 'user_312', 'user_313', 'user_314', 'user_315', 'user_316', 'user_317', 'user_318', 'user_319', 'user_320', 'user_321', 'user_322', 'user_323', 'user_324', 'user_325', 'user_326', 'user_327', 'user_328', 'user_329', 'user_330', 'user_331', 'user_332', 'user_333', 'user_334', 'user_335', 'user_336', 'user_337', 'user_338', 'user_339', 'user_340', 'user_341', 'user_342', 'user_343', 'user_344', 'user_345', 'user_346', 'user_347', 'user_348', 'user_349', 'user_350', 'user_351', 'user_352', 'user_353', 'user_354', 'user_355', 'user_356', 'user_357', 'user_358', 'user_359', 'user_360', 'user_361', 'user_362', 'user_363', 'user_364', 'user_365', 'user_366', 'user_367', 'user_368', 'user_369', 'user_370', 'user_371', 'user_372', 'user_373', 'user_374', 'user_375', 'user_376', 'user_377', 'user_378', 'user_379', 'user_380', 'user_381', 'user_382', 'user_383', 'user_384', 'user_385', 'user_386', 'user_387', 'user_388', 'user_389', 'user_390', 'user_391', 'user_392', 'user_393', 'user_394', 'user_396', 'user_397', 'user_398', 'user_399', 'user_400', 'user_401', 'user_402', 'user_403', 'user_404', 'user_405', 'user_406', 'user_407', 'user_409', 'user_410', 'user_411', 'user_412', 'user_413', 'user_414', 'user_415', 'user_416', 'user_417', 'user_418', 'user_419', 'user_420', 'user_421', 'user_423', 'user_424', 'user_425', 'user_427', 'user_428', 'user_429', 'user_430', 'user_431', 'user_432', 'user_433', 'user_434', 'user_435', 'user_436', 'user_437', 'user_438', 'user_439', 'user_440', 'user_441', 'user_442', 'user_444', 'user_445', 'user_446', 'user_447', 'user_448', 'user_449', 'user_450', 'user_451', 'user_452', 'user_453', 'user_454', 'user_455', 'user_456', 'user_457', 'user_458', 'user_459', 'user_460', 'user_461', 'user_462', 'user_463', 'user_464', 'user_465', 'user_466', 'user_467', 'user_468', 'user_469', 'user_470', 'user_471', 'user_472', 'user_473', 'user_474', 'user_475', 'user_476', 'user_477', 'user_478', 'user_479', 'user_480', 'user_481', 'user_482', 'user_483', 'user_484', 'user_485', 'user_486', 'user_487', 'user_488', 'user_489', 'user_490', 'user_491', 'user_492', 'user_493', 'user_494', 'user_495', 'user_496', 'user_497', 'user_498', 'user_499', 'user_500', 'user_501', 'user_502', 'user_503', 'user_504', 'user_505', 'user_506', 'user_507', 'user_508', 'user_509', 'user_510', 'user_511', 'user_512', 'user_513', 'user_514', 'user_515', 'user_516', 'user_517', 'user_518', 'user_519', 'user_520', 'user_521', 'user_522', 'user_523', 'user_524', 'user_525', 'user_526', 'user_527', 'user_528', 'user_529', 'user_530', 'user_531', 'user_532', 'user_533', 'user_534', 'user_535', 'user_536', 'user_537', 'user_538', 'user_539', 'user_540', 'user_541', 'user_542', 'user_543', 'user_544', 'user_545', 'user_546', 'user_547', 'user_548', 'user_549', 'user_550', 'user_551', 'user_552', 'user_553', 'user_554', 'user_555', 'user_556', 'user_557', 'user_558', 'user_559', 'user_560', 'user_561', 'user_562', 'user_563', 'user_564', 'user_565', 'user_566', 'user_567', 'user_568', 'user_569', 'user_570', 'user_571', 'user_572', 'user_573', 'user_574', 'user_575', 'user_576', 'user_577', 'user_578', 'user_579', 'user_580', 'user_581', 'user_582', 'user_583', 'user_584', 'user_585', 'user_586', 'user_587', 'user_588', 'user_589', 'user_590', 'user_591', 'user_592', 'user_593', 'user_594', 'user_595', 'user_597', 'user_598', 'user_599', 'user_601', 'user_602', 'user_603', 'user_604', 'user_605', 'user_606']
known_activities = ['Block Purchase Order Item', 'Cancel Goods Receipt', 'Cancel Invoice Receipt', 'Cancel Subsequent Invoice', 'Change Approval for Purchase Order',
'Change Currency', 'Change Delivery Indicator', 'Change Final Invoice Indicator', 'Change Price', 'Change Quantity', 'Change Rejection Indicator',
'Change Storage Location', 'Change payment term', 'Clear Invoice', 'Create Purchase Order Item', 'Create Purchase Requisition Item',
'Delete Purchase Order Item', 'Reactivate Purchase Order Item', 'Receive Order Confirmation', 'Record Goods Receipt', 'Record Invoice Receipt',
'Record Service Entry Sheet', 'Record Subsequent Invoice', 'Release Purchase Order', 'Release Purchase Requisition', 'Remove Payment Block',
'SRM: Awaiting Approval', 'SRM: Change was Transmitted', 'SRM: Complete', 'SRM: Created', 'SRM: Deleted', 'SRM: Document Completed', 'SRM: Held',
'SRM: In Transfer to Execution Syst.', 'SRM: Incomplete', 'SRM: Ordered', 'SRM: Transaction Completed', 'SRM: Transfer Failed (E.Sys.)',
'Set Payment Block', 'Update Order Confirmation', 'Vendor creates debit memo', 'Vendor creates invoice']

ii1 = ['intercase_n_1__Block Purchase Order Item', 'intercase_n_1__Cancel Goods Receipt', 'intercase_n_1__Cancel Invoice Receipt', 'intercase_n_1__Cancel Subsequent Invoice', 'intercase_n_1__Change Approval for Purchase Order', 'intercase_n_1__Change Currency', 'intercase_n_1__Change Delivery Indicator', 'intercase_n_1__Change Final Invoice Indicator', 'intercase_n_1__Change Price', 'intercase_n_1__Change Quantity', 'intercase_n_1__Change Rejection Indicator', 'intercase_n_1__Change Storage Location', 'intercase_n_1__Change payment term', 'intercase_n_1__Clear Invoice', 'intercase_n_1__Create Purchase Order Item', 'intercase_n_1__Create Purchase Requisition Item', 'intercase_n_1__Delete Purchase Order Item', 'intercase_n_1__Reactivate Purchase Order Item', 'intercase_n_1__Receive Order Confirmation', 'intercase_n_1__Record Goods Receipt', 'intercase_n_1__Record Invoice Receipt', 'intercase_n_1__Record Service Entry Sheet', 'intercase_n_1__Record Subsequent Invoice', 'intercase_n_1__Release Purchase Order', 'intercase_n_1__Release Purchase Requisition', 'intercase_n_1__Remove Payment Block', 'intercase_n_1__SRM: Awaiting Approval', 'intercase_n_1__SRM: Change was Transmitted', 'intercase_n_1__SRM: Complete', 'intercase_n_1__SRM: Created', 'intercase_n_1__SRM: Deleted', 'intercase_n_1__SRM: Document Completed', 'intercase_n_1__SRM: Held', 'intercase_n_1__SRM: In Transfer to Execution Syst.', 'intercase_n_1__SRM: Incomplete', 'intercase_n_1__SRM: Ordered', 'intercase_n_1__SRM: Transaction Completed', 'intercase_n_1__SRM: Transfer Failed (E.Sys.)', 'intercase_n_1__Set Payment Block', 'intercase_n_1__Update Order Confirmation', 'intercase_n_1__Vendor creates debit memo', 'intercase_n_1__Vendor creates invoice']
ii3 = ['intercase_n_3__Block Purchase Order Item_Change Quantity_Reactivate Purchase Order Item', 'intercase_n_3__Block Purchase Order Item_Delete Purchase Order Item_Reactivate Purchase Order Item', 'intercase_n_3__Block Purchase Order Item_Reactivate Purchase Order Item_Block Purchase Order Item', 'intercase_n_3__Block Purchase Order Item_Reactivate Purchase Order Item_Change Delivery Indicator', 'intercase_n_3__Block Purchase Order Item_Reactivate Purchase Order Item_Change Quantity', 'intercase_n_3__Block Purchase Order Item_Reactivate Purchase Order Item_Record Goods Receipt', 'intercase_n_3__Block Purchase Order Item_Reactivate Purchase Order Item_Record Invoice Receipt', 'intercase_n_3__Block Purchase Order Item_Reactivate Purchase Order Item_Vendor creates invoice', 'intercase_n_3__Block Purchase Order Item_Vendor creates debit memo_Reactivate Purchase Order Item', 'intercase_n_3__Block Purchase Order Item_Vendor creates invoice_Delete Purchase Order Item', 'intercase_n_3__Block Purchase Order Item_Vendor creates invoice_Reactivate Purchase Order Item', 'intercase_n_3__Block Purchase Order Item_Vendor creates invoice_Vendor creates debit memo', 'intercase_n_3__Cancel Goods Receipt_Block Purchase Order Item_Reactivate Purchase Order Item', 'intercase_n_3__Cancel Goods Receipt_Cancel Goods Receipt_Cancel Goods Receipt', 'intercase_n_3__Cancel Goods Receipt_Cancel Goods Receipt_Change Delivery Indicator', 'intercase_n_3__Cancel Goods Receipt_Cancel Goods Receipt_Record Goods Receipt', 'intercase_n_3__Cancel Goods Receipt_Cancel Goods Receipt_Record Invoice Receipt', 'intercase_n_3__Cancel Goods Receipt_Cancel Goods Receipt_Record Service Entry Sheet', 'intercase_n_3__Cancel Goods Receipt_Cancel Goods Receipt_Vendor creates debit memo', 'intercase_n_3__Cancel Goods Receipt_Cancel Goods Receipt_Vendor creates invoice', 'intercase_n_3__Cancel Goods Receipt_Cancel Invoice Receipt_Cancel Invoice Receipt', 'intercase_n_3__Cancel Goods Receipt_Cancel Invoice Receipt_Change Delivery Indicator', 'intercase_n_3__Cancel Goods Receipt_Cancel Invoice Receipt_Clear Invoice', 'intercase_n_3__Cancel Goods Receipt_Cancel Invoice Receipt_Remove Payment Block', 'intercase_n_3__Cancel Goods Receipt_Change Delivery Indicator_Cancel Invoice Receipt', 'intercase_n_3__Cancel Goods Receipt_Change Delivery Indicator_Change Delivery Indicator', 'intercase_n_3__Cancel Goods Receipt_Change Delivery Indicator_Clear Invoice', 'intercase_n_3__Cancel Goods Receipt_Change Delivery Indicator_Record Goods Receipt', 'intercase_n_3__Cancel Goods Receipt_Change Delivery Indicator_Record Invoice Receipt', 'intercase_n_3__Cancel Goods Receipt_Change Delivery Indicator_Remove Payment Block', 'intercase_n_3__Cancel Goods Receipt_Change Delivery Indicator_Vendor creates debit memo', 'intercase_n_3__Cancel Goods Receipt_Change Delivery Indicator_Vendor creates invoice', 'intercase_n_3__Cancel Goods Receipt_Change Price_Cancel Goods Receipt', 'intercase_n_3__Cancel Goods Receipt_Change Price_Change Price', 'intercase_n_3__Cancel Goods Receipt_Change Price_Change Quantity', 'intercase_n_3__Cancel Goods Receipt_Change Price_Record Goods Receipt', 'intercase_n_3__Cancel Goods Receipt_Change Price_Vendor creates debit memo', 'intercase_n_3__Cancel Goods Receipt_Change Quantity_Cancel Goods Receipt', 'intercase_n_3__Cancel Goods Receipt_Change Quantity_Change Delivery Indicator', 'intercase_n_3__Cancel Goods Receipt_Change Quantity_Change Price', 'intercase_n_3__Cancel Goods Receipt_Change Quantity_Change Quantity', 'intercase_n_3__Cancel Goods Receipt_Change Quantity_Clear Invoice', 'intercase_n_3__Cancel Goods Receipt_Change Quantity_Record Goods Receipt', 'intercase_n_3__Cancel Goods Receipt_Change Quantity_Record Invoice Receipt', 'intercase_n_3__Cancel Goods Receipt_Change Quantity_Remove Payment Block', 'intercase_n_3__Cancel Goods Receipt_Change Quantity_Vendor creates debit memo', 'intercase_n_3__Cancel Goods Receipt_Change Quantity_Vendor creates invoice', 'intercase_n_3__Cancel Goods Receipt_Change Storage Location_Record Goods Receipt', 'intercase_n_3__Cancel Goods Receipt_Clear Invoice_Record Goods Receipt', 'intercase_n_3__Cancel Goods Receipt_Clear Invoice_Record Invoice Receipt', 'intercase_n_3__Cancel Goods Receipt_Clear Invoice_Vendor creates debit memo', 'intercase_n_3__Cancel Goods Receipt_Clear Invoice_Vendor creates invoice', 'intercase_n_3__Cancel Goods Receipt_Delete Purchase Order Item_Reactivate Purchase Order Item', 'intercase_n_3__Cancel Goods Receipt_Record Goods Receipt_Cancel Goods Receipt', 'intercase_n_3__Cancel Goods Receipt_Record Goods Receipt_Change Approval for Purchase Order', 'intercase_n_3__Cancel Goods Receipt_Record Goods Receipt_Change Delivery Indicator', 'intercase_n_3__Cancel Goods Receipt_Record Goods Receipt_Change Price', 'intercase_n_3__Cancel Goods Receipt_Record Goods Receipt_Change Quantity', 'intercase_n_3__Cancel Goods Receipt_Record Goods Receipt_Clear Invoice', 'intercase_n_3__Cancel Goods Receipt_Record Goods Receipt_Record Goods Receipt', 'intercase_n_3__Cancel Goods Receipt_Record Goods Receipt_Record Invoice Receipt', 'intercase_n_3__Cancel Goods Receipt_Record Goods Receipt_Record Service Entry Sheet', 'intercase_n_3__Cancel Goods Receipt_Record Goods Receipt_Remove Payment Block', 'intercase_n_3__Cancel Goods Receipt_Record Goods Receipt_Vendor creates debit memo', 'intercase_n_3__Cancel Goods Receipt_Record Goods Receipt_Vendor creates invoice', 'intercase_n_3__Cancel Goods Receipt_Record Invoice Receipt_Cancel Invoice Receipt', 'intercase_n_3__Cancel Goods Receipt_Record Invoice Receipt_Change Delivery Indicator', 'intercase_n_3__Cancel Goods Receipt_Record Invoice Receipt_Clear Invoice', 'intercase_n_3__Cancel Goods Receipt_Record Invoice Receipt_Record Goods Receipt', 'intercase_n_3__Cancel Goods Receipt_Record Invoice Receipt_Record Invoice Receipt', 'intercase_n_3__Cancel Goods Receipt_Record Invoice Receipt_Record Service Entry Sheet', 'intercase_n_3__Cancel Goods Receipt_Record Invoice Receipt_Remove Payment Block', 'intercase_n_3__Cancel Goods Receipt_Record Invoice Receipt_Vendor creates invoice', 'intercase_n_3__Cancel Goods Receipt_Record Service Entry Sheet_Record Goods Receipt', 'intercase_n_3__Cancel Goods Receipt_Record Service Entry Sheet_Record Service Entry Sheet', 'intercase_n_3__Cancel Goods Receipt_Record Service Entry Sheet_Vendor creates invoice', 'intercase_n_3__Cancel Goods Receipt_Release Purchase Order_Change Approval for Purchase Order', 'intercase_n_3__Cancel Goods Receipt_Remove Payment Block_Change Quantity', 'intercase_n_3__Cancel Goods Receipt_Remove Payment Block_Record Goods Receipt', 'intercase_n_3__Cancel Goods Receipt_Remove Payment Block_Vendor creates debit memo', 'intercase_n_3__Cancel Goods Receipt_Remove Payment Block_Vendor creates invoice', 'intercase_n_3__Cancel Goods Receipt_SRM: Complete_SRM: Awaiting Approval', 'intercase_n_3__Cancel Goods Receipt_SRM: Created_SRM: Complete', 'intercase_n_3__Cancel Goods Receipt_SRM: Ordered_SRM: In Transfer to Execution Syst.', 'intercase_n_3__Cancel Goods Receipt_Vendor creates debit memo_Cancel Invoice Receipt', 'intercase_n_3__Cancel Goods Receipt_Vendor creates debit memo_Change Delivery Indicator', 'intercase_n_3__Cancel Goods Receipt_Vendor creates debit memo_Change Price', 'intercase_n_3__Cancel Goods Receipt_Vendor creates debit memo_Change Quantity', 'intercase_n_3__Cancel Goods Receipt_Vendor creates debit memo_Clear Invoice', 'intercase_n_3__Cancel Goods Receipt_Vendor creates debit memo_Record Invoice Receipt', 'intercase_n_3__Cancel Goods Receipt_Vendor creates debit memo_Vendor creates debit memo', 'intercase_n_3__Cancel Goods Receipt_Vendor creates debit memo_Vendor creates invoice', 'intercase_n_3__Cancel Goods Receipt_Vendor creates invoice_Cancel Goods Receipt', 'intercase_n_3__Cancel Goods Receipt_Vendor creates invoice_Change Delivery Indicator', 'intercase_n_3__Cancel Goods Receipt_Vendor creates invoice_Change Price', 'intercase_n_3__Cancel Goods Receipt_Vendor creates invoice_Change Quantity', 'intercase_n_3__Cancel Goods Receipt_Vendor creates invoice_Clear Invoice', 'intercase_n_3__Cancel Goods Receipt_Vendor creates invoice_Record Goods Receipt', 'intercase_n_3__Cancel Goods Receipt_Vendor creates invoice_Record Invoice Receipt', 'intercase_n_3__Cancel Goods Receipt_Vendor creates invoice_Record Service Entry Sheet', 'intercase_n_3__Cancel Goods Receipt_Vendor creates invoice_Vendor creates debit memo', 'intercase_n_3__Cancel Goods Receipt_Vendor creates invoice_Vendor creates invoice', 'intercase_n_3__Cancel Invoice Receipt_Cancel Goods Receipt_Cancel Goods Receipt', 'intercase_n_3__Cancel Invoice Receipt_Cancel Goods Receipt_Change Delivery Indicator', 'intercase_n_3__Cancel Invoice Receipt_Cancel Goods Receipt_Record Goods Receipt', 'intercase_n_3__Cancel Invoice Receipt_Cancel Goods Receipt_Remove Payment Block', 'intercase_n_3__Cancel Invoice Receipt_Cancel Invoice Receipt_Cancel Goods Receipt', 'intercase_n_3__Cancel Invoice Receipt_Cancel Invoice Receipt_Cancel Invoice Receipt', 'intercase_n_3__Cancel Invoice Receipt_Cancel Invoice Receipt_Change Price', 'intercase_n_3__Cancel Invoice Receipt_Cancel Invoice Receipt_Clear Invoice', 'intercase_n_3__Cancel Invoice Receipt_Cancel Invoice Receipt_Record Goods Receipt', 'intercase_n_3__Cancel Invoice Receipt_Cancel Invoice Receipt_Record Invoice Receipt', 'intercase_n_3__Cancel Invoice Receipt_Cancel Invoice Receipt_Remove Payment Block', 'intercase_n_3__Cancel Invoice Receipt_Cancel Invoice Receipt_Vendor creates invoice', 'intercase_n_3__Cancel Invoice Receipt_Cancel Subsequent Invoice_Clear Invoice', 'intercase_n_3__Cancel Invoice Receipt_Cancel Subsequent Invoice_Remove Payment Block', 'intercase_n_3__Cancel Invoice Receipt_Change Delivery Indicator_Change Delivery Indicator', 'intercase_n_3__Cancel Invoice Receipt_Change Delivery Indicator_Clear Invoice', 'intercase_n_3__Cancel Invoice Receipt_Change Delivery Indicator_Record Invoice Receipt', 'intercase_n_3__Cancel Invoice Receipt_Change Price_Change Price', 'intercase_n_3__Cancel Invoice Receipt_Change Price_Record Goods Receipt', 'intercase_n_3__Cancel Invoice Receipt_Change Price_Remove Payment Block', 'intercase_n_3__Cancel Invoice Receipt_Change Quantity_Record Invoice Receipt', 'intercase_n_3__Cancel Invoice Receipt_Clear Invoice_Block Purchase Order Item', 'intercase_n_3__Cancel Invoice Receipt_Clear Invoice_Cancel Goods Receipt', 'intercase_n_3__Cancel Invoice Receipt_Clear Invoice_Cancel Invoice Receipt', 'intercase_n_3__Cancel Invoice Receipt_Clear Invoice_Cancel Subsequent Invoice', 'intercase_n_3__Cancel Invoice Receipt_Clear Invoice_Change Delivery Indicator', 'intercase_n_3__Cancel Invoice Receipt_Clear Invoice_Change Price', 'intercase_n_3__Cancel Invoice Receipt_Clear Invoice_Change Quantity', 'intercase_n_3__Cancel Invoice Receipt_Clear Invoice_Clear Invoice', 'intercase_n_3__Cancel Invoice Receipt_Clear Invoice_Record Goods Receipt', 'intercase_n_3__Cancel Invoice Receipt_Clear Invoice_Record Invoice Receipt', 'intercase_n_3__Cancel Invoice Receipt_Clear Invoice_Record Subsequent Invoice', 'intercase_n_3__Cancel Invoice Receipt_Clear Invoice_Remove Payment Block', 'intercase_n_3__Cancel Invoice Receipt_Clear Invoice_SRM: Ordered', 'intercase_n_3__Cancel Invoice Receipt_Clear Invoice_Vendor creates debit memo', 'intercase_n_3__Cancel Invoice Receipt_Clear Invoice_Vendor creates invoice', 'intercase_n_3__Cancel Invoice Receipt_Delete Purchase Order Item_Remove Payment Block', 'intercase_n_3__Cancel Invoice Receipt_Record Goods Receipt_Cancel Goods Receipt', 'intercase_n_3__Cancel Invoice Receipt_Record Goods Receipt_Record Invoice Receipt', 'intercase_n_3__Cancel Invoice Receipt_Record Goods Receipt_Record Service Entry Sheet', 'intercase_n_3__Cancel Invoice Receipt_Record Invoice Receipt_Cancel Invoice Receipt', 'intercase_n_3__Cancel Invoice Receipt_Record Invoice Receipt_Cancel Subsequent Invoice', 'intercase_n_3__Cancel Invoice Receipt_Record Invoice Receipt_Change Price', 'intercase_n_3__Cancel Invoice Receipt_Record Invoice Receipt_Clear Invoice', 'intercase_n_3__Cancel Invoice Receipt_Record Invoice Receipt_Record Goods Receipt', 'intercase_n_3__Cancel Invoice Receipt_Record Invoice Receipt_Record Invoice Receipt', 'intercase_n_3__Cancel Invoice Receipt_Record Invoice Receipt_Remove Payment Block', 'intercase_n_3__Cancel Invoice Receipt_Record Invoice Receipt_Set Payment Block', 'intercase_n_3__Cancel Invoice Receipt_Record Invoice Receipt_Vendor creates debit memo', 'intercase_n_3__Cancel Invoice Receipt_Record Invoice Receipt_Vendor creates invoice', 'intercase_n_3__Cancel Invoice Receipt_Record Service Entry Sheet_Record Goods Receipt', 'intercase_n_3__Cancel Invoice Receipt_Record Subsequent Invoice_Cancel Subsequent Invoice', 'intercase_n_3__Cancel Invoice Receipt_Record Subsequent Invoice_Clear Invoice', 'intercase_n_3__Cancel Invoice Receipt_Record Subsequent Invoice_Remove Payment Block', 'intercase_n_3__Cancel Invoice Receipt_Remove Payment Block_Cancel Invoice Receipt', 'intercase_n_3__Cancel Invoice Receipt_Remove Payment Block_Change Delivery Indicator', 'intercase_n_3__Cancel Invoice Receipt_Remove Payment Block_Change Quantity', 'intercase_n_3__Cancel Invoice Receipt_Remove Payment Block_Clear Invoice', 'intercase_n_3__Cancel Invoice Receipt_Remove Payment Block_Delete Purchase Order Item', 'intercase_n_3__Cancel Invoice Receipt_Remove Payment Block_Record Goods Receipt', 'intercase_n_3__Cancel Invoice Receipt_Remove Payment Block_Record Invoice Receipt', 'intercase_n_3__Cancel Invoice Receipt_Remove Payment Block_Remove Payment Block', 'intercase_n_3__Cancel Invoice Receipt_Remove Payment Block_Set Payment Block', 'intercase_n_3__Cancel Invoice Receipt_Remove Payment Block_Vendor creates debit memo', 'intercase_n_3__Cancel Invoice Receipt_Remove Payment Block_Vendor creates invoice', 'intercase_n_3__Cancel Invoice Receipt_Vendor creates debit memo_Cancel Invoice Receipt', 'intercase_n_3__Cancel Invoice Receipt_Vendor creates debit memo_Change Quantity', 'intercase_n_3__Cancel Invoice Receipt_Vendor creates debit memo_Clear Invoice', 'intercase_n_3__Cancel Invoice Receipt_Vendor creates debit memo_Record Invoice Receipt', 'intercase_n_3__Cancel Invoice Receipt_Vendor creates debit memo_Remove Payment Block', 'intercase_n_3__Cancel Invoice Receipt_Vendor creates debit memo_Vendor creates invoice', 'intercase_n_3__Cancel Invoice Receipt_Vendor creates invoice_Cancel Goods Receipt', 'intercase_n_3__Cancel Invoice Receipt_Vendor creates invoice_Cancel Subsequent Invoice', 'intercase_n_3__Cancel Invoice Receipt_Vendor creates invoice_Clear Invoice', 'intercase_n_3__Cancel Invoice Receipt_Vendor creates invoice_Record Goods Receipt', 'intercase_n_3__Cancel Invoice Receipt_Vendor creates invoice_Record Invoice Receipt', 'intercase_n_3__Cancel Invoice Receipt_Vendor creates invoice_Record Service Entry Sheet', 'intercase_n_3__Cancel Invoice Receipt_Vendor creates invoice_Remove Payment Block', 'intercase_n_3__Cancel Invoice Receipt_Vendor creates invoice_Vendor creates debit memo', 'intercase_n_3__Cancel Invoice Receipt_Vendor creates invoice_Vendor creates invoice', 'intercase_n_3__Cancel Subsequent Invoice_Cancel Invoice Receipt_Cancel Invoice Receipt', 'intercase_n_3__Cancel Subsequent Invoice_Cancel Subsequent Invoice_Record Subsequent Invoice', 'intercase_n_3__Cancel Subsequent Invoice_Cancel Subsequent Invoice_Remove Payment Block', 'intercase_n_3__Cancel Subsequent Invoice_Change Price_Remove Payment Block', 'intercase_n_3__Cancel Subsequent Invoice_Clear Invoice_Cancel Goods Receipt', 'intercase_n_3__Cancel Subsequent Invoice_Clear Invoice_Cancel Invoice Receipt', 'intercase_n_3__Cancel Subsequent Invoice_Clear Invoice_Cancel Subsequent Invoice', 'intercase_n_3__Cancel Subsequent Invoice_Clear Invoice_Change Delivery Indicator', 'intercase_n_3__Cancel Subsequent Invoice_Clear Invoice_Change Price', 'intercase_n_3__Cancel Subsequent Invoice_Clear Invoice_Clear Invoice', 'intercase_n_3__Cancel Subsequent Invoice_Clear Invoice_Record Goods Receipt', 'intercase_n_3__Cancel Subsequent Invoice_Clear Invoice_Record Invoice Receipt', 'intercase_n_3__Cancel Subsequent Invoice_Clear Invoice_Record Subsequent Invoice', 'intercase_n_3__Cancel Subsequent Invoice_Clear Invoice_Remove Payment Block', 'intercase_n_3__Cancel Subsequent Invoice_Record Goods Receipt_Record Goods Receipt', 'intercase_n_3__Cancel Subsequent Invoice_Record Goods Receipt_Record Service Entry Sheet', 'intercase_n_3__Cancel Subsequent Invoice_Record Goods Receipt_Remove Payment Block', 'intercase_n_3__Cancel Subsequent Invoice_Record Invoice Receipt_Cancel Invoice Receipt', 'intercase_n_3__Cancel Subsequent Invoice_Record Invoice Receipt_Clear Invoice', 'intercase_n_3__Cancel Subsequent Invoice_Record Invoice Receipt_Remove Payment Block', 'intercase_n_3__Cancel Subsequent Invoice_Record Subsequent Invoice_Clear Invoice', 'intercase_n_3__Cancel Subsequent Invoice_Record Subsequent Invoice_Remove Payment Block', 'intercase_n_3__Cancel Subsequent Invoice_Remove Payment Block_Clear Invoice', 'intercase_n_3__Cancel Subsequent Invoice_Remove Payment Block_Record Invoice Receipt', 'intercase_n_3__Cancel Subsequent Invoice_Remove Payment Block_Record Subsequent Invoice', 'intercase_n_3__Cancel Subsequent Invoice_Remove Payment Block_Set Payment Block', 'intercase_n_3__Cancel Subsequent Invoice_Remove Payment Block_Vendor creates debit memo', 'intercase_n_3__Cancel Subsequent Invoice_Remove Payment Block_Vendor creates invoice', 'intercase_n_3__Cancel Subsequent Invoice_Vendor creates debit memo_Clear Invoice', 'intercase_n_3__Cancel Subsequent Invoice_Vendor creates debit memo_Remove Payment Block', 'intercase_n_3__Cancel Subsequent Invoice_Vendor creates debit memo_Vendor creates invoice', 'intercase_n_3__Cancel Subsequent Invoice_Vendor creates invoice_Clear Invoice', 'intercase_n_3__Cancel Subsequent Invoice_Vendor creates invoice_Vendor creates invoice', 'intercase_n_3__Change Approval for Purchase Order', 'intercase_n_3__Change Approval for Purchase Order_Block Purchase Order Item_Reactivate Purchase Order Item', 'intercase_n_3__Change Approval for Purchase Order_Cancel Invoice Receipt_Clear Invoice', 'intercase_n_3__Change Approval for Purchase Order_Change Approval for Purchase Order', 'intercase_n_3__Change Approval for Purchase Order_Change Approval for Purchase Order_Block Purchase Order Item', 'intercase_n_3__Change Approval for Purchase Order_Change Approval for Purchase Order_Change Approval for Purchase Order', 'intercase_n_3__Change Approval for Purchase Order_Change Approval for Purchase Order_Change Price', 'intercase_n_3__Change Approval for Purchase Order_Change Approval for Purchase Order_Change Quantity', 'intercase_n_3__Change Approval for Purchase Order_Change Approval for Purchase Order_Clear Invoice', 'intercase_n_3__Change Approval for Purchase Order_Change Approval for Purchase Order_Create Purchase Order Item', 'intercase_n_3__Change Approval for Purchase Order_Change Approval for Purchase Order_Delete Purchase Order Item', 'intercase_n_3__Change Approval for Purchase Order_Change Approval for Purchase Order_Record Invoice Receipt', 'intercase_n_3__Change Approval for Purchase Order_Change Approval for Purchase Order_Release Purchase Order', 'intercase_n_3__Change Approval for Purchase Order_Change Approval for Purchase Order_Vendor creates debit memo', 'intercase_n_3__Change Approval for Purchase Order_Change Approval for Purchase Order_Vendor creates invoice', 'intercase_n_3__Change Approval for Purchase Order_Change Price_Change Approval for Purchase Order', 'intercase_n_3__Change Approval for Purchase Order_Change Price_Change Price', 'intercase_n_3__Change Approval for Purchase Order_Change Price_Change Quantity', 'intercase_n_3__Change Approval for Purchase Order_Change Price_Delete Purchase Order Item', 'intercase_n_3__Change Approval for Purchase Order_Change Price_Receive Order Confirmation', 'intercase_n_3__Change Approval for Purchase Order_Change Price_Record Goods Receipt', 'intercase_n_3__Change Approval for Purchase Order_Change Price_Record Invoice Receipt', 'intercase_n_3__Change Approval for Purchase Order_Change Price_Release Purchase Order', 'intercase_n_3__Change Approval for Purchase Order_Change Price_Vendor creates invoice', 'intercase_n_3__Change Approval for Purchase Order_Change Quantity_Change Approval for Purchase Order', 'intercase_n_3__Change Approval for Purchase Order_Change Quantity_Change Price', 'intercase_n_3__Change Approval for Purchase Order_Change Quantity_Change Quantity', 'intercase_n_3__Change Approval for Purchase Order_Change Quantity_Change Storage Location', 'intercase_n_3__Change Approval for Purchase Order_Change Quantity_Reactivate Purchase Order Item', 'intercase_n_3__Change Approval for Purchase Order_Change Quantity_Receive Order Confirmation', 'intercase_n_3__Change Approval for Purchase Order_Change Quantity_Record Goods Receipt', 'intercase_n_3__Change Approval for Purchase Order_Change Quantity_Release Purchase Order', 'intercase_n_3__Change Approval for Purchase Order_Change Quantity_Vendor creates invoice', 'intercase_n_3__Change Approval for Purchase Order_Change Storage Location_Change Price', 'intercase_n_3__Change Approval for Purchase Order_Change Storage Location_Change Storage Location', 'intercase_n_3__Change Approval for Purchase Order_Change Storage Location_Vendor creates invoice', 'intercase_n_3__Change Approval for Purchase Order_Clear Invoice_Change Approval for Purchase Order', 'intercase_n_3__Change Approval for Purchase Order_Clear Invoice_Vendor creates invoice', 'intercase_n_3__Change Approval for Purchase Order_Create Purchase Order Item', 'intercase_n_3__Change Approval for Purchase Order_Create Purchase Order Item_Cancel Goods Receipt', 'intercase_n_3__Change Approval for Purchase Order_Create Purchase Order Item_Change Approval for Purchase Order', 'intercase_n_3__Change Approval for Purchase Order_Create Purchase Order Item_Change Quantity', 'intercase_n_3__Change Approval for Purchase Order_Create Purchase Order Item_Delete Purchase Order Item', 'intercase_n_3__Change Approval for Purchase Order_Create Purchase Order Item_Record Goods Receipt', 'intercase_n_3__Change Approval for Purchase Order_Create Purchase Order Item_Record Invoice Receipt', 'intercase_n_3__Change Approval for Purchase Order_Create Purchase Order Item_Release Purchase Order', 'intercase_n_3__Change Approval for Purchase Order_Create Purchase Order Item_Vendor creates debit memo', 'intercase_n_3__Change Approval for Purchase Order_Create Purchase Order Item_Vendor creates invoice', 'intercase_n_3__Change Approval for Purchase Order_Delete Purchase Order Item_Release Purchase Order', 'intercase_n_3__Change Approval for Purchase Order_Receive Order Confirmation_Change Quantity', 'intercase_n_3__Change Approval for Purchase Order_Receive Order Confirmation_Delete Purchase Order Item', 'intercase_n_3__Change Approval for Purchase Order_Receive Order Confirmation_Record Goods Receipt', 'intercase_n_3__Change Approval for Purchase Order_Receive Order Confirmation_Vendor creates invoice', 'intercase_n_3__Change Approval for Purchase Order_Record Goods Receipt_Cancel Goods Receipt', 'intercase_n_3__Change Approval for Purchase Order_Record Goods Receipt_Change Approval for Purchase Order', 'intercase_n_3__Change Approval for Purchase Order_Record Goods Receipt_Change Quantity', 'intercase_n_3__Change Approval for Purchase Order_Record Goods Receipt_Receive Order Confirmation', 'intercase_n_3__Change Approval for Purchase Order_Record Goods Receipt_Record Invoice Receipt', 'intercase_n_3__Change Approval for Purchase Order_Record Goods Receipt_Release Purchase Order', 'intercase_n_3__Change Approval for Purchase Order_Record Goods Receipt_Remove Payment Block', 'intercase_n_3__Change Approval for Purchase Order_Record Goods Receipt_Vendor creates invoice', 'intercase_n_3__Change Approval for Purchase Order_Record Invoice Receipt_Cancel Invoice Receipt', 'intercase_n_3__Change Approval for Purchase Order_Record Invoice Receipt_Change Approval for Purchase Order', 'intercase_n_3__Change Approval for Purchase Order_Record Invoice Receipt_Change Price', 'intercase_n_3__Change Approval for Purchase Order_Record Invoice Receipt_Clear Invoice', 'intercase_n_3__Change Approval for Purchase Order_Record Invoice Receipt_Record Goods Receipt', 'intercase_n_3__Change Approval for Purchase Order_Record Invoice Receipt_Record Invoice Receipt', 'intercase_n_3__Change Approval for Purchase Order_Record Invoice Receipt_Release Purchase Order', 'intercase_n_3__Change Approval for Purchase Order_Record Invoice Receipt_Remove Payment Block', 'intercase_n_3__Change Approval for Purchase Order_Record Invoice Receipt_Vendor creates invoice', 'intercase_n_3__Change Approval for Purchase Order_Release Purchase Order', 'intercase_n_3__Change Approval for Purchase Order_Release Purchase Order_Change Approval for Purchase Order', 'intercase_n_3__Change Approval for Purchase Order_Release Purchase Order_Change Price', 'intercase_n_3__Change Approval for Purchase Order_Release Purchase Order_Change Quantity', 'intercase_n_3__Change Approval for Purchase Order_Release Purchase Order_Clear Invoice', 'intercase_n_3__Change Approval for Purchase Order_Release Purchase Order_Create Purchase Order Item', 'intercase_n_3__Change Approval for Purchase Order_Release Purchase Order_Delete Purchase Order Item', 'intercase_n_3__Change Approval for Purchase Order_Release Purchase Order_Vendor creates invoice', 'intercase_n_3__Change Approval for Purchase Order_Remove Payment Block_Release Purchase Order', 'intercase_n_3__Change Approval for Purchase Order_Vendor creates debit memo_Vendor creates invoice', 'intercase_n_3__Change Approval for Purchase Order_Vendor creates invoice', 'intercase_n_3__Change Approval for Purchase Order_Vendor creates invoice_Block Purchase Order Item', 'intercase_n_3__Change Approval for Purchase Order_Vendor creates invoice_Change Approval for Purchase Order', 'intercase_n_3__Change Approval for Purchase Order_Vendor creates invoice_Change Price', 'intercase_n_3__Change Approval for Purchase Order_Vendor creates invoice_Change Quantity', 'intercase_n_3__Change Approval for Purchase Order_Vendor creates invoice_Change Storage Location', 'intercase_n_3__Change Approval for Purchase Order_Vendor creates invoice_Create Purchase Order Item', 'intercase_n_3__Change Approval for Purchase Order_Vendor creates invoice_Record Goods Receipt', 'intercase_n_3__Change Approval for Purchase Order_Vendor creates invoice_Record Invoice Receipt', 'intercase_n_3__Change Approval for Purchase Order_Vendor creates invoice_Release Purchase Order', 'intercase_n_3__Change Approval for Purchase Order_Vendor creates invoice_Vendor creates debit memo', 'intercase_n_3__Change Approval for Purchase Order_Vendor creates invoice_Vendor creates invoice', 'intercase_n_3__Change Currency', 'intercase_n_3__Change Currency_Change Price_Change Approval for Purchase Order', 'intercase_n_3__Change Currency_Change Price_Record Goods Receipt', 'intercase_n_3__Change Currency_Change Quantity_Change Quantity', 'intercase_n_3__Change Currency_Change payment term_Change Price', 'intercase_n_3__Change Currency_Create Purchase Order Item', 'intercase_n_3__Change Currency_Create Purchase Order Item_Change Quantity', 'intercase_n_3__Change Currency_Create Purchase Order Item_Vendor creates invoice', 'intercase_n_3__Change Currency_Vendor creates debit memo_Vendor creates invoice', 'intercase_n_3__Change Currency_Vendor creates invoice_Record Goods Receipt', 'intercase_n_3__Change Currency_Vendor creates invoice_Record Invoice Receipt', 'intercase_n_3__Change Currency_Vendor creates invoice_Vendor creates debit memo', 'intercase_n_3__Change Delivery Indicator_Block Purchase Order Item_Change Quantity', 'intercase_n_3__Change Delivery Indicator_Block Purchase Order Item_Reactivate Purchase Order Item', 'intercase_n_3__Change Delivery Indicator_Cancel Goods Receipt_Change Quantity', 'intercase_n_3__Change Delivery Indicator_Cancel Goods Receipt_Record Goods Receipt', 'intercase_n_3__Change Delivery Indicator_Cancel Invoice Receipt_Clear Invoice', 'intercase_n_3__Change Delivery Indicator_Cancel Invoice Receipt_Record Invoice Receipt', 'intercase_n_3__Change Delivery Indicator_Cancel Invoice Receipt_Remove Payment Block', 'intercase_n_3__Change Delivery Indicator_Cancel Invoice Receipt_Vendor creates debit memo', 'intercase_n_3__Change Delivery Indicator_Cancel Subsequent Invoice_Clear Invoice', 'intercase_n_3__Change Delivery Indicator_Cancel Subsequent Invoice_Record Subsequent Invoice', 'intercase_n_3__Change Delivery Indicator_Change Delivery Indicator_Cancel Goods Receipt', 'intercase_n_3__Change Delivery Indicator_Change Delivery Indicator_Change Delivery Indicator', 'intercase_n_3__Change Delivery Indicator_Change Delivery Indicator_Change Quantity', 'intercase_n_3__Change Delivery Indicator_Change Delivery Indicator_Clear Invoice', 'intercase_n_3__Change Delivery Indicator_Change Delivery Indicator_Reactivate Purchase Order Item', 'intercase_n_3__Change Delivery Indicator_Change Delivery Indicator_Record Goods Receipt', 'intercase_n_3__Change Delivery Indicator_Change Delivery Indicator_Record Invoice Receipt', 'intercase_n_3__Change Delivery Indicator_Change Delivery Indicator_Remove Payment Block', 'intercase_n_3__Change Delivery Indicator_Change Delivery Indicator_Vendor creates debit memo', 'intercase_n_3__Change Delivery Indicator_Change Delivery Indicator_Vendor creates invoice', 'intercase_n_3__Change Delivery Indicator_Change Final Invoice Indicator_Record Invoice Receipt', 'intercase_n_3__Change Delivery Indicator_Change Final Invoice Indicator_SRM: In Transfer to Execution Syst.', 'intercase_n_3__Change Delivery Indicator_Change Price_Change Quantity', 'intercase_n_3__Change Delivery Indicator_Change Price_Record Goods Receipt', 'intercase_n_3__Change Delivery Indicator_Change Price_Record Invoice Receipt', 'intercase_n_3__Change Delivery Indicator_Change Price_Remove Payment Block', 'intercase_n_3__Change Delivery Indicator_Change Quantity_Cancel Invoice Receipt', 'intercase_n_3__Change Delivery Indicator_Change Quantity_Change Delivery Indicator', 'intercase_n_3__Change Delivery Indicator_Change Quantity_Change Price', 'intercase_n_3__Change Delivery Indicator_Change Quantity_Change Quantity', 'intercase_n_3__Change Delivery Indicator_Change Quantity_Clear Invoice', 'intercase_n_3__Change Delivery Indicator_Change Quantity_Delete Purchase Order Item', 'intercase_n_3__Change Delivery Indicator_Change Quantity_Record Goods Receipt', 'intercase_n_3__Change Delivery Indicator_Change Quantity_Record Invoice Receipt', 'intercase_n_3__Change Delivery Indicator_Change Quantity_Remove Payment Block', 'intercase_n_3__Change Delivery Indicator_Change Quantity_Vendor creates invoice', 'intercase_n_3__Change Delivery Indicator_Change Storage Location_Change Storage Location', 'intercase_n_3__Change Delivery Indicator_Clear Invoice_Cancel Invoice Receipt', 'intercase_n_3__Change Delivery Indicator_Clear Invoice_Change Delivery Indicator', 'intercase_n_3__Change Delivery Indicator_Clear Invoice_Change Quantity', 'intercase_n_3__Change Delivery Indicator_Clear Invoice_Clear Invoice', 'intercase_n_3__Change Delivery Indicator_Clear Invoice_Record Goods Receipt', 'intercase_n_3__Change Delivery Indicator_Clear Invoice_Record Invoice Receipt', 'intercase_n_3__Change Delivery Indicator_Reactivate Purchase Order Item_Change Quantity', 'intercase_n_3__Change Delivery Indicator_Record Goods Receipt_Cancel Goods Receipt', 'intercase_n_3__Change Delivery Indicator_Record Goods Receipt_Cancel Invoice Receipt', 'intercase_n_3__Change Delivery Indicator_Record Goods Receipt_Cancel Subsequent Invoice', 'intercase_n_3__Change Delivery Indicator_Record Goods Receipt_Change Delivery Indicator', 'intercase_n_3__Change Delivery Indicator_Record Goods Receipt_Change Price', 'intercase_n_3__Change Delivery Indicator_Record Goods Receipt_Change Quantity', 'intercase_n_3__Change Delivery Indicator_Record Goods Receipt_Clear Invoice', 'intercase_n_3__Change Delivery Indicator_Record Goods Receipt_Record Goods Receipt', 'intercase_n_3__Change Delivery Indicator_Record Goods Receipt_Record Invoice Receipt', 'intercase_n_3__Change Delivery Indicator_Record Goods Receipt_Remove Payment Block', 'intercase_n_3__Change Delivery Indicator_Record Goods Receipt_Vendor creates debit memo', 'intercase_n_3__Change Delivery Indicator_Record Goods Receipt_Vendor creates invoice', 'intercase_n_3__Change Delivery Indicator_Record Invoice Receipt_Cancel Invoice Receipt', 'intercase_n_3__Change Delivery Indicator_Record Invoice Receipt_Cancel Subsequent Invoice', 'intercase_n_3__Change Delivery Indicator_Record Invoice Receipt_Change Delivery Indicator', 'intercase_n_3__Change Delivery Indicator_Record Invoice Receipt_Change Price', 'intercase_n_3__Change Delivery Indicator_Record Invoice Receipt_Change Quantity', 'intercase_n_3__Change Delivery Indicator_Record Invoice Receipt_Clear Invoice', 'intercase_n_3__Change Delivery Indicator_Record Invoice Receipt_Record Goods Receipt', 'intercase_n_3__Change Delivery Indicator_Record Invoice Receipt_Record Invoice Receipt', 'intercase_n_3__Change Delivery Indicator_Record Invoice Receipt_Record Subsequent Invoice', 'intercase_n_3__Change Delivery Indicator_Record Invoice Receipt_Remove Payment Block', 'intercase_n_3__Change Delivery Indicator_Record Invoice Receipt_Vendor creates debit memo', 'intercase_n_3__Change Delivery Indicator_Record Invoice Receipt_Vendor creates invoice', 'intercase_n_3__Change Delivery Indicator_Record Subsequent Invoice_Clear Invoice', 'intercase_n_3__Change Delivery Indicator_Release Purchase Order_Change Quantity', 'intercase_n_3__Change Delivery Indicator_Remove Payment Block_Cancel Invoice Receipt', 'intercase_n_3__Change Delivery Indicator_Remove Payment Block_Change Delivery Indicator', 'intercase_n_3__Change Delivery Indicator_Remove Payment Block_Clear Invoice', 'intercase_n_3__Change Delivery Indicator_Remove Payment Block_Record Goods Receipt', 'intercase_n_3__Change Delivery Indicator_Remove Payment Block_Record Invoice Receipt', 'intercase_n_3__Change Delivery Indicator_Remove Payment Block_Vendor creates debit memo', 'intercase_n_3__Change Delivery Indicator_SRM: Created_SRM: Complete', 'intercase_n_3__Change Delivery Indicator_Vendor creates debit memo_Cancel Invoice Receipt', 'intercase_n_3__Change Delivery Indicator_Vendor creates debit memo_Cancel Subsequent Invoice', 'intercase_n_3__Change Delivery Indicator_Vendor creates debit memo_Change Delivery Indicator', 'intercase_n_3__Change Delivery Indicator_Vendor creates debit memo_Change Price', 'intercase_n_3__Change Delivery Indicator_Vendor creates debit memo_Clear Invoice', 'intercase_n_3__Change Delivery Indicator_Vendor creates debit memo_Record Goods Receipt', 'intercase_n_3__Change Delivery Indicator_Vendor creates debit memo_Record Invoice Receipt', 'intercase_n_3__Change Delivery Indicator_Vendor creates debit memo_Remove Payment Block', 'intercase_n_3__Change Delivery Indicator_Vendor creates debit memo_Vendor creates invoice', 'intercase_n_3__Change Delivery Indicator_Vendor creates invoice_Change Delivery Indicator', 'intercase_n_3__Change Delivery Indicator_Vendor creates invoice_Change Price', 'intercase_n_3__Change Delivery Indicator_Vendor creates invoice_Change Quantity', 'intercase_n_3__Change Delivery Indicator_Vendor creates invoice_Clear Invoice', 'intercase_n_3__Change Delivery Indicator_Vendor creates invoice_Record Goods Receipt', 'intercase_n_3__Change Delivery Indicator_Vendor creates invoice_Record Invoice Receipt', 'intercase_n_3__Change Delivery Indicator_Vendor creates invoice_Release Purchase Order', 'intercase_n_3__Change Delivery Indicator_Vendor creates invoice_Vendor creates debit memo', 'intercase_n_3__Change Delivery Indicator_Vendor creates invoice_Vendor creates invoice', 'intercase_n_3__Change Final Invoice Indicator_SRM: In Transfer to Execution Syst._SRM: In Transfer to Execution Syst.', 'intercase_n_3__Change Price_Block Purchase Order Item_Reactivate Purchase Order Item', 'intercase_n_3__Change Price_Cancel Goods Receipt_Cancel Goods Receipt', 'intercase_n_3__Change Price_Cancel Goods Receipt_Cancel Invoice Receipt', 'intercase_n_3__Change Price_Cancel Goods Receipt_Record Goods Receipt', 'intercase_n_3__Change Price_Cancel Goods Receipt_Vendor creates debit memo', 'intercase_n_3__Change Price_Cancel Invoice Receipt_Clear Invoice', 'intercase_n_3__Change Price_Cancel Invoice Receipt_Record Invoice Receipt', 'intercase_n_3__Change Price_Cancel Invoice Receipt_Remove Payment Block', 'intercase_n_3__Change Price_Cancel Subsequent Invoice_Clear Invoice', 'intercase_n_3__Change Price_Cancel Subsequent Invoice_Remove Payment Block', 'intercase_n_3__Change Price_Change Approval for Purchase Order_Change Approval for Purchase Order', 'intercase_n_3__Change Price_Change Approval for Purchase Order_Change Price', 'intercase_n_3__Change Price_Change Approval for Purchase Order_Change Quantity', 'intercase_n_3__Change Price_Change Approval for Purchase Order_Record Goods Receipt', 'intercase_n_3__Change Price_Change Approval for Purchase Order_Record Invoice Receipt', 'intercase_n_3__Change Price_Change Approval for Purchase Order_Release Purchase Order', 'intercase_n_3__Change Price_Change Approval for Purchase Order_Vendor creates invoice', 'intercase_n_3__Change Price_Change Currency_Change Price', 'intercase_n_3__Change Price_Change Currency_Change Quantity', 'intercase_n_3__Change Price_Change Currency_Vendor creates debit memo', 'intercase_n_3__Change Price_Change Currency_Vendor creates invoice', 'intercase_n_3__Change Price_Change Delivery Indicator_Change Delivery Indicator', 'intercase_n_3__Change Price_Change Delivery Indicator_Change Quantity', 'intercase_n_3__Change Price_Change Delivery Indicator_Record Goods Receipt', 'intercase_n_3__Change Price_Change Delivery Indicator_Record Invoice Receipt', 'intercase_n_3__Change Price_Change Delivery Indicator_Record Subsequent Invoice', 'intercase_n_3__Change Price_Change Delivery Indicator_Remove Payment Block', 'intercase_n_3__Change Price_Change Delivery Indicator_Vendor creates invoice', 'intercase_n_3__Change Price_Change Price_Change Approval for Purchase Order', 'intercase_n_3__Change Price_Change Price_Change Currency', 'intercase_n_3__Change Price_Change Price_Change Price', 'intercase_n_3__Change Price_Change Price_Change Quantity', 'intercase_n_3__Change Price_Change Price_Change Storage Location', 'intercase_n_3__Change Price_Change Price_Clear Invoice', 'intercase_n_3__Change Price_Change Price_Receive Order Confirmation', 'intercase_n_3__Change Price_Change Price_Record Goods Receipt', 'intercase_n_3__Change Price_Change Price_Record Invoice Receipt', 'intercase_n_3__Change Price_Change Price_Record Subsequent Invoice', 'intercase_n_3__Change Price_Change Price_Release Purchase Order', 'intercase_n_3__Change Price_Change Price_Remove Payment Block', 'intercase_n_3__Change Price_Change Price_Vendor creates debit memo', 'intercase_n_3__Change Price_Change Price_Vendor creates invoice', 'intercase_n_3__Change Price_Change Quantity_Change Approval for Purchase Order', 'intercase_n_3__Change Price_Change Quantity_Change Delivery Indicator', 'intercase_n_3__Change Price_Change Quantity_Change Price', 'intercase_n_3__Change Price_Change Quantity_Change Quantity', 'intercase_n_3__Change Price_Change Quantity_Change Storage Location', 'intercase_n_3__Change Price_Change Quantity_Clear Invoice', 'intercase_n_3__Change Price_Change Quantity_Receive Order Confirmation', 'intercase_n_3__Change Price_Change Quantity_Record Goods Receipt', 'intercase_n_3__Change Price_Change Quantity_Record Invoice Receipt', 'intercase_n_3__Change Price_Change Quantity_Release Purchase Order', 'intercase_n_3__Change Price_Change Quantity_Remove Payment Block', 'intercase_n_3__Change Price_Change Quantity_Update Order Confirmation', 'intercase_n_3__Change Price_Change Quantity_Vendor creates debit memo', 'intercase_n_3__Change Price_Change Quantity_Vendor creates invoice', 'intercase_n_3__Change Price_Change Storage Location_Change Price', 'intercase_n_3__Change Price_Change Storage Location_Change Quantity', 'intercase_n_3__Change Price_Change Storage Location_Receive Order Confirmation', 'intercase_n_3__Change Price_Change Storage Location_Record Goods Receipt', 'intercase_n_3__Change Price_Change Storage Location_Vendor creates invoice', 'intercase_n_3__Change Price_Change payment term_Record Goods Receipt', 'intercase_n_3__Change Price_Clear Invoice_Cancel Invoice Receipt', 'intercase_n_3__Change Price_Clear Invoice_Clear Invoice', 'intercase_n_3__Change Price_Clear Invoice_Record Goods Receipt', 'intercase_n_3__Change Price_Clear Invoice_Record Invoice Receipt', 'intercase_n_3__Change Price_Clear Invoice_Record Subsequent Invoice', 'intercase_n_3__Change Price_Clear Invoice_Vendor creates debit memo', 'intercase_n_3__Change Price_Clear Invoice_Vendor creates invoice', 'intercase_n_3__Change Price_Delete Purchase Order Item_Reactivate Purchase Order Item', 'intercase_n_3__Change Price_Receive Order Confirmation_Record Goods Receipt', 'intercase_n_3__Change Price_Receive Order Confirmation_Record Invoice Receipt', 'intercase_n_3__Change Price_Receive Order Confirmation_Update Order Confirmation', 'intercase_n_3__Change Price_Receive Order Confirmation_Vendor creates invoice', 'intercase_n_3__Change Price_Record Goods Receipt_Cancel Goods Receipt', 'intercase_n_3__Change Price_Record Goods Receipt_Cancel Invoice Receipt', 'intercase_n_3__Change Price_Record Goods Receipt_Change Delivery Indicator', 'intercase_n_3__Change Price_Record Goods Receipt_Change Price', 'intercase_n_3__Change Price_Record Goods Receipt_Change Quantity', 'intercase_n_3__Change Price_Record Goods Receipt_Clear Invoice', 'intercase_n_3__Change Price_Record Goods Receipt_Record Goods Receipt', 'intercase_n_3__Change Price_Record Goods Receipt_Record Invoice Receipt', 'intercase_n_3__Change Price_Record Goods Receipt_Record Service Entry Sheet', 'intercase_n_3__Change Price_Record Goods Receipt_Remove Payment Block', 'intercase_n_3__Change Price_Record Goods Receipt_Vendor creates debit memo', 'intercase_n_3__Change Price_Record Goods Receipt_Vendor creates invoice', 'intercase_n_3__Change Price_Record Invoice Receipt_Cancel Invoice Receipt', 'intercase_n_3__Change Price_Record Invoice Receipt_Change Delivery Indicator', 'intercase_n_3__Change Price_Record Invoice Receipt_Change Price', 'intercase_n_3__Change Price_Record Invoice Receipt_Change Quantity', 'intercase_n_3__Change Price_Record Invoice Receipt_Clear Invoice', 'intercase_n_3__Change Price_Record Invoice Receipt_Record Goods Receipt', 'intercase_n_3__Change Price_Record Invoice Receipt_Record Invoice Receipt', 'intercase_n_3__Change Price_Record Invoice Receipt_Record Service Entry Sheet', 'intercase_n_3__Change Price_Record Invoice Receipt_Record Subsequent Invoice', 'intercase_n_3__Change Price_Record Invoice Receipt_Remove Payment Block', 'intercase_n_3__Change Price_Record Invoice Receipt_Vendor creates debit memo', 'intercase_n_3__Change Price_Record Invoice Receipt_Vendor creates invoice', 'intercase_n_3__Change Price_Record Service Entry Sheet_Change Price', 'intercase_n_3__Change Price_Record Service Entry Sheet_Record Goods Receipt', 'intercase_n_3__Change Price_Record Service Entry Sheet_Record Invoice Receipt', 'intercase_n_3__Change Price_Record Subsequent Invoice_Cancel Invoice Receipt', 'intercase_n_3__Change Price_Record Subsequent Invoice_Clear Invoice', 'intercase_n_3__Change Price_Record Subsequent Invoice_Remove Payment Block', 'intercase_n_3__Change Price_Release Purchase Order_Change Approval for Purchase Order', 'intercase_n_3__Change Price_Release Purchase Order_Change Price', 'intercase_n_3__Change Price_Remove Payment Block_Cancel Goods Receipt', 'intercase_n_3__Change Price_Remove Payment Block_Cancel Invoice Receipt', 'intercase_n_3__Change Price_Remove Payment Block_Cancel Subsequent Invoice', 'intercase_n_3__Change Price_Remove Payment Block_Change Delivery Indicator', 'intercase_n_3__Change Price_Remove Payment Block_Change Price', 'intercase_n_3__Change Price_Remove Payment Block_Change Quantity', 'intercase_n_3__Change Price_Remove Payment Block_Clear Invoice', 'intercase_n_3__Change Price_Remove Payment Block_Record Goods Receipt', 'intercase_n_3__Change Price_Remove Payment Block_Record Invoice Receipt', 'intercase_n_3__Change Price_Remove Payment Block_Release Purchase Order', 'intercase_n_3__Change Price_Remove Payment Block_Remove Payment Block', 'intercase_n_3__Change Price_Remove Payment Block_Vendor creates debit memo', 'intercase_n_3__Change Price_Remove Payment Block_Vendor creates invoice', 'intercase_n_3__Change Price_SRM: Complete_SRM: Awaiting Approval', 'intercase_n_3__Change Price_SRM: Created_Record Goods Receipt', 'intercase_n_3__Change Price_SRM: Created_SRM: Complete', 'intercase_n_3__Change Price_SRM: In Transfer to Execution Syst._Cancel Goods Receipt', 'intercase_n_3__Change Price_SRM: Transaction Completed_Change Delivery Indicator', 'intercase_n_3__Change Price_Update Order Confirmation_Vendor creates invoice', 'intercase_n_3__Change Price_Vendor creates debit memo_Cancel Goods Receipt', 'intercase_n_3__Change Price_Vendor creates debit memo_Cancel Invoice Receipt', 'intercase_n_3__Change Price_Vendor creates debit memo_Cancel Subsequent Invoice', 'intercase_n_3__Change Price_Vendor creates debit memo_Clear Invoice', 'intercase_n_3__Change Price_Vendor creates debit memo_Record Invoice Receipt', 'intercase_n_3__Change Price_Vendor creates debit memo_Vendor creates invoice', 'intercase_n_3__Change Price_Vendor creates invoice_Block Purchase Order Item', 'intercase_n_3__Change Price_Vendor creates invoice_Change Delivery Indicator', 'intercase_n_3__Change Price_Vendor creates invoice_Change Price', 'intercase_n_3__Change Price_Vendor creates invoice_Change Quantity', 'intercase_n_3__Change Price_Vendor creates invoice_Change Storage Location', 'intercase_n_3__Change Price_Vendor creates invoice_Clear Invoice', 'intercase_n_3__Change Price_Vendor creates invoice_Record Goods Receipt', 'intercase_n_3__Change Price_Vendor creates invoice_Record Invoice Receipt', 'intercase_n_3__Change Price_Vendor creates invoice_Record Service Entry Sheet', 'intercase_n_3__Change Price_Vendor creates invoice_Remove Payment Block', 'intercase_n_3__Change Price_Vendor creates invoice_SRM: Created', 'intercase_n_3__Change Price_Vendor creates invoice_Vendor creates debit memo', 'intercase_n_3__Change Price_Vendor creates invoice_Vendor creates invoice', 'intercase_n_3__Change Quantity_Block Purchase Order Item_Change Quantity', 'intercase_n_3__Change Quantity_Block Purchase Order Item_Reactivate Purchase Order Item', 'intercase_n_3__Change Quantity_Block Purchase Order Item_Vendor creates invoice', 'intercase_n_3__Change Quantity_Cancel Goods Receipt_Cancel Goods Receipt', 'intercase_n_3__Change Quantity_Cancel Goods Receipt_Cancel Invoice Receipt', 'intercase_n_3__Change Quantity_Cancel Goods Receipt_Change Quantity', 'intercase_n_3__Change Quantity_Cancel Goods Receipt_Record Goods Receipt', 'intercase_n_3__Change Quantity_Cancel Goods Receipt_Vendor creates debit memo', 'intercase_n_3__Change Quantity_Cancel Invoice Receipt_Clear Invoice', 'intercase_n_3__Change Quantity_Cancel Invoice Receipt_Remove Payment Block', 'intercase_n_3__Change Quantity_Change Approval for Purchase Order_Cancel Invoice Receipt', 'intercase_n_3__Change Quantity_Change Approval for Purchase Order_Change Price', 'intercase_n_3__Change Quantity_Change Approval for Purchase Order_Change Quantity', 'intercase_n_3__Change Quantity_Change Approval for Purchase Order_Delete Purchase Order Item', 'intercase_n_3__Change Quantity_Change Approval for Purchase Order_Receive Order Confirmation', 'intercase_n_3__Change Quantity_Change Approval for Purchase Order_Record Goods Receipt', 'intercase_n_3__Change Quantity_Change Approval for Purchase Order_Release Purchase Order', 'intercase_n_3__Change Quantity_Change Approval for Purchase Order_Vendor creates debit memo', 'intercase_n_3__Change Quantity_Change Approval for Purchase Order_Vendor creates invoice', 'intercase_n_3__Change Quantity_Change Delivery Indicator_Block Purchase Order Item', 'intercase_n_3__Change Quantity_Change Delivery Indicator_Cancel Goods Receipt', 'intercase_n_3__Change Quantity_Change Delivery Indicator_Cancel Invoice Receipt', 'intercase_n_3__Change Quantity_Change Delivery Indicator_Change Delivery Indicator', 'intercase_n_3__Change Quantity_Change Delivery Indicator_Change Price', 'intercase_n_3__Change Quantity_Change Delivery Indicator_Change Quantity', 'intercase_n_3__Change Quantity_Change Delivery Indicator_Change Storage Location', 'intercase_n_3__Change Quantity_Change Delivery Indicator_Clear Invoice', 'intercase_n_3__Change Quantity_Change Delivery Indicator_Record Goods Receipt', 'intercase_n_3__Change Quantity_Change Delivery Indicator_Record Invoice Receipt', 'intercase_n_3__Change Quantity_Change Delivery Indicator_Release Purchase Order', 'intercase_n_3__Change Quantity_Change Delivery Indicator_Remove Payment Block', 'intercase_n_3__Change Quantity_Change Delivery Indicator_Vendor creates debit memo', 'intercase_n_3__Change Quantity_Change Delivery Indicator_Vendor creates invoice', 'intercase_n_3__Change Quantity_Change Price_Cancel Goods Receipt', 'intercase_n_3__Change Quantity_Change Price_Cancel Invoice Receipt', 'intercase_n_3__Change Quantity_Change Price_Change Approval for Purchase Order', 'intercase_n_3__Change Quantity_Change Price_Change Delivery Indicator', 'intercase_n_3__Change Quantity_Change Price_Change Price', 'intercase_n_3__Change Quantity_Change Price_Change Quantity', 'intercase_n_3__Change Quantity_Change Price_Receive Order Confirmation', 'intercase_n_3__Change Quantity_Change Price_Record Goods Receipt', 'intercase_n_3__Change Quantity_Change Price_Record Invoice Receipt', 'intercase_n_3__Change Quantity_Change Price_Remove Payment Block', 'intercase_n_3__Change Quantity_Change Price_Update Order Confirmation', 'intercase_n_3__Change Quantity_Change Price_Vendor creates debit memo', 'intercase_n_3__Change Quantity_Change Price_Vendor creates invoice', 'intercase_n_3__Change Quantity_Change Quantity_Block Purchase Order Item', 'intercase_n_3__Change Quantity_Change Quantity_Change Approval for Purchase Order', 'intercase_n_3__Change Quantity_Change Quantity_Change Delivery Indicator', 'intercase_n_3__Change Quantity_Change Quantity_Change Price', 'intercase_n_3__Change Quantity_Change Quantity_Change Quantity', 'intercase_n_3__Change Quantity_Change Quantity_Change Storage Location', 'intercase_n_3__Change Quantity_Change Quantity_Delete Purchase Order Item', 'intercase_n_3__Change Quantity_Change Quantity_Receive Order Confirmation', 'intercase_n_3__Change Quantity_Change Quantity_Record Goods Receipt', 'intercase_n_3__Change Quantity_Change Quantity_Record Invoice Receipt', 'intercase_n_3__Change Quantity_Change Quantity_Release Purchase Order', 'intercase_n_3__Change Quantity_Change Quantity_Remove Payment Block', 'intercase_n_3__Change Quantity_Change Quantity_Update Order Confirmation', 'intercase_n_3__Change Quantity_Change Quantity_Vendor creates debit memo', 'intercase_n_3__Change Quantity_Change Quantity_Vendor creates invoice', 'intercase_n_3__Change Quantity_Change Storage Location_Change Price', 'intercase_n_3__Change Quantity_Change Storage Location_Change Quantity', 'intercase_n_3__Change Quantity_Change Storage Location_Change Storage Location', 'intercase_n_3__Change Quantity_Change Storage Location_Record Goods Receipt', 'intercase_n_3__Change Quantity_Change Storage Location_Vendor creates invoice', 'intercase_n_3__Change Quantity_Clear Invoice_Cancel Goods Receipt', 'intercase_n_3__Change Quantity_Clear Invoice_Cancel Invoice Receipt', 'intercase_n_3__Change Quantity_Clear Invoice_Change Delivery Indicator', 'intercase_n_3__Change Quantity_Clear Invoice_Change Quantity', 'intercase_n_3__Change Quantity_Clear Invoice_Record Goods Receipt', 'intercase_n_3__Change Quantity_Clear Invoice_Record Invoice Receipt', 'intercase_n_3__Change Quantity_Clear Invoice_Vendor creates invoice', 'intercase_n_3__Change Quantity_Delete Purchase Order Item_Change Approval for Purchase Order', 'intercase_n_3__Change Quantity_Delete Purchase Order Item_Change Quantity', 'intercase_n_3__Change Quantity_Delete Purchase Order Item_Reactivate Purchase Order Item', 'intercase_n_3__Change Quantity_Delete Purchase Order Item_Vendor creates invoice', 'intercase_n_3__Change Quantity_Reactivate Purchase Order Item_Change Price', 'intercase_n_3__Change Quantity_Reactivate Purchase Order Item_Change Quantity', 'intercase_n_3__Change Quantity_Reactivate Purchase Order Item_Record Goods Receipt', 'intercase_n_3__Change Quantity_Reactivate Purchase Order Item_Vendor creates invoice', 'intercase_n_3__Change Quantity_Receive Order Confirmation_Change Price', 'intercase_n_3__Change Quantity_Receive Order Confirmation_Change Quantity', 'intercase_n_3__Change Quantity_Receive Order Confirmation_Record Goods Receipt', 'intercase_n_3__Change Quantity_Receive Order Confirmation_Update Order Confirmation', 'intercase_n_3__Change Quantity_Receive Order Confirmation_Vendor creates invoice', 'intercase_n_3__Change Quantity_Record Goods Receipt_Cancel Goods Receipt', 'intercase_n_3__Change Quantity_Record Goods Receipt_Cancel Invoice Receipt', 'intercase_n_3__Change Quantity_Record Goods Receipt_Change Approval for Purchase Order', 'intercase_n_3__Change Quantity_Record Goods Receipt_Change Delivery Indicator', 'intercase_n_3__Change Quantity_Record Goods Receipt_Change Price', 'intercase_n_3__Change Quantity_Record Goods Receipt_Change Quantity', 'intercase_n_3__Change Quantity_Record Goods Receipt_Clear Invoice', 'intercase_n_3__Change Quantity_Record Goods Receipt_Delete Purchase Order Item', 'intercase_n_3__Change Quantity_Record Goods Receipt_Receive Order Confirmation', 'intercase_n_3__Change Quantity_Record Goods Receipt_Record Goods Receipt', 'intercase_n_3__Change Quantity_Record Goods Receipt_Record Invoice Receipt', 'intercase_n_3__Change Quantity_Record Goods Receipt_Release Purchase Order', 'intercase_n_3__Change Quantity_Record Goods Receipt_Remove Payment Block', 'intercase_n_3__Change Quantity_Record Goods Receipt_Vendor creates debit memo', 'intercase_n_3__Change Quantity_Record Goods Receipt_Vendor creates invoice', 'intercase_n_3__Change Quantity_Record Invoice Receipt_Block Purchase Order Item', 'intercase_n_3__Change Quantity_Record Invoice Receipt_Cancel Invoice Receipt', 'intercase_n_3__Change Quantity_Record Invoice Receipt_Change Delivery Indicator', 'intercase_n_3__Change Quantity_Record Invoice Receipt_Change Price', 'intercase_n_3__Change Quantity_Record Invoice Receipt_Change Quantity', 'intercase_n_3__Change Quantity_Record Invoice Receipt_Clear Invoice', 'intercase_n_3__Change Quantity_Record Invoice Receipt_Receive Order Confirmation', 'intercase_n_3__Change Quantity_Record Invoice Receipt_Record Goods Receipt', 'intercase_n_3__Change Quantity_Record Invoice Receipt_Record Invoice Receipt', 'intercase_n_3__Change Quantity_Record Invoice Receipt_Remove Payment Block', 'intercase_n_3__Change Quantity_Record Invoice Receipt_Vendor creates debit memo', 'intercase_n_3__Change Quantity_Record Invoice Receipt_Vendor creates invoice', 'intercase_n_3__Change Quantity_Release Purchase Order_Change Approval for Purchase Order', 'intercase_n_3__Change Quantity_Release Purchase Order_Change Price', 'intercase_n_3__Change Quantity_Release Purchase Order_Change Quantity', 'intercase_n_3__Change Quantity_Release Purchase Order_Vendor creates invoice', 'intercase_n_3__Change Quantity_Remove Payment Block_Cancel Invoice Receipt', 'intercase_n_3__Change Quantity_Remove Payment Block_Change Delivery Indicator', 'intercase_n_3__Change Quantity_Remove Payment Block_Change Quantity', 'intercase_n_3__Change Quantity_Remove Payment Block_Clear Invoice', 'intercase_n_3__Change Quantity_Remove Payment Block_Record Goods Receipt', 'intercase_n_3__Change Quantity_Update Order Confirmation_Change Price', 'intercase_n_3__Change Quantity_Update Order Confirmation_Change Quantity', 'intercase_n_3__Change Quantity_Update Order Confirmation_Record Goods Receipt', 'intercase_n_3__Change Quantity_Update Order Confirmation_Update Order Confirmation', 'intercase_n_3__Change Quantity_Update Order Confirmation_Vendor creates invoice', 'intercase_n_3__Change Quantity_Vendor creates debit memo_Cancel Invoice Receipt', 'intercase_n_3__Change Quantity_Vendor creates debit memo_Change Delivery Indicator', 'intercase_n_3__Change Quantity_Vendor creates debit memo_Record Goods Receipt', 'intercase_n_3__Change Quantity_Vendor creates debit memo_Record Invoice Receipt', 'intercase_n_3__Change Quantity_Vendor creates debit memo_Remove Payment Block', 'intercase_n_3__Change Quantity_Vendor creates debit memo_Vendor creates debit memo', 'intercase_n_3__Change Quantity_Vendor creates debit memo_Vendor creates invoice', 'intercase_n_3__Change Quantity_Vendor creates invoice_Cancel Invoice Receipt', 'intercase_n_3__Change Quantity_Vendor creates invoice_Change Approval for Purchase Order', 'intercase_n_3__Change Quantity_Vendor creates invoice_Change Delivery Indicator', 'intercase_n_3__Change Quantity_Vendor creates invoice_Change Price', 'intercase_n_3__Change Quantity_Vendor creates invoice_Change Quantity', 'intercase_n_3__Change Quantity_Vendor creates invoice_Change Storage Location', 'intercase_n_3__Change Quantity_Vendor creates invoice_Clear Invoice', 'intercase_n_3__Change Quantity_Vendor creates invoice_Delete Purchase Order Item', 'intercase_n_3__Change Quantity_Vendor creates invoice_Receive Order Confirmation', 'intercase_n_3__Change Quantity_Vendor creates invoice_Record Goods Receipt', 'intercase_n_3__Change Quantity_Vendor creates invoice_Record Invoice Receipt', 'intercase_n_3__Change Quantity_Vendor creates invoice_Remove Payment Block', 'intercase_n_3__Change Quantity_Vendor creates invoice_Vendor creates debit memo', 'intercase_n_3__Change Quantity_Vendor creates invoice_Vendor creates invoice', 'intercase_n_3__Change Rejection Indicator_Change Rejection Indicator_Reactivate Purchase Order Item', 'intercase_n_3__Change Rejection Indicator_Reactivate Purchase Order Item_Record Goods Receipt', 'intercase_n_3__Change Storage Location_Cancel Goods Receipt_Vendor creates debit memo', 'intercase_n_3__Change Storage Location_Change Delivery Indicator_Change Delivery Indicator', 'intercase_n_3__Change Storage Location_Change Price_Change Quantity', 'intercase_n_3__Change Storage Location_Change Price_Record Goods Receipt', 'intercase_n_3__Change Storage Location_Change Price_Vendor creates invoice', 'intercase_n_3__Change Storage Location_Change Quantity_Change Price', 'intercase_n_3__Change Storage Location_Change Quantity_Change Quantity', 'intercase_n_3__Change Storage Location_Change Quantity_Record Goods Receipt', 'intercase_n_3__Change Storage Location_Change Quantity_Vendor creates invoice', 'intercase_n_3__Change Storage Location_Change Storage Location_Cancel Goods Receipt', 'intercase_n_3__Change Storage Location_Change Storage Location_Change Quantity', 'intercase_n_3__Change Storage Location_Change Storage Location_Change Storage Location', 'intercase_n_3__Change Storage Location_Change Storage Location_Record Goods Receipt', 'intercase_n_3__Change Storage Location_Change Storage Location_Record Invoice Receipt', 'intercase_n_3__Change Storage Location_Change Storage Location_Release Purchase Order', 'intercase_n_3__Change Storage Location_Change Storage Location_Remove Payment Block', 'intercase_n_3__Change Storage Location_Change Storage Location_Vendor creates invoice', 'intercase_n_3__Change Storage Location_Delete Purchase Order Item_Reactivate Purchase Order Item', 'intercase_n_3__Change Storage Location_Receive Order Confirmation_Record Goods Receipt', 'intercase_n_3__Change Storage Location_Receive Order Confirmation_Vendor creates invoice', 'intercase_n_3__Change Storage Location_Record Goods Receipt_Cancel Goods Receipt', 'intercase_n_3__Change Storage Location_Record Goods Receipt_Change Price', 'intercase_n_3__Change Storage Location_Record Goods Receipt_Change Quantity', 'intercase_n_3__Change Storage Location_Record Goods Receipt_Change Storage Location', 'intercase_n_3__Change Storage Location_Record Goods Receipt_Record Goods Receipt', 'intercase_n_3__Change Storage Location_Record Goods Receipt_Record Invoice Receipt', 'intercase_n_3__Change Storage Location_Record Goods Receipt_Remove Payment Block', 'intercase_n_3__Change Storage Location_Record Goods Receipt_Vendor creates invoice', 'intercase_n_3__Change Storage Location_Record Invoice Receipt_Remove Payment Block', 'intercase_n_3__Change Storage Location_Release Purchase Order_Delete Purchase Order Item', 'intercase_n_3__Change Storage Location_Remove Payment Block_Change Delivery Indicator', 'intercase_n_3__Change Storage Location_Vendor creates invoice_Change Price', 'intercase_n_3__Change Storage Location_Vendor creates invoice_Change Quantity', 'intercase_n_3__Change Storage Location_Vendor creates invoice_Record Goods Receipt', 'intercase_n_3__Change Storage Location_Vendor creates invoice_Record Invoice Receipt', 'intercase_n_3__Change Storage Location_Vendor creates invoice_Vendor creates invoice', 'intercase_n_3__Change payment term_Change Price_Record Goods Receipt', 'intercase_n_3__Change payment term_Record Goods Receipt_Vendor creates invoice', 'intercase_n_3__Clear Invoice_Block Purchase Order Item_Reactivate Purchase Order Item', 'intercase_n_3__Clear Invoice_Cancel Goods Receipt_Cancel Goods Receipt', 'intercase_n_3__Clear Invoice_Cancel Goods Receipt_Change Quantity', 'intercase_n_3__Clear Invoice_Cancel Goods Receipt_Record Goods Receipt', 'intercase_n_3__Clear Invoice_Cancel Goods Receipt_Record Service Entry Sheet', 'intercase_n_3__Clear Invoice_Cancel Goods Receipt_Vendor creates debit memo', 'intercase_n_3__Clear Invoice_Cancel Invoice Receipt_Cancel Goods Receipt', 'intercase_n_3__Clear Invoice_Cancel Invoice Receipt_Cancel Invoice Receipt', 'intercase_n_3__Clear Invoice_Cancel Invoice Receipt_Change Delivery Indicator', 'intercase_n_3__Clear Invoice_Cancel Invoice Receipt_Change Quantity', 'intercase_n_3__Clear Invoice_Cancel Invoice Receipt_Clear Invoice', 'intercase_n_3__Clear Invoice_Cancel Invoice Receipt_Record Goods Receipt', 'intercase_n_3__Clear Invoice_Cancel Invoice Receipt_Record Invoice Receipt', 'intercase_n_3__Clear Invoice_Cancel Invoice Receipt_Record Service Entry Sheet', 'intercase_n_3__Clear Invoice_Cancel Invoice Receipt_Record Subsequent Invoice', 'intercase_n_3__Clear Invoice_Cancel Invoice Receipt_Remove Payment Block', 'intercase_n_3__Clear Invoice_Cancel Invoice Receipt_Vendor creates debit memo', 'intercase_n_3__Clear Invoice_Cancel Invoice Receipt_Vendor creates invoice', 'intercase_n_3__Clear Invoice_Cancel Subsequent Invoice_Clear Invoice', 'intercase_n_3__Clear Invoice_Cancel Subsequent Invoice_Record Goods Receipt', 'intercase_n_3__Clear Invoice_Cancel Subsequent Invoice_Record Invoice Receipt', 'intercase_n_3__Clear Invoice_Cancel Subsequent Invoice_Record Subsequent Invoice', 'intercase_n_3__Clear Invoice_Cancel Subsequent Invoice_Remove Payment Block', 'intercase_n_3__Clear Invoice_Change Approval for Purchase Order_Change Approval for Purchase Order', 'intercase_n_3__Clear Invoice_Change Approval for Purchase Order_Record Invoice Receipt', 'intercase_n_3__Clear Invoice_Change Delivery Indicator_Cancel Subsequent Invoice', 'intercase_n_3__Clear Invoice_Change Delivery Indicator_Change Quantity', 'intercase_n_3__Clear Invoice_Change Delivery Indicator_Record Goods Receipt', 'intercase_n_3__Clear Invoice_Change Delivery Indicator_Record Invoice Receipt', 'intercase_n_3__Clear Invoice_Change Delivery Indicator_Remove Payment Block', 'intercase_n_3__Clear Invoice_Change Delivery Indicator_Vendor creates invoice', 'intercase_n_3__Clear Invoice_Change Price_Cancel Invoice Receipt', 'intercase_n_3__Clear Invoice_Change Price_Cancel Subsequent Invoice', 'intercase_n_3__Clear Invoice_Change Price_Change Price', 'intercase_n_3__Clear Invoice_Change Price_Change Quantity', 'intercase_n_3__Clear Invoice_Change Price_Clear Invoice', 'intercase_n_3__Clear Invoice_Change Price_Record Goods Receipt', 'intercase_n_3__Clear Invoice_Change Price_Record Invoice Receipt', 'intercase_n_3__Clear Invoice_Change Price_Record Service Entry Sheet', 'intercase_n_3__Clear Invoice_Change Price_Record Subsequent Invoice', 'intercase_n_3__Clear Invoice_Change Price_Remove Payment Block', 'intercase_n_3__Clear Invoice_Change Price_Vendor creates invoice', 'intercase_n_3__Clear Invoice_Change Quantity_Cancel Invoice Receipt', 'intercase_n_3__Clear Invoice_Change Quantity_Cancel Subsequent Invoice', 'intercase_n_3__Clear Invoice_Change Quantity_Change Delivery Indicator', 'intercase_n_3__Clear Invoice_Change Quantity_Delete Purchase Order Item', 'intercase_n_3__Clear Invoice_Change Quantity_Record Goods Receipt', 'intercase_n_3__Clear Invoice_Change Quantity_Record Invoice Receipt', 'intercase_n_3__Clear Invoice_Change Quantity_Vendor creates invoice', 'intercase_n_3__Clear Invoice_Clear Invoice_Cancel Goods Receipt', 'intercase_n_3__Clear Invoice_Clear Invoice_Cancel Invoice Receipt', 'intercase_n_3__Clear Invoice_Clear Invoice_Cancel Subsequent Invoice', 'intercase_n_3__Clear Invoice_Clear Invoice_Change Delivery Indicator', 'intercase_n_3__Clear Invoice_Clear Invoice_Change Price', 'intercase_n_3__Clear Invoice_Clear Invoice_Clear Invoice', 'intercase_n_3__Clear Invoice_Clear Invoice_Record Goods Receipt', 'intercase_n_3__Clear Invoice_Clear Invoice_Record Invoice Receipt', 'intercase_n_3__Clear Invoice_Clear Invoice_Record Service Entry Sheet', 'intercase_n_3__Clear Invoice_Clear Invoice_Remove Payment Block', 'intercase_n_3__Clear Invoice_Clear Invoice_SRM: Created', 'intercase_n_3__Clear Invoice_Clear Invoice_SRM: Ordered', 'intercase_n_3__Clear Invoice_Clear Invoice_Set Payment Block', 'intercase_n_3__Clear Invoice_Clear Invoice_Vendor creates debit memo', 'intercase_n_3__Clear Invoice_Clear Invoice_Vendor creates invoice', 'intercase_n_3__Clear Invoice_Record Goods Receipt_Cancel Goods Receipt', 'intercase_n_3__Clear Invoice_Record Goods Receipt_Cancel Invoice Receipt', 'intercase_n_3__Clear Invoice_Record Goods Receipt_Change Delivery Indicator', 'intercase_n_3__Clear Invoice_Record Goods Receipt_Change Quantity', 'intercase_n_3__Clear Invoice_Record Goods Receipt_Clear Invoice', 'intercase_n_3__Clear Invoice_Record Goods Receipt_Record Goods Receipt', 'intercase_n_3__Clear Invoice_Record Goods Receipt_Record Invoice Receipt', 'intercase_n_3__Clear Invoice_Record Goods Receipt_Record Service Entry Sheet', 'intercase_n_3__Clear Invoice_Record Goods Receipt_Remove Payment Block', 'intercase_n_3__Clear Invoice_Record Goods Receipt_Vendor creates invoice', 'intercase_n_3__Clear Invoice_Record Invoice Receipt_Cancel Goods Receipt', 'intercase_n_3__Clear Invoice_Record Invoice Receipt_Cancel Invoice Receipt', 'intercase_n_3__Clear Invoice_Record Invoice Receipt_Cancel Subsequent Invoice', 'intercase_n_3__Clear Invoice_Record Invoice Receipt_Change Approval for Purchase Order', 'intercase_n_3__Clear Invoice_Record Invoice Receipt_Change Delivery Indicator', 'intercase_n_3__Clear Invoice_Record Invoice Receipt_Change Price', 'intercase_n_3__Clear Invoice_Record Invoice Receipt_Change Quantity', 'intercase_n_3__Clear Invoice_Record Invoice Receipt_Clear Invoice', 'intercase_n_3__Clear Invoice_Record Invoice Receipt_Record Goods Receipt', 'intercase_n_3__Clear Invoice_Record Invoice Receipt_Record Invoice Receipt', 'intercase_n_3__Clear Invoice_Record Invoice Receipt_Record Service Entry Sheet', 'intercase_n_3__Clear Invoice_Record Invoice Receipt_Record Subsequent Invoice', 'intercase_n_3__Clear Invoice_Record Invoice Receipt_Remove Payment Block', 'intercase_n_3__Clear Invoice_Record Invoice Receipt_SRM: Created', 'intercase_n_3__Clear Invoice_Record Invoice Receipt_Vendor creates debit memo', 'intercase_n_3__Clear Invoice_Record Invoice Receipt_Vendor creates invoice', 'intercase_n_3__Clear Invoice_Record Service Entry Sheet_Cancel Invoice Receipt', 'intercase_n_3__Clear Invoice_Record Service Entry Sheet_Change Price', 'intercase_n_3__Clear Invoice_Record Service Entry Sheet_Record Goods Receipt', 'intercase_n_3__Clear Invoice_Record Service Entry Sheet_Record Invoice Receipt', 'intercase_n_3__Clear Invoice_Record Service Entry Sheet_Record Service Entry Sheet', 'intercase_n_3__Clear Invoice_Record Subsequent Invoice_Cancel Subsequent Invoice', 'intercase_n_3__Clear Invoice_Record Subsequent Invoice_Clear Invoice', 'intercase_n_3__Clear Invoice_Record Subsequent Invoice_Record Subsequent Invoice', 'intercase_n_3__Clear Invoice_Record Subsequent Invoice_Remove Payment Block', 'intercase_n_3__Clear Invoice_Remove Payment Block_Cancel Invoice Receipt', 'intercase_n_3__Clear Invoice_Remove Payment Block_Cancel Subsequent Invoice', 'intercase_n_3__Clear Invoice_Remove Payment Block_Change Price', 'intercase_n_3__Clear Invoice_Remove Payment Block_Clear Invoice', 'intercase_n_3__Clear Invoice_Remove Payment Block_Record Goods Receipt', 'intercase_n_3__Clear Invoice_Remove Payment Block_Record Invoice Receipt', 'intercase_n_3__Clear Invoice_Remove Payment Block_Record Service Entry Sheet', 'intercase_n_3__Clear Invoice_Remove Payment Block_Remove Payment Block', 'intercase_n_3__Clear Invoice_Remove Payment Block_Set Payment Block', 'intercase_n_3__Clear Invoice_Remove Payment Block_Vendor creates invoice', 'intercase_n_3__Clear Invoice_SRM: Complete_SRM: Awaiting Approval', 'intercase_n_3__Clear Invoice_SRM: Created_Record Service Entry Sheet', 'intercase_n_3__Clear Invoice_SRM: Created_SRM: Complete', 'intercase_n_3__Clear Invoice_SRM: In Transfer to Execution Syst._Record Goods Receipt', 'intercase_n_3__Clear Invoice_SRM: In Transfer to Execution Syst._SRM: In Transfer to Execution Syst.', 'intercase_n_3__Clear Invoice_SRM: In Transfer to Execution Syst._SRM: Ordered', 'intercase_n_3__Clear Invoice_SRM: Ordered_SRM: In Transfer to Execution Syst.', 'intercase_n_3__Clear Invoice_Set Payment Block_Clear Invoice', 'intercase_n_3__Clear Invoice_Vendor creates debit memo_Cancel Goods Receipt', 'intercase_n_3__Clear Invoice_Vendor creates debit memo_Cancel Invoice Receipt', 'intercase_n_3__Clear Invoice_Vendor creates debit memo_Cancel Subsequent Invoice', 'intercase_n_3__Clear Invoice_Vendor creates debit memo_Clear Invoice', 'intercase_n_3__Clear Invoice_Vendor creates debit memo_Record Goods Receipt', 'intercase_n_3__Clear Invoice_Vendor creates debit memo_Record Invoice Receipt', 'intercase_n_3__Clear Invoice_Vendor creates debit memo_Remove Payment Block', 'intercase_n_3__Clear Invoice_Vendor creates debit memo_Vendor creates debit memo', 'intercase_n_3__Clear Invoice_Vendor creates debit memo_Vendor creates invoice', 'intercase_n_3__Clear Invoice_Vendor creates invoice_Cancel Goods Receipt', 'intercase_n_3__Clear Invoice_Vendor creates invoice_Cancel Invoice Receipt', 'intercase_n_3__Clear Invoice_Vendor creates invoice_Cancel Subsequent Invoice', 'intercase_n_3__Clear Invoice_Vendor creates invoice_Change Approval for Purchase Order', 'intercase_n_3__Clear Invoice_Vendor creates invoice_Change Price', 'intercase_n_3__Clear Invoice_Vendor creates invoice_Change Quantity', 'intercase_n_3__Clear Invoice_Vendor creates invoice_Clear Invoice', 'intercase_n_3__Clear Invoice_Vendor creates invoice_Record Goods Receipt', 'intercase_n_3__Clear Invoice_Vendor creates invoice_Record Invoice Receipt', 'intercase_n_3__Clear Invoice_Vendor creates invoice_Record Service Entry Sheet', 'intercase_n_3__Clear Invoice_Vendor creates invoice_Record Subsequent Invoice', 'intercase_n_3__Clear Invoice_Vendor creates invoice_Remove Payment Block', 'intercase_n_3__Clear Invoice_Vendor creates invoice_SRM: Created', 'intercase_n_3__Clear Invoice_Vendor creates invoice_SRM: In Transfer to Execution Syst.', 'intercase_n_3__Clear Invoice_Vendor creates invoice_Vendor creates debit memo', 'intercase_n_3__Clear Invoice_Vendor creates invoice_Vendor creates invoice', 'intercase_n_3__Create Purchase Order Item', 'intercase_n_3__Create Purchase Order Item_Block Purchase Order Item', 'intercase_n_3__Create Purchase Order Item_Block Purchase Order Item_Delete Purchase Order Item', 'intercase_n_3__Create Purchase Order Item_Block Purchase Order Item_Reactivate Purchase Order Item', 'intercase_n_3__Create Purchase Order Item_Block Purchase Order Item_Vendor creates invoice', 'intercase_n_3__Create Purchase Order Item_Cancel Goods Receipt', 'intercase_n_3__Create Purchase Order Item_Cancel Goods Receipt_Cancel Invoice Receipt', 'intercase_n_3__Create Purchase Order Item_Cancel Goods Receipt_Change Price', 'intercase_n_3__Create Purchase Order Item_Cancel Goods Receipt_Change Quantity', 'intercase_n_3__Create Purchase Order Item_Cancel Goods Receipt_Record Goods Receipt', 'intercase_n_3__Create Purchase Order Item_Cancel Goods Receipt_Release Purchase Order', 'intercase_n_3__Create Purchase Order Item_Cancel Goods Receipt_Vendor creates debit memo', 'intercase_n_3__Create Purchase Order Item_Cancel Goods Receipt_Vendor creates invoice', 'intercase_n_3__Create Purchase Order Item_Change Approval for Purchase Order', 'intercase_n_3__Create Purchase Order Item_Change Approval for Purchase Order_Change Approval for Purchase Order', 'intercase_n_3__Create Purchase Order Item_Change Approval for Purchase Order_Change Price', 'intercase_n_3__Create Purchase Order Item_Change Approval for Purchase Order_Change Quantity', 'intercase_n_3__Create Purchase Order Item_Change Approval for Purchase Order_Change Storage Location', 'intercase_n_3__Create Purchase Order Item_Change Approval for Purchase Order_Delete Purchase Order Item', 'intercase_n_3__Create Purchase Order Item_Change Approval for Purchase Order_Receive Order Confirmation', 'intercase_n_3__Create Purchase Order Item_Change Approval for Purchase Order_Record Goods Receipt', 'intercase_n_3__Create Purchase Order Item_Change Approval for Purchase Order_Record Invoice Receipt', 'intercase_n_3__Create Purchase Order Item_Change Approval for Purchase Order_Release Purchase Order', 'intercase_n_3__Create Purchase Order Item_Change Approval for Purchase Order_Vendor creates debit memo', 'intercase_n_3__Create Purchase Order Item_Change Approval for Purchase Order_Vendor creates invoice', 'intercase_n_3__Create Purchase Order Item_Change Currency', 'intercase_n_3__Create Purchase Order Item_Change Currency_Change Price', 'intercase_n_3__Create Purchase Order Item_Change Currency_Change payment term', 'intercase_n_3__Create Purchase Order Item_Change Delivery Indicator', 'intercase_n_3__Create Purchase Order Item_Change Delivery Indicator_Change Delivery Indicator', 'intercase_n_3__Create Purchase Order Item_Change Delivery Indicator_Delete Purchase Order Item', 'intercase_n_3__Create Purchase Order Item_Change Delivery Indicator_Record Goods Receipt', 'intercase_n_3__Create Purchase Order Item_Change Delivery Indicator_Vendor creates debit memo', 'intercase_n_3__Create Purchase Order Item_Change Delivery Indicator_Vendor creates invoice', 'intercase_n_3__Create Purchase Order Item_Change Final Invoice Indicator', 'intercase_n_3__Create Purchase Order Item_Change Price', 'intercase_n_3__Create Purchase Order Item_Change Price_Block Purchase Order Item', 'intercase_n_3__Create Purchase Order Item_Change Price_Cancel Goods Receipt', 'intercase_n_3__Create Purchase Order Item_Change Price_Change Approval for Purchase Order', 'intercase_n_3__Create Purchase Order Item_Change Price_Change Currency', 'intercase_n_3__Create Purchase Order Item_Change Price_Change Delivery Indicator', 'intercase_n_3__Create Purchase Order Item_Change Price_Change Price', 'intercase_n_3__Create Purchase Order Item_Change Price_Change Quantity', 'intercase_n_3__Create Purchase Order Item_Change Price_Change Storage Location', 'intercase_n_3__Create Purchase Order Item_Change Price_Change payment term', 'intercase_n_3__Create Purchase Order Item_Change Price_Receive Order Confirmation', 'intercase_n_3__Create Purchase Order Item_Change Price_Record Goods Receipt', 'intercase_n_3__Create Purchase Order Item_Change Price_Record Invoice Receipt', 'intercase_n_3__Create Purchase Order Item_Change Price_Record Service Entry Sheet', 'intercase_n_3__Create Purchase Order Item_Change Price_Release Purchase Order', 'intercase_n_3__Create Purchase Order Item_Change Price_Vendor creates debit memo', 'intercase_n_3__Create Purchase Order Item_Change Price_Vendor creates invoice', 'intercase_n_3__Create Purchase Order Item_Change Quantity', 'intercase_n_3__Create Purchase Order Item_Change Quantity_Block Purchase Order Item', 'intercase_n_3__Create Purchase Order Item_Change Quantity_Cancel Goods Receipt', 'intercase_n_3__Create Purchase Order Item_Change Quantity_Change Approval for Purchase Order', 'intercase_n_3__Create Purchase Order Item_Change Quantity_Change Delivery Indicator', 'intercase_n_3__Create Purchase Order Item_Change Quantity_Change Price', 'intercase_n_3__Create Purchase Order Item_Change Quantity_Change Quantity', 'intercase_n_3__Create Purchase Order Item_Change Quantity_Change Storage Location', 'intercase_n_3__Create Purchase Order Item_Change Quantity_Delete Purchase Order Item', 'intercase_n_3__Create Purchase Order Item_Change Quantity_Receive Order Confirmation', 'intercase_n_3__Create Purchase Order Item_Change Quantity_Record Goods Receipt', 'intercase_n_3__Create Purchase Order Item_Change Quantity_Record Invoice Receipt', 'intercase_n_3__Create Purchase Order Item_Change Quantity_Release Purchase Order', 'intercase_n_3__Create Purchase Order Item_Change Quantity_Vendor creates debit memo', 'intercase_n_3__Create Purchase Order Item_Change Quantity_Vendor creates invoice', 'intercase_n_3__Create Purchase Order Item_Change Storage Location', 'intercase_n_3__Create Purchase Order Item_Change Storage Location_Cancel Goods Receipt', 'intercase_n_3__Create Purchase Order Item_Change Storage Location_Change Delivery Indicator', 'intercase_n_3__Create Purchase Order Item_Change Storage Location_Change Price', 'intercase_n_3__Create Purchase Order Item_Change Storage Location_Change Quantity', 'intercase_n_3__Create Purchase Order Item_Change Storage Location_Change Storage Location', 'intercase_n_3__Create Purchase Order Item_Change Storage Location_Delete Purchase Order Item', 'intercase_n_3__Create Purchase Order Item_Change Storage Location_Receive Order Confirmation', 'intercase_n_3__Create Purchase Order Item_Change Storage Location_Record Goods Receipt', 'intercase_n_3__Create Purchase Order Item_Change Storage Location_Vendor creates invoice', 'intercase_n_3__Create Purchase Order Item_Change payment term', 'intercase_n_3__Create Purchase Order Item_Change payment term_Change Storage Location', 'intercase_n_3__Create Purchase Order Item_Change payment term_Record Goods Receipt', 'intercase_n_3__Create Purchase Order Item_Delete Purchase Order Item', 'intercase_n_3__Create Purchase Order Item_Delete Purchase Order Item_Change Delivery Indicator', 'intercase_n_3__Create Purchase Order Item_Delete Purchase Order Item_Change Quantity', 'intercase_n_3__Create Purchase Order Item_Delete Purchase Order Item_Change Rejection Indicator', 'intercase_n_3__Create Purchase Order Item_Delete Purchase Order Item_Reactivate Purchase Order Item', 'intercase_n_3__Create Purchase Order Item_Delete Purchase Order Item_Release Purchase Order', 'intercase_n_3__Create Purchase Order Item_Delete Purchase Order Item_Vendor creates invoice', 'intercase_n_3__Create Purchase Order Item_Receive Order Confirmation', 'intercase_n_3__Create Purchase Order Item_Receive Order Confirmation_Change Approval for Purchase Order', 'intercase_n_3__Create Purchase Order Item_Receive Order Confirmation_Change Delivery Indicator', 'intercase_n_3__Create Purchase Order Item_Receive Order Confirmation_Change Price', 'intercase_n_3__Create Purchase Order Item_Receive Order Confirmation_Change Quantity', 'intercase_n_3__Create Purchase Order Item_Receive Order Confirmation_Change Storage Location', 'intercase_n_3__Create Purchase Order Item_Receive Order Confirmation_Delete Purchase Order Item', 'intercase_n_3__Create Purchase Order Item_Receive Order Confirmation_Record Goods Receipt', 'intercase_n_3__Create Purchase Order Item_Receive Order Confirmation_Record Invoice Receipt', 'intercase_n_3__Create Purchase Order Item_Receive Order Confirmation_Update Order Confirmation', 'intercase_n_3__Create Purchase Order Item_Receive Order Confirmation_Vendor creates debit memo', 'intercase_n_3__Create Purchase Order Item_Receive Order Confirmation_Vendor creates invoice', 'intercase_n_3__Create Purchase Order Item_Record Goods Receipt', 'intercase_n_3__Create Purchase Order Item_Record Goods Receipt_Block Purchase Order Item', 'intercase_n_3__Create Purchase Order Item_Record Goods Receipt_Cancel Goods Receipt', 'intercase_n_3__Create Purchase Order Item_Record Goods Receipt_Change Approval for Purchase Order', 'intercase_n_3__Create Purchase Order Item_Record Goods Receipt_Change Delivery Indicator', 'intercase_n_3__Create Purchase Order Item_Record Goods Receipt_Change Price', 'intercase_n_3__Create Purchase Order Item_Record Goods Receipt_Change Quantity', 'intercase_n_3__Create Purchase Order Item_Record Goods Receipt_Change Storage Location', 'intercase_n_3__Create Purchase Order Item_Record Goods Receipt_Clear Invoice', 'intercase_n_3__Create Purchase Order Item_Record Goods Receipt_Delete Purchase Order Item', 'intercase_n_3__Create Purchase Order Item_Record Goods Receipt_Receive Order Confirmation', 'intercase_n_3__Create Purchase Order Item_Record Goods Receipt_Record Goods Receipt', 'intercase_n_3__Create Purchase Order Item_Record Goods Receipt_Record Invoice Receipt', 'intercase_n_3__Create Purchase Order Item_Record Goods Receipt_Record Service Entry Sheet', 'intercase_n_3__Create Purchase Order Item_Record Goods Receipt_Release Purchase Order', 'intercase_n_3__Create Purchase Order Item_Record Goods Receipt_Vendor creates debit memo', 'intercase_n_3__Create Purchase Order Item_Record Goods Receipt_Vendor creates invoice', 'intercase_n_3__Create Purchase Order Item_Record Invoice Receipt', 'intercase_n_3__Create Purchase Order Item_Record Invoice Receipt_Change Approval for Purchase Order', 'intercase_n_3__Create Purchase Order Item_Record Invoice Receipt_Change Quantity', 'intercase_n_3__Create Purchase Order Item_Record Invoice Receipt_Clear Invoice', 'intercase_n_3__Create Purchase Order Item_Record Invoice Receipt_Record Goods Receipt', 'intercase_n_3__Create Purchase Order Item_Record Invoice Receipt_Remove Payment Block', 'intercase_n_3__Create Purchase Order Item_Record Invoice Receipt_Vendor creates debit memo', 'intercase_n_3__Create Purchase Order Item_Record Invoice Receipt_Vendor creates invoice', 'intercase_n_3__Create Purchase Order Item_Record Service Entry Sheet', 'intercase_n_3__Create Purchase Order Item_Record Service Entry Sheet_Record Goods Receipt', 'intercase_n_3__Create Purchase Order Item_Record Service Entry Sheet_Record Service Entry Sheet', 'intercase_n_3__Create Purchase Order Item_Record Service Entry Sheet_Vendor creates invoice', 'intercase_n_3__Create Purchase Order Item_Release Purchase Order', 'intercase_n_3__Create Purchase Order Item_Release Purchase Order_Change Approval for Purchase Order', 'intercase_n_3__Create Purchase Order Item_Release Purchase Order_Change Quantity', 'intercase_n_3__Create Purchase Order Item_Release Purchase Order_Delete Purchase Order Item', 'intercase_n_3__Create Purchase Order Item_Release Purchase Order_Vendor creates invoice', 'intercase_n_3__Create Purchase Order Item_SRM: Change was Transmitted_Record Goods Receipt', 'intercase_n_3__Create Purchase Order Item_SRM: Change was Transmitted_Record Invoice Receipt', 'intercase_n_3__Create Purchase Order Item_SRM: Change was Transmitted_Record Service Entry Sheet', 'intercase_n_3__Create Purchase Order Item_SRM: Change was Transmitted_SRM: Complete', 'intercase_n_3__Create Purchase Order Item_SRM: Change was Transmitted_SRM: Created', 'intercase_n_3__Create Purchase Order Item_SRM: Change was Transmitted_SRM: Ordered', 'intercase_n_3__Create Purchase Order Item_SRM: Change was Transmitted_Vendor creates debit memo', 'intercase_n_3__Create Purchase Order Item_SRM: Change was Transmitted_Vendor creates invoice', 'intercase_n_3__Create Purchase Order Item_SRM: Complete_SRM: Awaiting Approval', 'intercase_n_3__Create Purchase Order Item_SRM: Created', 'intercase_n_3__Create Purchase Order Item_SRM: Created_SRM: Complete', 'intercase_n_3__Create Purchase Order Item_SRM: Created_SRM: Created', 'intercase_n_3__Create Purchase Order Item_SRM: In Transfer to Execution Syst._Record Goods Receipt', 'intercase_n_3__Create Purchase Order Item_SRM: In Transfer to Execution Syst._SRM: Change was Transmitted', 'intercase_n_3__Create Purchase Order Item_SRM: In Transfer to Execution Syst._SRM: Created', 'intercase_n_3__Create Purchase Order Item_SRM: In Transfer to Execution Syst._SRM: Ordered', 'intercase_n_3__Create Purchase Order Item_SRM: In Transfer to Execution Syst._Vendor creates invoice', 'intercase_n_3__Create Purchase Order Item_SRM: Ordered_SRM: Change was Transmitted', 'intercase_n_3__Create Purchase Order Item_SRM: Ordered_SRM: In Transfer to Execution Syst.', 'intercase_n_3__Create Purchase Order Item_Update Order Confirmation', 'intercase_n_3__Create Purchase Order Item_Update Order Confirmation_Change Quantity', 'intercase_n_3__Create Purchase Order Item_Vendor creates debit memo', 'intercase_n_3__Create Purchase Order Item_Vendor creates debit memo_Cancel Goods Receipt', 'intercase_n_3__Create Purchase Order Item_Vendor creates debit memo_Cancel Invoice Receipt', 'intercase_n_3__Create Purchase Order Item_Vendor creates debit memo_Record Goods Receipt', 'intercase_n_3__Create Purchase Order Item_Vendor creates debit memo_Vendor creates invoice', 'intercase_n_3__Create Purchase Order Item_Vendor creates invoice', 'intercase_n_3__Create Purchase Order Item_Vendor creates invoice_Block Purchase Order Item', 'intercase_n_3__Create Purchase Order Item_Vendor creates invoice_Change Approval for Purchase Order', 'intercase_n_3__Create Purchase Order Item_Vendor creates invoice_Change Delivery Indicator', 'intercase_n_3__Create Purchase Order Item_Vendor creates invoice_Change Price', 'intercase_n_3__Create Purchase Order Item_Vendor creates invoice_Change Quantity', 'intercase_n_3__Create Purchase Order Item_Vendor creates invoice_Change Storage Location', 'intercase_n_3__Create Purchase Order Item_Vendor creates invoice_Delete Purchase Order Item', 'intercase_n_3__Create Purchase Order Item_Vendor creates invoice_Receive Order Confirmation', 'intercase_n_3__Create Purchase Order Item_Vendor creates invoice_Record Goods Receipt', 'intercase_n_3__Create Purchase Order Item_Vendor creates invoice_Record Invoice Receipt', 'intercase_n_3__Create Purchase Order Item_Vendor creates invoice_Record Service Entry Sheet', 'intercase_n_3__Create Purchase Order Item_Vendor creates invoice_Release Purchase Order', 'intercase_n_3__Create Purchase Order Item_Vendor creates invoice_SRM: Created', 'intercase_n_3__Create Purchase Order Item_Vendor creates invoice_SRM: In Transfer to Execution Syst.', 'intercase_n_3__Create Purchase Order Item_Vendor creates invoice_Vendor creates debit memo', 'intercase_n_3__Create Purchase Order Item_Vendor creates invoice_Vendor creates invoice', 'intercase_n_3__Create Purchase Requisition Item', 'intercase_n_3__Create Purchase Requisition Item_Create Purchase Order Item', 'intercase_n_3__Create Purchase Requisition Item_Create Purchase Order Item_Change Approval for Purchase Order', 'intercase_n_3__Create Purchase Requisition Item_Create Purchase Order Item_Change Delivery Indicator', 'intercase_n_3__Create Purchase Requisition Item_Create Purchase Order Item_Change Price', 'intercase_n_3__Create Purchase Requisition Item_Create Purchase Order Item_Change Quantity', 'intercase_n_3__Create Purchase Requisition Item_Create Purchase Order Item_Change Storage Location', 'intercase_n_3__Create Purchase Requisition Item_Create Purchase Order Item_Change payment term', 'intercase_n_3__Create Purchase Requisition Item_Create Purchase Order Item_Delete Purchase Order Item', 'intercase_n_3__Create Purchase Requisition Item_Create Purchase Order Item_Receive Order Confirmation', 'intercase_n_3__Create Purchase Requisition Item_Create Purchase Order Item_Record Goods Receipt', 'intercase_n_3__Create Purchase Requisition Item_Create Purchase Order Item_Record Invoice Receipt', 'intercase_n_3__Create Purchase Requisition Item_Create Purchase Order Item_Release Purchase Order', 'intercase_n_3__Create Purchase Requisition Item_Create Purchase Order Item_Vendor creates debit memo', 'intercase_n_3__Create Purchase Requisition Item_Create Purchase Order Item_Vendor creates invoice', 'intercase_n_3__Create Purchase Requisition Item_Release Purchase Requisition', 'intercase_n_3__Create Purchase Requisition Item_Release Purchase Requisition_Create Purchase Order Item', 'intercase_n_3__Delete Purchase Order Item_Change Approval for Purchase Order_Change Quantity', 'intercase_n_3__Delete Purchase Order Item_Change Approval for Purchase Order_Release Purchase Order', 'intercase_n_3__Delete Purchase Order Item_Change Delivery Indicator_Change Delivery Indicator', 'intercase_n_3__Delete Purchase Order Item_Change Quantity_Reactivate Purchase Order Item', 'intercase_n_3__Delete Purchase Order Item_Change Rejection Indicator_Change Rejection Indicator', 'intercase_n_3__Delete Purchase Order Item_Reactivate Purchase Order Item_Cancel Goods Receipt', 'intercase_n_3__Delete Purchase Order Item_Reactivate Purchase Order Item_Change Delivery Indicator', 'intercase_n_3__Delete Purchase Order Item_Reactivate Purchase Order Item_Change Price', 'intercase_n_3__Delete Purchase Order Item_Reactivate Purchase Order Item_Change Quantity', 'intercase_n_3__Delete Purchase Order Item_Reactivate Purchase Order Item_Change Storage Location', 'intercase_n_3__Delete Purchase Order Item_Reactivate Purchase Order Item_Clear Invoice', 'intercase_n_3__Delete Purchase Order Item_Reactivate Purchase Order Item_Delete Purchase Order Item', 'intercase_n_3__Delete Purchase Order Item_Reactivate Purchase Order Item_Receive Order Confirmation', 'intercase_n_3__Delete Purchase Order Item_Reactivate Purchase Order Item_Record Goods Receipt', 'intercase_n_3__Delete Purchase Order Item_Reactivate Purchase Order Item_Record Invoice Receipt', 'intercase_n_3__Delete Purchase Order Item_Reactivate Purchase Order Item_Remove Payment Block', 'intercase_n_3__Delete Purchase Order Item_Reactivate Purchase Order Item_Vendor creates invoice', 'intercase_n_3__Delete Purchase Order Item_Release Purchase Order_Change Approval for Purchase Order', 'intercase_n_3__Delete Purchase Order Item_SRM: In Transfer to Execution Syst._SRM: Transaction Completed', 'intercase_n_3__Delete Purchase Order Item_Vendor creates invoice_Reactivate Purchase Order Item', 'intercase_n_3__Delete Purchase Order Item_Vendor creates invoice_Vendor creates debit memo', 'intercase_n_3__Reactivate Purchase Order Item_Block Purchase Order Item_Reactivate Purchase Order Item', 'intercase_n_3__Reactivate Purchase Order Item_Block Purchase Order Item_Vendor creates invoice', 'intercase_n_3__Reactivate Purchase Order Item_Cancel Goods Receipt_Change Delivery Indicator', 'intercase_n_3__Reactivate Purchase Order Item_Change Delivery Indicator_Change Delivery Indicator', 'intercase_n_3__Reactivate Purchase Order Item_Change Price_Change Currency', 'intercase_n_3__Reactivate Purchase Order Item_Change Price_Change Delivery Indicator', 'intercase_n_3__Reactivate Purchase Order Item_Change Price_Change Price', 'intercase_n_3__Reactivate Purchase Order Item_Change Price_Change Quantity', 'intercase_n_3__Reactivate Purchase Order Item_Change Price_Record Goods Receipt', 'intercase_n_3__Reactivate Purchase Order Item_Change Quantity_Block Purchase Order Item', 'intercase_n_3__Reactivate Purchase Order Item_Change Quantity_Change Delivery Indicator', 'intercase_n_3__Reactivate Purchase Order Item_Change Quantity_Change Quantity', 'intercase_n_3__Reactivate Purchase Order Item_Change Quantity_Record Goods Receipt', 'intercase_n_3__Reactivate Purchase Order Item_Change Quantity_Record Invoice Receipt', 'intercase_n_3__Reactivate Purchase Order Item_Change Quantity_Remove Payment Block', 'intercase_n_3__Reactivate Purchase Order Item_Change Quantity_Vendor creates debit memo', 'intercase_n_3__Reactivate Purchase Order Item_Change Quantity_Vendor creates invoice', 'intercase_n_3__Reactivate Purchase Order Item_Change Storage Location_Record Goods Receipt', 'intercase_n_3__Reactivate Purchase Order Item_Clear Invoice_Vendor creates invoice', 'intercase_n_3__Reactivate Purchase Order Item_Delete Purchase Order Item_Reactivate Purchase Order Item', 'intercase_n_3__Reactivate Purchase Order Item_Delete Purchase Order Item_Vendor creates invoice', 'intercase_n_3__Reactivate Purchase Order Item_Receive Order Confirmation_Record Goods Receipt', 'intercase_n_3__Reactivate Purchase Order Item_Receive Order Confirmation_Vendor creates invoice', 'intercase_n_3__Reactivate Purchase Order Item_Record Goods Receipt_Cancel Goods Receipt', 'intercase_n_3__Reactivate Purchase Order Item_Record Goods Receipt_Change Quantity', 'intercase_n_3__Reactivate Purchase Order Item_Record Goods Receipt_Clear Invoice', 'intercase_n_3__Reactivate Purchase Order Item_Record Goods Receipt_Record Goods Receipt', 'intercase_n_3__Reactivate Purchase Order Item_Record Goods Receipt_Record Invoice Receipt', 'intercase_n_3__Reactivate Purchase Order Item_Record Goods Receipt_Remove Payment Block', 'intercase_n_3__Reactivate Purchase Order Item_Record Goods Receipt_Vendor creates debit memo', 'intercase_n_3__Reactivate Purchase Order Item_Record Goods Receipt_Vendor creates invoice', 'intercase_n_3__Reactivate Purchase Order Item_Record Invoice Receipt_Change Delivery Indicator', 'intercase_n_3__Reactivate Purchase Order Item_Record Invoice Receipt_Change Quantity', 'intercase_n_3__Reactivate Purchase Order Item_Record Invoice Receipt_Record Goods Receipt', 'intercase_n_3__Reactivate Purchase Order Item_Record Invoice Receipt_Remove Payment Block', 'intercase_n_3__Reactivate Purchase Order Item_Record Invoice Receipt_Vendor creates invoice', 'intercase_n_3__Reactivate Purchase Order Item_Vendor creates invoice_Change Quantity', 'intercase_n_3__Reactivate Purchase Order Item_Vendor creates invoice_Record Goods Receipt', 'intercase_n_3__Reactivate Purchase Order Item_Vendor creates invoice_Record Invoice Receipt', 'intercase_n_3__Reactivate Purchase Order Item_Vendor creates invoice_Vendor creates invoice', 'intercase_n_3__Receive Order Confirmation_Change Approval for Purchase Order_Vendor creates invoice', 'intercase_n_3__Receive Order Confirmation_Change Delivery Indicator_Change Quantity', 'intercase_n_3__Receive Order Confirmation_Change Delivery Indicator_Record Goods Receipt', 'intercase_n_3__Receive Order Confirmation_Change Delivery Indicator_Remove Payment Block', 'intercase_n_3__Receive Order Confirmation_Change Price_Change Price', 'intercase_n_3__Receive Order Confirmation_Change Price_Change Quantity', 'intercase_n_3__Receive Order Confirmation_Change Price_Record Goods Receipt', 'intercase_n_3__Receive Order Confirmation_Change Price_Record Invoice Receipt', 'intercase_n_3__Receive Order Confirmation_Change Price_Update Order Confirmation', 'intercase_n_3__Receive Order Confirmation_Change Price_Vendor creates invoice', 'intercase_n_3__Receive Order Confirmation_Change Quantity_Change Delivery Indicator', 'intercase_n_3__Receive Order Confirmation_Change Quantity_Change Price', 'intercase_n_3__Receive Order Confirmation_Change Quantity_Change Quantity', 'intercase_n_3__Receive Order Confirmation_Change Quantity_Change Storage Location', 'intercase_n_3__Receive Order Confirmation_Change Quantity_Delete Purchase Order Item', 'intercase_n_3__Receive Order Confirmation_Change Quantity_Record Goods Receipt', 'intercase_n_3__Receive Order Confirmation_Change Quantity_Record Invoice Receipt', 'intercase_n_3__Receive Order Confirmation_Change Quantity_Update Order Confirmation', 'intercase_n_3__Receive Order Confirmation_Change Quantity_Vendor creates debit memo', 'intercase_n_3__Receive Order Confirmation_Change Quantity_Vendor creates invoice', 'intercase_n_3__Receive Order Confirmation_Change Storage Location_Record Goods Receipt', 'intercase_n_3__Receive Order Confirmation_Change Storage Location_Vendor creates invoice', 'intercase_n_3__Receive Order Confirmation_Delete Purchase Order Item_Reactivate Purchase Order Item', 'intercase_n_3__Receive Order Confirmation_Delete Purchase Order Item_Vendor creates invoice', 'intercase_n_3__Receive Order Confirmation_Record Goods Receipt_Cancel Goods Receipt', 'intercase_n_3__Receive Order Confirmation_Record Goods Receipt_Change Delivery Indicator', 'intercase_n_3__Receive Order Confirmation_Record Goods Receipt_Change Price', 'intercase_n_3__Receive Order Confirmation_Record Goods Receipt_Change Quantity', 'intercase_n_3__Receive Order Confirmation_Record Goods Receipt_Record Goods Receipt', 'intercase_n_3__Receive Order Confirmation_Record Goods Receipt_Record Invoice Receipt', 'intercase_n_3__Receive Order Confirmation_Record Goods Receipt_Release Purchase Order', 'intercase_n_3__Receive Order Confirmation_Record Goods Receipt_Remove Payment Block', 'intercase_n_3__Receive Order Confirmation_Record Goods Receipt_Update Order Confirmation', 'intercase_n_3__Receive Order Confirmation_Record Goods Receipt_Vendor creates debit memo', 'intercase_n_3__Receive Order Confirmation_Record Goods Receipt_Vendor creates invoice', 'intercase_n_3__Receive Order Confirmation_Record Invoice Receipt_Cancel Invoice Receipt', 'intercase_n_3__Receive Order Confirmation_Record Invoice Receipt_Change Price', 'intercase_n_3__Receive Order Confirmation_Record Invoice Receipt_Change Quantity', 'intercase_n_3__Receive Order Confirmation_Record Invoice Receipt_Clear Invoice', 'intercase_n_3__Receive Order Confirmation_Record Invoice Receipt_Record Goods Receipt', 'intercase_n_3__Receive Order Confirmation_Record Invoice Receipt_Remove Payment Block', 'intercase_n_3__Receive Order Confirmation_Record Invoice Receipt_Vendor creates invoice', 'intercase_n_3__Receive Order Confirmation_Update Order Confirmation_Change Price', 'intercase_n_3__Receive Order Confirmation_Update Order Confirmation_Change Quantity', 'intercase_n_3__Receive Order Confirmation_Update Order Confirmation_Delete Purchase Order Item', 'intercase_n_3__Receive Order Confirmation_Update Order Confirmation_Receive Order Confirmation', 'intercase_n_3__Receive Order Confirmation_Update Order Confirmation_Record Goods Receipt', 'intercase_n_3__Receive Order Confirmation_Update Order Confirmation_Record Invoice Receipt', 'intercase_n_3__Receive Order Confirmation_Update Order Confirmation_Update Order Confirmation', 'intercase_n_3__Receive Order Confirmation_Update Order Confirmation_Vendor creates debit memo', 'intercase_n_3__Receive Order Confirmation_Update Order Confirmation_Vendor creates invoice', 'intercase_n_3__Receive Order Confirmation_Vendor creates debit memo_Vendor creates invoice', 'intercase_n_3__Receive Order Confirmation_Vendor creates invoice_Change Delivery Indicator', 'intercase_n_3__Receive Order Confirmation_Vendor creates invoice_Change Price', 'intercase_n_3__Receive Order Confirmation_Vendor creates invoice_Change Quantity', 'intercase_n_3__Receive Order Confirmation_Vendor creates invoice_Record Goods Receipt', 'intercase_n_3__Receive Order Confirmation_Vendor creates invoice_Record Invoice Receipt', 'intercase_n_3__Receive Order Confirmation_Vendor creates invoice_Update Order Confirmation', 'intercase_n_3__Receive Order Confirmation_Vendor creates invoice_Vendor creates debit memo', 'intercase_n_3__Receive Order Confirmation_Vendor creates invoice_Vendor creates invoice', 'intercase_n_3__Record Goods Receipt_Block Purchase Order Item_Reactivate Purchase Order Item', 'intercase_n_3__Record Goods Receipt_Block Purchase Order Item_Vendor creates debit memo', 'intercase_n_3__Record Goods Receipt_Block Purchase Order Item_Vendor creates invoice', 'intercase_n_3__Record Goods Receipt_Cancel Goods Receipt_Block Purchase Order Item', 'intercase_n_3__Record Goods Receipt_Cancel Goods Receipt_Cancel Goods Receipt', 'intercase_n_3__Record Goods Receipt_Cancel Goods Receipt_Change Delivery Indicator', 'intercase_n_3__Record Goods Receipt_Cancel Goods Receipt_Change Price', 'intercase_n_3__Record Goods Receipt_Cancel Goods Receipt_Change Quantity', 'intercase_n_3__Record Goods Receipt_Cancel Goods Receipt_Change Storage Location', 'intercase_n_3__Record Goods Receipt_Cancel Goods Receipt_Clear Invoice', 'intercase_n_3__Record Goods Receipt_Cancel Goods Receipt_Delete Purchase Order Item', 'intercase_n_3__Record Goods Receipt_Cancel Goods Receipt_Record Goods Receipt', 'intercase_n_3__Record Goods Receipt_Cancel Goods Receipt_Record Invoice Receipt', 'intercase_n_3__Record Goods Receipt_Cancel Goods Receipt_Record Service Entry Sheet', 'intercase_n_3__Record Goods Receipt_Cancel Goods Receipt_Release Purchase Order', 'intercase_n_3__Record Goods Receipt_Cancel Goods Receipt_Remove Payment Block', 'intercase_n_3__Record Goods Receipt_Cancel Goods Receipt_SRM: Created', 'intercase_n_3__Record Goods Receipt_Cancel Goods Receipt_Vendor creates debit memo', 'intercase_n_3__Record Goods Receipt_Cancel Goods Receipt_Vendor creates invoice', 'intercase_n_3__Record Goods Receipt_Cancel Invoice Receipt_Cancel Invoice Receipt', 'intercase_n_3__Record Goods Receipt_Cancel Invoice Receipt_Clear Invoice', 'intercase_n_3__Record Goods Receipt_Cancel Invoice Receipt_Record Invoice Receipt', 'intercase_n_3__Record Goods Receipt_Cancel Invoice Receipt_Remove Payment Block', 'intercase_n_3__Record Goods Receipt_Cancel Subsequent Invoice_Remove Payment Block', 'intercase_n_3__Record Goods Receipt_Change Approval for Purchase Order_Cancel Goods Receipt', 'intercase_n_3__Record Goods Receipt_Change Approval for Purchase Order_Change Quantity', 'intercase_n_3__Record Goods Receipt_Change Approval for Purchase Order_Record Invoice Receipt', 'intercase_n_3__Record Goods Receipt_Change Approval for Purchase Order_Release Purchase Order', 'intercase_n_3__Record Goods Receipt_Change Delivery Indicator_Cancel Goods Receipt', 'intercase_n_3__Record Goods Receipt_Change Delivery Indicator_Cancel Invoice Receipt', 'intercase_n_3__Record Goods Receipt_Change Delivery Indicator_Change Delivery Indicator', 'intercase_n_3__Record Goods Receipt_Change Delivery Indicator_Change Price', 'intercase_n_3__Record Goods Receipt_Change Delivery Indicator_Change Quantity', 'intercase_n_3__Record Goods Receipt_Change Delivery Indicator_Clear Invoice', 'intercase_n_3__Record Goods Receipt_Change Delivery Indicator_Record Goods Receipt', 'intercase_n_3__Record Goods Receipt_Change Delivery Indicator_Record Invoice Receipt', 'intercase_n_3__Record Goods Receipt_Change Delivery Indicator_Remove Payment Block', 'intercase_n_3__Record Goods Receipt_Change Delivery Indicator_Vendor creates debit memo', 'intercase_n_3__Record Goods Receipt_Change Delivery Indicator_Vendor creates invoice', 'intercase_n_3__Record Goods Receipt_Change Price_Cancel Goods Receipt', 'intercase_n_3__Record Goods Receipt_Change Price_Cancel Invoice Receipt', 'intercase_n_3__Record Goods Receipt_Change Price_Change Price', 'intercase_n_3__Record Goods Receipt_Change Price_Change Quantity', 'intercase_n_3__Record Goods Receipt_Change Price_Clear Invoice', 'intercase_n_3__Record Goods Receipt_Change Price_Receive Order Confirmation', 'intercase_n_3__Record Goods Receipt_Change Price_Record Goods Receipt', 'intercase_n_3__Record Goods Receipt_Change Price_Record Invoice Receipt', 'intercase_n_3__Record Goods Receipt_Change Price_Record Service Entry Sheet', 'intercase_n_3__Record Goods Receipt_Change Price_Release Purchase Order', 'intercase_n_3__Record Goods Receipt_Change Price_Remove Payment Block', 'intercase_n_3__Record Goods Receipt_Change Price_Vendor creates debit memo', 'intercase_n_3__Record Goods Receipt_Change Price_Vendor creates invoice', 'intercase_n_3__Record Goods Receipt_Change Quantity_Block Purchase Order Item', 'intercase_n_3__Record Goods Receipt_Change Quantity_Cancel Goods Receipt', 'intercase_n_3__Record Goods Receipt_Change Quantity_Cancel Invoice Receipt', 'intercase_n_3__Record Goods Receipt_Change Quantity_Change Delivery Indicator', 'intercase_n_3__Record Goods Receipt_Change Quantity_Change Price', 'intercase_n_3__Record Goods Receipt_Change Quantity_Change Quantity', 'intercase_n_3__Record Goods Receipt_Change Quantity_Clear Invoice', 'intercase_n_3__Record Goods Receipt_Change Quantity_Record Goods Receipt', 'intercase_n_3__Record Goods Receipt_Change Quantity_Record Invoice Receipt', 'intercase_n_3__Record Goods Receipt_Change Quantity_Remove Payment Block', 'intercase_n_3__Record Goods Receipt_Change Quantity_Vendor creates debit memo', 'intercase_n_3__Record Goods Receipt_Change Quantity_Vendor creates invoice', 'intercase_n_3__Record Goods Receipt_Change Storage Location_Change Storage Location', 'intercase_n_3__Record Goods Receipt_Change Storage Location_Record Invoice Receipt', 'intercase_n_3__Record Goods Receipt_Change Storage Location_Vendor creates invoice', 'intercase_n_3__Record Goods Receipt_Clear Invoice_Cancel Goods Receipt', 'intercase_n_3__Record Goods Receipt_Clear Invoice_Cancel Invoice Receipt', 'intercase_n_3__Record Goods Receipt_Clear Invoice_Clear Invoice', 'intercase_n_3__Record Goods Receipt_Clear Invoice_Record Goods Receipt', 'intercase_n_3__Record Goods Receipt_Clear Invoice_Record Invoice Receipt', 'intercase_n_3__Record Goods Receipt_Clear Invoice_Record Service Entry Sheet', 'intercase_n_3__Record Goods Receipt_Clear Invoice_Remove Payment Block', 'intercase_n_3__Record Goods Receipt_Clear Invoice_Vendor creates debit memo', 'intercase_n_3__Record Goods Receipt_Clear Invoice_Vendor creates invoice', 'intercase_n_3__Record Goods Receipt_Delete Purchase Order Item_Remove Payment Block', 'intercase_n_3__Record Goods Receipt_Receive Order Confirmation_Record Invoice Receipt', 'intercase_n_3__Record Goods Receipt_Receive Order Confirmation_Update Order Confirmation', 'intercase_n_3__Record Goods Receipt_Receive Order Confirmation_Vendor creates invoice', 'intercase_n_3__Record Goods Receipt_Record Goods Receipt_Cancel Goods Receipt', 'intercase_n_3__Record Goods Receipt_Record Goods Receipt_Cancel Invoice Receipt', 'intercase_n_3__Record Goods Receipt_Record Goods Receipt_Change Delivery Indicator', 'intercase_n_3__Record Goods Receipt_Record Goods Receipt_Change Price', 'intercase_n_3__Record Goods Receipt_Record Goods Receipt_Change Quantity', 'intercase_n_3__Record Goods Receipt_Record Goods Receipt_Clear Invoice', 'intercase_n_3__Record Goods Receipt_Record Goods Receipt_Receive Order Confirmation', 'intercase_n_3__Record Goods Receipt_Record Goods Receipt_Record Goods Receipt', 'intercase_n_3__Record Goods Receipt_Record Goods Receipt_Record Invoice Receipt', 'intercase_n_3__Record Goods Receipt_Record Goods Receipt_Record Service Entry Sheet', 'intercase_n_3__Record Goods Receipt_Record Goods Receipt_Remove Payment Block', 'intercase_n_3__Record Goods Receipt_Record Goods Receipt_Vendor creates debit memo', 'intercase_n_3__Record Goods Receipt_Record Goods Receipt_Vendor creates invoice', 'intercase_n_3__Record Goods Receipt_Record Invoice Receipt_Block Purchase Order Item', 'intercase_n_3__Record Goods Receipt_Record Invoice Receipt_Cancel Goods Receipt', 'intercase_n_3__Record Goods Receipt_Record Invoice Receipt_Cancel Invoice Receipt', 'intercase_n_3__Record Goods Receipt_Record Invoice Receipt_Cancel Subsequent Invoice', 'intercase_n_3__Record Goods Receipt_Record Invoice Receipt_Change Approval for Purchase Order', 'intercase_n_3__Record Goods Receipt_Record Invoice Receipt_Change Delivery Indicator', 'intercase_n_3__Record Goods Receipt_Record Invoice Receipt_Change Price', 'intercase_n_3__Record Goods Receipt_Record Invoice Receipt_Change Quantity', 'intercase_n_3__Record Goods Receipt_Record Invoice Receipt_Change Storage Location', 'intercase_n_3__Record Goods Receipt_Record Invoice Receipt_Clear Invoice', 'intercase_n_3__Record Goods Receipt_Record Invoice Receipt_Delete Purchase Order Item', 'intercase_n_3__Record Goods Receipt_Record Invoice Receipt_Receive Order Confirmation', 'intercase_n_3__Record Goods Receipt_Record Invoice Receipt_Record Goods Receipt', 'intercase_n_3__Record Goods Receipt_Record Invoice Receipt_Record Invoice Receipt', 'intercase_n_3__Record Goods Receipt_Record Invoice Receipt_Record Service Entry Sheet', 'intercase_n_3__Record Goods Receipt_Record Invoice Receipt_Record Subsequent Invoice', 'intercase_n_3__Record Goods Receipt_Record Invoice Receipt_Release Purchase Order', 'intercase_n_3__Record Goods Receipt_Record Invoice Receipt_Remove Payment Block', 'intercase_n_3__Record Goods Receipt_Record Invoice Receipt_SRM: Created', 'intercase_n_3__Record Goods Receipt_Record Invoice Receipt_SRM: In Transfer to Execution Syst.', 'intercase_n_3__Record Goods Receipt_Record Invoice Receipt_Set Payment Block', 'intercase_n_3__Record Goods Receipt_Record Invoice Receipt_Vendor creates debit memo', 'intercase_n_3__Record Goods Receipt_Record Invoice Receipt_Vendor creates invoice', 'intercase_n_3__Record Goods Receipt_Record Service Entry Sheet_Cancel Goods Receipt', 'intercase_n_3__Record Goods Receipt_Record Service Entry Sheet_Cancel Invoice Receipt', 'intercase_n_3__Record Goods Receipt_Record Service Entry Sheet_Cancel Subsequent Invoice', 'intercase_n_3__Record Goods Receipt_Record Service Entry Sheet_Change Price', 'intercase_n_3__Record Goods Receipt_Record Service Entry Sheet_Clear Invoice', 'intercase_n_3__Record Goods Receipt_Record Service Entry Sheet_Record Goods Receipt', 'intercase_n_3__Record Goods Receipt_Record Service Entry Sheet_Record Invoice Receipt', 'intercase_n_3__Record Goods Receipt_Record Service Entry Sheet_Record Service Entry Sheet', 'intercase_n_3__Record Goods Receipt_Record Service Entry Sheet_Remove Payment Block', 'intercase_n_3__Record Goods Receipt_Record Service Entry Sheet_SRM: Complete', 'intercase_n_3__Record Goods Receipt_Record Service Entry Sheet_SRM: Created', 'intercase_n_3__Record Goods Receipt_Record Service Entry Sheet_SRM: Ordered', 'intercase_n_3__Record Goods Receipt_Record Service Entry Sheet_Vendor creates debit memo', 'intercase_n_3__Record Goods Receipt_Record Service Entry Sheet_Vendor creates invoice', 'intercase_n_3__Record Goods Receipt_Release Purchase Order_Change Approval for Purchase Order', 'intercase_n_3__Record Goods Receipt_Release Purchase Order_Change Price', 'intercase_n_3__Record Goods Receipt_Remove Payment Block_Cancel Goods Receipt', 'intercase_n_3__Record Goods Receipt_Remove Payment Block_Cancel Invoice Receipt', 'intercase_n_3__Record Goods Receipt_Remove Payment Block_Cancel Subsequent Invoice', 'intercase_n_3__Record Goods Receipt_Remove Payment Block_Change Delivery Indicator', 'intercase_n_3__Record Goods Receipt_Remove Payment Block_Change Price', 'intercase_n_3__Record Goods Receipt_Remove Payment Block_Change Quantity', 'intercase_n_3__Record Goods Receipt_Remove Payment Block_Clear Invoice', 'intercase_n_3__Record Goods Receipt_Remove Payment Block_Receive Order Confirmation', 'intercase_n_3__Record Goods Receipt_Remove Payment Block_Record Goods Receipt', 'intercase_n_3__Record Goods Receipt_Remove Payment Block_Record Invoice Receipt', 'intercase_n_3__Record Goods Receipt_Remove Payment Block_Record Subsequent Invoice', 'intercase_n_3__Record Goods Receipt_Remove Payment Block_Release Purchase Order', 'intercase_n_3__Record Goods Receipt_Remove Payment Block_Remove Payment Block', 'intercase_n_3__Record Goods Receipt_Remove Payment Block_Set Payment Block', 'intercase_n_3__Record Goods Receipt_Remove Payment Block_Update Order Confirmation', 'intercase_n_3__Record Goods Receipt_Remove Payment Block_Vendor creates debit memo', 'intercase_n_3__Record Goods Receipt_Remove Payment Block_Vendor creates invoice', 'intercase_n_3__Record Goods Receipt_SRM: Complete_SRM: Awaiting Approval', 'intercase_n_3__Record Goods Receipt_SRM: Created_SRM: Complete', 'intercase_n_3__Record Goods Receipt_SRM: Created_SRM: In Transfer to Execution Syst.', 'intercase_n_3__Record Goods Receipt_Update Order Confirmation_Change Quantity', 'intercase_n_3__Record Goods Receipt_Update Order Confirmation_Record Invoice Receipt', 'intercase_n_3__Record Goods Receipt_Update Order Confirmation_Update Order Confirmation', 'intercase_n_3__Record Goods Receipt_Update Order Confirmation_Vendor creates invoice', 'intercase_n_3__Record Goods Receipt_Vendor creates debit memo_Cancel Goods Receipt', 'intercase_n_3__Record Goods Receipt_Vendor creates debit memo_Cancel Invoice Receipt', 'intercase_n_3__Record Goods Receipt_Vendor creates debit memo_Cancel Subsequent Invoice', 'intercase_n_3__Record Goods Receipt_Vendor creates debit memo_Change Delivery Indicator', 'intercase_n_3__Record Goods Receipt_Vendor creates debit memo_Change Price', 'intercase_n_3__Record Goods Receipt_Vendor creates debit memo_Change Quantity', 'intercase_n_3__Record Goods Receipt_Vendor creates debit memo_Record Goods Receipt', 'intercase_n_3__Record Goods Receipt_Vendor creates debit memo_Record Invoice Receipt', 'intercase_n_3__Record Goods Receipt_Vendor creates debit memo_Remove Payment Block', 'intercase_n_3__Record Goods Receipt_Vendor creates debit memo_Vendor creates debit memo', 'intercase_n_3__Record Goods Receipt_Vendor creates debit memo_Vendor creates invoice', 'intercase_n_3__Record Goods Receipt_Vendor creates invoice_Block Purchase Order Item', 'intercase_n_3__Record Goods Receipt_Vendor creates invoice_Cancel Goods Receipt', 'intercase_n_3__Record Goods Receipt_Vendor creates invoice_Cancel Invoice Receipt', 'intercase_n_3__Record Goods Receipt_Vendor creates invoice_Change Approval for Purchase Order', 'intercase_n_3__Record Goods Receipt_Vendor creates invoice_Change Delivery Indicator', 'intercase_n_3__Record Goods Receipt_Vendor creates invoice_Change Price', 'intercase_n_3__Record Goods Receipt_Vendor creates invoice_Change Quantity', 'intercase_n_3__Record Goods Receipt_Vendor creates invoice_Clear Invoice', 'intercase_n_3__Record Goods Receipt_Vendor creates invoice_Receive Order Confirmation', 'intercase_n_3__Record Goods Receipt_Vendor creates invoice_Record Goods Receipt', 'intercase_n_3__Record Goods Receipt_Vendor creates invoice_Record Invoice Receipt', 'intercase_n_3__Record Goods Receipt_Vendor creates invoice_Record Service Entry Sheet', 'intercase_n_3__Record Goods Receipt_Vendor creates invoice_Release Purchase Order', 'intercase_n_3__Record Goods Receipt_Vendor creates invoice_Remove Payment Block', 'intercase_n_3__Record Goods Receipt_Vendor creates invoice_SRM: In Transfer to Execution Syst.', 'intercase_n_3__Record Goods Receipt_Vendor creates invoice_Update Order Confirmation', 'intercase_n_3__Record Goods Receipt_Vendor creates invoice_Vendor creates debit memo', 'intercase_n_3__Record Goods Receipt_Vendor creates invoice_Vendor creates invoice', 'intercase_n_3__Record Invoice Receipt_Block Purchase Order Item_Change Quantity', 'intercase_n_3__Record Invoice Receipt_Block Purchase Order Item_Delete Purchase Order Item', 'intercase_n_3__Record Invoice Receipt_Block Purchase Order Item_Reactivate Purchase Order Item', 'intercase_n_3__Record Invoice Receipt_Block Purchase Order Item_Vendor creates invoice', 'intercase_n_3__Record Invoice Receipt_Cancel Goods Receipt_Cancel Goods Receipt', 'intercase_n_3__Record Invoice Receipt_Cancel Goods Receipt_Cancel Invoice Receipt', 'intercase_n_3__Record Invoice Receipt_Cancel Goods Receipt_Change Delivery Indicator', 'intercase_n_3__Record Invoice Receipt_Cancel Goods Receipt_Change Price', 'intercase_n_3__Record Invoice Receipt_Cancel Goods Receipt_Change Quantity', 'intercase_n_3__Record Invoice Receipt_Cancel Goods Receipt_Clear Invoice', 'intercase_n_3__Record Invoice Receipt_Cancel Goods Receipt_Record Goods Receipt', 'intercase_n_3__Record Invoice Receipt_Cancel Goods Receipt_Record Invoice Receipt', 'intercase_n_3__Record Invoice Receipt_Cancel Goods Receipt_Remove Payment Block', 'intercase_n_3__Record Invoice Receipt_Cancel Goods Receipt_Vendor creates debit memo', 'intercase_n_3__Record Invoice Receipt_Cancel Goods Receipt_Vendor creates invoice', 'intercase_n_3__Record Invoice Receipt_Cancel Invoice Receipt_Cancel Goods Receipt', 'intercase_n_3__Record Invoice Receipt_Cancel Invoice Receipt_Cancel Invoice Receipt', 'intercase_n_3__Record Invoice Receipt_Cancel Invoice Receipt_Cancel Subsequent Invoice', 'intercase_n_3__Record Invoice Receipt_Cancel Invoice Receipt_Change Delivery Indicator', 'intercase_n_3__Record Invoice Receipt_Cancel Invoice Receipt_Change Price', 'intercase_n_3__Record Invoice Receipt_Cancel Invoice Receipt_Clear Invoice', 'intercase_n_3__Record Invoice Receipt_Cancel Invoice Receipt_Delete Purchase Order Item', 'intercase_n_3__Record Invoice Receipt_Cancel Invoice Receipt_Record Goods Receipt', 'intercase_n_3__Record Invoice Receipt_Cancel Invoice Receipt_Record Invoice Receipt', 'intercase_n_3__Record Invoice Receipt_Cancel Invoice Receipt_Remove Payment Block', 'intercase_n_3__Record Invoice Receipt_Cancel Invoice Receipt_Vendor creates debit memo', 'intercase_n_3__Record Invoice Receipt_Cancel Invoice Receipt_Vendor creates invoice', 'intercase_n_3__Record Invoice Receipt_Cancel Subsequent Invoice_Clear Invoice', 'intercase_n_3__Record Invoice Receipt_Cancel Subsequent Invoice_Record Goods Receipt', 'intercase_n_3__Record Invoice Receipt_Cancel Subsequent Invoice_Record Invoice Receipt', 'intercase_n_3__Record Invoice Receipt_Cancel Subsequent Invoice_Record Subsequent Invoice', 'intercase_n_3__Record Invoice Receipt_Cancel Subsequent Invoice_Remove Payment Block', 'intercase_n_3__Record Invoice Receipt_Change Approval for Purchase Order_Change Approval for Purchase Order', 'intercase_n_3__Record Invoice Receipt_Change Approval for Purchase Order_Delete Purchase Order Item', 'intercase_n_3__Record Invoice Receipt_Change Approval for Purchase Order_Release Purchase Order', 'intercase_n_3__Record Invoice Receipt_Change Delivery Indicator_Cancel Invoice Receipt', 'intercase_n_3__Record Invoice Receipt_Change Delivery Indicator_Change Delivery Indicator', 'intercase_n_3__Record Invoice Receipt_Change Delivery Indicator_Change Price', 'intercase_n_3__Record Invoice Receipt_Change Delivery Indicator_Change Quantity', 'intercase_n_3__Record Invoice Receipt_Change Delivery Indicator_Clear Invoice', 'intercase_n_3__Record Invoice Receipt_Change Delivery Indicator_Record Goods Receipt', 'intercase_n_3__Record Invoice Receipt_Change Delivery Indicator_Record Invoice Receipt', 'intercase_n_3__Record Invoice Receipt_Change Delivery Indicator_Remove Payment Block', 'intercase_n_3__Record Invoice Receipt_Change Delivery Indicator_Vendor creates debit memo', 'intercase_n_3__Record Invoice Receipt_Change Delivery Indicator_Vendor creates invoice', 'intercase_n_3__Record Invoice Receipt_Change Price_Cancel Invoice Receipt', 'intercase_n_3__Record Invoice Receipt_Change Price_Cancel Subsequent Invoice', 'intercase_n_3__Record Invoice Receipt_Change Price_Change Approval for Purchase Order', 'intercase_n_3__Record Invoice Receipt_Change Price_Change Delivery Indicator', 'intercase_n_3__Record Invoice Receipt_Change Price_Change Price', 'intercase_n_3__Record Invoice Receipt_Change Price_Change Quantity', 'intercase_n_3__Record Invoice Receipt_Change Price_Clear Invoice', 'intercase_n_3__Record Invoice Receipt_Change Price_Record Goods Receipt', 'intercase_n_3__Record Invoice Receipt_Change Price_Record Invoice Receipt', 'intercase_n_3__Record Invoice Receipt_Change Price_Record Service Entry Sheet', 'intercase_n_3__Record Invoice Receipt_Change Price_Record Subsequent Invoice', 'intercase_n_3__Record Invoice Receipt_Change Price_Remove Payment Block', 'intercase_n_3__Record Invoice Receipt_Change Price_Vendor creates debit memo', 'intercase_n_3__Record Invoice Receipt_Change Price_Vendor creates invoice', 'intercase_n_3__Record Invoice Receipt_Change Quantity_Block Purchase Order Item', 'intercase_n_3__Record Invoice Receipt_Change Quantity_Cancel Invoice Receipt', 'intercase_n_3__Record Invoice Receipt_Change Quantity_Change Delivery Indicator', 'intercase_n_3__Record Invoice Receipt_Change Quantity_Change Price', 'intercase_n_3__Record Invoice Receipt_Change Quantity_Change Quantity', 'intercase_n_3__Record Invoice Receipt_Change Quantity_Clear Invoice', 'intercase_n_3__Record Invoice Receipt_Change Quantity_Delete Purchase Order Item', 'intercase_n_3__Record Invoice Receipt_Change Quantity_Receive Order Confirmation', 'intercase_n_3__Record Invoice Receipt_Change Quantity_Record Goods Receipt', 'intercase_n_3__Record Invoice Receipt_Change Quantity_Record Invoice Receipt', 'intercase_n_3__Record Invoice Receipt_Change Quantity_Remove Payment Block', 'intercase_n_3__Record Invoice Receipt_Change Quantity_Vendor creates debit memo', 'intercase_n_3__Record Invoice Receipt_Change Quantity_Vendor creates invoice', 'intercase_n_3__Record Invoice Receipt_Change Storage Location_Change Quantity', 'intercase_n_3__Record Invoice Receipt_Change Storage Location_Change Storage Location', 'intercase_n_3__Record Invoice Receipt_Change Storage Location_Record Goods Receipt', 'intercase_n_3__Record Invoice Receipt_Clear Invoice_Block Purchase Order Item', 'intercase_n_3__Record Invoice Receipt_Clear Invoice_Cancel Goods Receipt', 'intercase_n_3__Record Invoice Receipt_Clear Invoice_Cancel Invoice Receipt', 'intercase_n_3__Record Invoice Receipt_Clear Invoice_Cancel Subsequent Invoice', 'intercase_n_3__Record Invoice Receipt_Clear Invoice_Change Approval for Purchase Order', 'intercase_n_3__Record Invoice Receipt_Clear Invoice_Change Delivery Indicator', 'intercase_n_3__Record Invoice Receipt_Clear Invoice_Change Price', 'intercase_n_3__Record Invoice Receipt_Clear Invoice_Change Quantity', 'intercase_n_3__Record Invoice Receipt_Clear Invoice_Clear Invoice', 'intercase_n_3__Record Invoice Receipt_Clear Invoice_Record Goods Receipt', 'intercase_n_3__Record Invoice Receipt_Clear Invoice_Record Invoice Receipt', 'intercase_n_3__Record Invoice Receipt_Clear Invoice_Record Service Entry Sheet', 'intercase_n_3__Record Invoice Receipt_Clear Invoice_Record Subsequent Invoice', 'intercase_n_3__Record Invoice Receipt_Clear Invoice_Remove Payment Block', 'intercase_n_3__Record Invoice Receipt_Clear Invoice_SRM: Created', 'intercase_n_3__Record Invoice Receipt_Clear Invoice_SRM: In Transfer to Execution Syst.', 'intercase_n_3__Record Invoice Receipt_Clear Invoice_Vendor creates debit memo', 'intercase_n_3__Record Invoice Receipt_Clear Invoice_Vendor creates invoice', 'intercase_n_3__Record Invoice Receipt_Delete Purchase Order Item_Reactivate Purchase Order Item', 'intercase_n_3__Record Invoice Receipt_Receive Order Confirmation_Change Delivery Indicator', 'intercase_n_3__Record Invoice Receipt_Receive Order Confirmation_Record Goods Receipt', 'intercase_n_3__Record Invoice Receipt_Receive Order Confirmation_Vendor creates invoice', 'intercase_n_3__Record Invoice Receipt_Record Goods Receipt_Cancel Goods Receipt', 'intercase_n_3__Record Invoice Receipt_Record Goods Receipt_Cancel Invoice Receipt', 'intercase_n_3__Record Invoice Receipt_Record Goods Receipt_Cancel Subsequent Invoice', 'intercase_n_3__Record Invoice Receipt_Record Goods Receipt_Change Delivery Indicator', 'intercase_n_3__Record Invoice Receipt_Record Goods Receipt_Change Price', 'intercase_n_3__Record Invoice Receipt_Record Goods Receipt_Change Quantity', 'intercase_n_3__Record Invoice Receipt_Record Goods Receipt_Clear Invoice', 'intercase_n_3__Record Invoice Receipt_Record Goods Receipt_Delete Purchase Order Item', 'intercase_n_3__Record Invoice Receipt_Record Goods Receipt_Record Goods Receipt', 'intercase_n_3__Record Invoice Receipt_Record Goods Receipt_Record Invoice Receipt', 'intercase_n_3__Record Invoice Receipt_Record Goods Receipt_Record Service Entry Sheet', 'intercase_n_3__Record Invoice Receipt_Record Goods Receipt_Release Purchase Order', 'intercase_n_3__Record Invoice Receipt_Record Goods Receipt_Remove Payment Block', 'intercase_n_3__Record Invoice Receipt_Record Goods Receipt_Vendor creates debit memo', 'intercase_n_3__Record Invoice Receipt_Record Goods Receipt_Vendor creates invoice', 'intercase_n_3__Record Invoice Receipt_Record Invoice Receipt_Cancel Goods Receipt', 'intercase_n_3__Record Invoice Receipt_Record Invoice Receipt_Cancel Invoice Receipt', 'intercase_n_3__Record Invoice Receipt_Record Invoice Receipt_Cancel Subsequent Invoice', 'intercase_n_3__Record Invoice Receipt_Record Invoice Receipt_Change Approval for Purchase Order', 'intercase_n_3__Record Invoice Receipt_Record Invoice Receipt_Change Delivery Indicator', 'intercase_n_3__Record Invoice Receipt_Record Invoice Receipt_Change Price', 'intercase_n_3__Record Invoice Receipt_Record Invoice Receipt_Change Quantity', 'intercase_n_3__Record Invoice Receipt_Record Invoice Receipt_Clear Invoice', 'intercase_n_3__Record Invoice Receipt_Record Invoice Receipt_Record Goods Receipt', 'intercase_n_3__Record Invoice Receipt_Record Invoice Receipt_Record Invoice Receipt', 'intercase_n_3__Record Invoice Receipt_Record Invoice Receipt_Record Service Entry Sheet', 'intercase_n_3__Record Invoice Receipt_Record Invoice Receipt_Record Subsequent Invoice', 'intercase_n_3__Record Invoice Receipt_Record Invoice Receipt_Remove Payment Block', 'intercase_n_3__Record Invoice Receipt_Record Invoice Receipt_SRM: Created', 'intercase_n_3__Record Invoice Receipt_Record Invoice Receipt_SRM: In Transfer to Execution Syst.', 'intercase_n_3__Record Invoice Receipt_Record Invoice Receipt_SRM: Ordered', 'intercase_n_3__Record Invoice Receipt_Record Invoice Receipt_Vendor creates debit memo', 'intercase_n_3__Record Invoice Receipt_Record Invoice Receipt_Vendor creates invoice', 'intercase_n_3__Record Invoice Receipt_Record Service Entry Sheet_Record Goods Receipt', 'intercase_n_3__Record Invoice Receipt_Record Service Entry Sheet_Record Service Entry Sheet', 'intercase_n_3__Record Invoice Receipt_Record Service Entry Sheet_Vendor creates invoice', 'intercase_n_3__Record Invoice Receipt_Record Subsequent Invoice_Cancel Subsequent Invoice', 'intercase_n_3__Record Invoice Receipt_Record Subsequent Invoice_Clear Invoice', 'intercase_n_3__Record Invoice Receipt_Record Subsequent Invoice_Record Subsequent Invoice', 'intercase_n_3__Record Invoice Receipt_Record Subsequent Invoice_Remove Payment Block', 'intercase_n_3__Record Invoice Receipt_Record Subsequent Invoice_Vendor creates invoice', 'intercase_n_3__Record Invoice Receipt_Release Purchase Order_Change Approval for Purchase Order', 'intercase_n_3__Record Invoice Receipt_Release Purchase Order_Change Price', 'intercase_n_3__Record Invoice Receipt_Release Purchase Order_Remove Payment Block', 'intercase_n_3__Record Invoice Receipt_Remove Payment Block_Cancel Goods Receipt', 'intercase_n_3__Record Invoice Receipt_Remove Payment Block_Cancel Invoice Receipt', 'intercase_n_3__Record Invoice Receipt_Remove Payment Block_Cancel Subsequent Invoice', 'intercase_n_3__Record Invoice Receipt_Remove Payment Block_Change Approval for Purchase Order', 'intercase_n_3__Record Invoice Receipt_Remove Payment Block_Change Delivery Indicator', 'intercase_n_3__Record Invoice Receipt_Remove Payment Block_Change Price', 'intercase_n_3__Record Invoice Receipt_Remove Payment Block_Change Quantity', 'intercase_n_3__Record Invoice Receipt_Remove Payment Block_Change Storage Location', 'intercase_n_3__Record Invoice Receipt_Remove Payment Block_Clear Invoice', 'intercase_n_3__Record Invoice Receipt_Remove Payment Block_Record Goods Receipt', 'intercase_n_3__Record Invoice Receipt_Remove Payment Block_Record Invoice Receipt', 'intercase_n_3__Record Invoice Receipt_Remove Payment Block_Record Service Entry Sheet', 'intercase_n_3__Record Invoice Receipt_Remove Payment Block_Record Subsequent Invoice', 'intercase_n_3__Record Invoice Receipt_Remove Payment Block_Release Purchase Order', 'intercase_n_3__Record Invoice Receipt_Remove Payment Block_Remove Payment Block', 'intercase_n_3__Record Invoice Receipt_Remove Payment Block_Set Payment Block', 'intercase_n_3__Record Invoice Receipt_Remove Payment Block_Vendor creates debit memo', 'intercase_n_3__Record Invoice Receipt_Remove Payment Block_Vendor creates invoice', 'intercase_n_3__Record Invoice Receipt_SRM: Complete_SRM: Awaiting Approval', 'intercase_n_3__Record Invoice Receipt_SRM: Created_Clear Invoice', 'intercase_n_3__Record Invoice Receipt_SRM: Created_Record Goods Receipt', 'intercase_n_3__Record Invoice Receipt_SRM: Created_SRM: Complete', 'intercase_n_3__Record Invoice Receipt_SRM: In Transfer to Execution Syst._SRM: Ordered', 'intercase_n_3__Record Invoice Receipt_SRM: In Transfer to Execution Syst._SRM: Transfer Failed (E.Sys.)', 'intercase_n_3__Record Invoice Receipt_SRM: Ordered_SRM: In Transfer to Execution Syst.', 'intercase_n_3__Record Invoice Receipt_Set Payment Block_Cancel Invoice Receipt', 'intercase_n_3__Record Invoice Receipt_Set Payment Block_Cancel Subsequent Invoice', 'intercase_n_3__Record Invoice Receipt_Set Payment Block_Change Approval for Purchase Order', 'intercase_n_3__Record Invoice Receipt_Set Payment Block_Clear Invoice', 'intercase_n_3__Record Invoice Receipt_Set Payment Block_Remove Payment Block', 'intercase_n_3__Record Invoice Receipt_Update Order Confirmation_Record Goods Receipt', 'intercase_n_3__Record Invoice Receipt_Vendor creates debit memo_Cancel Goods Receipt', 'intercase_n_3__Record Invoice Receipt_Vendor creates debit memo_Cancel Invoice Receipt', 'intercase_n_3__Record Invoice Receipt_Vendor creates debit memo_Cancel Subsequent Invoice', 'intercase_n_3__Record Invoice Receipt_Vendor creates debit memo_Change Delivery Indicator', 'intercase_n_3__Record Invoice Receipt_Vendor creates debit memo_Change Price', 'intercase_n_3__Record Invoice Receipt_Vendor creates debit memo_Change Quantity', 'intercase_n_3__Record Invoice Receipt_Vendor creates debit memo_Clear Invoice', 'intercase_n_3__Record Invoice Receipt_Vendor creates debit memo_Record Goods Receipt', 'intercase_n_3__Record Invoice Receipt_Vendor creates debit memo_Record Invoice Receipt', 'intercase_n_3__Record Invoice Receipt_Vendor creates debit memo_Remove Payment Block', 'intercase_n_3__Record Invoice Receipt_Vendor creates debit memo_Vendor creates debit memo', 'intercase_n_3__Record Invoice Receipt_Vendor creates debit memo_Vendor creates invoice', 'intercase_n_3__Record Invoice Receipt_Vendor creates invoice_Block Purchase Order Item', 'intercase_n_3__Record Invoice Receipt_Vendor creates invoice_Cancel Goods Receipt', 'intercase_n_3__Record Invoice Receipt_Vendor creates invoice_Cancel Invoice Receipt', 'intercase_n_3__Record Invoice Receipt_Vendor creates invoice_Change Approval for Purchase Order', 'intercase_n_3__Record Invoice Receipt_Vendor creates invoice_Change Delivery Indicator', 'intercase_n_3__Record Invoice Receipt_Vendor creates invoice_Change Price', 'intercase_n_3__Record Invoice Receipt_Vendor creates invoice_Change Quantity', 'intercase_n_3__Record Invoice Receipt_Vendor creates invoice_Clear Invoice', 'intercase_n_3__Record Invoice Receipt_Vendor creates invoice_Record Goods Receipt', 'intercase_n_3__Record Invoice Receipt_Vendor creates invoice_Record Invoice Receipt', 'intercase_n_3__Record Invoice Receipt_Vendor creates invoice_Record Service Entry Sheet', 'intercase_n_3__Record Invoice Receipt_Vendor creates invoice_Record Subsequent Invoice', 'intercase_n_3__Record Invoice Receipt_Vendor creates invoice_Remove Payment Block', 'intercase_n_3__Record Invoice Receipt_Vendor creates invoice_Set Payment Block', 'intercase_n_3__Record Invoice Receipt_Vendor creates invoice_Vendor creates debit memo', 'intercase_n_3__Record Invoice Receipt_Vendor creates invoice_Vendor creates invoice', 'intercase_n_3__Record Service Entry Sheet_Cancel Goods Receipt_Cancel Goods Receipt', 'intercase_n_3__Record Service Entry Sheet_Cancel Goods Receipt_Clear Invoice', 'intercase_n_3__Record Service Entry Sheet_Cancel Goods Receipt_Record Goods Receipt', 'intercase_n_3__Record Service Entry Sheet_Cancel Goods Receipt_Record Invoice Receipt', 'intercase_n_3__Record Service Entry Sheet_Cancel Goods Receipt_Record Service Entry Sheet', 'intercase_n_3__Record Service Entry Sheet_Cancel Goods Receipt_Vendor creates invoice', 'intercase_n_3__Record Service Entry Sheet_Cancel Invoice Receipt_Cancel Goods Receipt', 'intercase_n_3__Record Service Entry Sheet_Cancel Invoice Receipt_Cancel Invoice Receipt', 'intercase_n_3__Record Service Entry Sheet_Cancel Invoice Receipt_Change Price', 'intercase_n_3__Record Service Entry Sheet_Cancel Invoice Receipt_Clear Invoice', 'intercase_n_3__Record Service Entry Sheet_Cancel Invoice Receipt_Record Invoice Receipt', 'intercase_n_3__Record Service Entry Sheet_Cancel Invoice Receipt_Vendor creates invoice', 'intercase_n_3__Record Service Entry Sheet_Cancel Subsequent Invoice_Cancel Invoice Receipt', 'intercase_n_3__Record Service Entry Sheet_Change Price_Cancel Goods Receipt', 'intercase_n_3__Record Service Entry Sheet_Change Price_Change Price', 'intercase_n_3__Record Service Entry Sheet_Change Price_Clear Invoice', 'intercase_n_3__Record Service Entry Sheet_Change Price_Record Goods Receipt', 'intercase_n_3__Record Service Entry Sheet_Change Price_Record Invoice Receipt', 'intercase_n_3__Record Service Entry Sheet_Change Price_Record Service Entry Sheet', 'intercase_n_3__Record Service Entry Sheet_Change Price_Vendor creates invoice', 'intercase_n_3__Record Service Entry Sheet_Clear Invoice_Cancel Goods Receipt', 'intercase_n_3__Record Service Entry Sheet_Clear Invoice_Clear Invoice', 'intercase_n_3__Record Service Entry Sheet_Clear Invoice_Record Goods Receipt', 'intercase_n_3__Record Service Entry Sheet_Clear Invoice_Record Invoice Receipt', 'intercase_n_3__Record Service Entry Sheet_Clear Invoice_Record Service Entry Sheet', 'intercase_n_3__Record Service Entry Sheet_Clear Invoice_Remove Payment Block', 'intercase_n_3__Record Service Entry Sheet_Clear Invoice_Vendor creates invoice', 'intercase_n_3__Record Service Entry Sheet_Record Goods Receipt_Cancel Goods Receipt', 'intercase_n_3__Record Service Entry Sheet_Record Goods Receipt_Change Price', 'intercase_n_3__Record Service Entry Sheet_Record Goods Receipt_Clear Invoice', 'intercase_n_3__Record Service Entry Sheet_Record Goods Receipt_Record Goods Receipt', 'intercase_n_3__Record Service Entry Sheet_Record Goods Receipt_Record Invoice Receipt', 'intercase_n_3__Record Service Entry Sheet_Record Goods Receipt_Record Service Entry Sheet', 'intercase_n_3__Record Service Entry Sheet_Record Goods Receipt_Remove Payment Block', 'intercase_n_3__Record Service Entry Sheet_Record Goods Receipt_SRM: Complete', 'intercase_n_3__Record Service Entry Sheet_Record Goods Receipt_SRM: Created', 'intercase_n_3__Record Service Entry Sheet_Record Goods Receipt_Vendor creates debit memo', 'intercase_n_3__Record Service Entry Sheet_Record Goods Receipt_Vendor creates invoice', 'intercase_n_3__Record Service Entry Sheet_Record Invoice Receipt_Cancel Goods Receipt', 'intercase_n_3__Record Service Entry Sheet_Record Invoice Receipt_Cancel Invoice Receipt', 'intercase_n_3__Record Service Entry Sheet_Record Invoice Receipt_Cancel Subsequent Invoice', 'intercase_n_3__Record Service Entry Sheet_Record Invoice Receipt_Change Price', 'intercase_n_3__Record Service Entry Sheet_Record Invoice Receipt_Clear Invoice', 'intercase_n_3__Record Service Entry Sheet_Record Invoice Receipt_Record Goods Receipt', 'intercase_n_3__Record Service Entry Sheet_Record Invoice Receipt_Record Invoice Receipt', 'intercase_n_3__Record Service Entry Sheet_Record Invoice Receipt_Record Service Entry Sheet', 'intercase_n_3__Record Service Entry Sheet_Record Invoice Receipt_Remove Payment Block', 'intercase_n_3__Record Service Entry Sheet_Record Invoice Receipt_SRM: Created', 'intercase_n_3__Record Service Entry Sheet_Record Invoice Receipt_SRM: In Transfer to Execution Syst.', 'intercase_n_3__Record Service Entry Sheet_Record Invoice Receipt_Set Payment Block', 'intercase_n_3__Record Service Entry Sheet_Record Invoice Receipt_Vendor creates debit memo', 'intercase_n_3__Record Service Entry Sheet_Record Invoice Receipt_Vendor creates invoice', 'intercase_n_3__Record Service Entry Sheet_Record Service Entry Sheet_Cancel Goods Receipt', 'intercase_n_3__Record Service Entry Sheet_Record Service Entry Sheet_Change Price', 'intercase_n_3__Record Service Entry Sheet_Record Service Entry Sheet_Clear Invoice', 'intercase_n_3__Record Service Entry Sheet_Record Service Entry Sheet_Record Goods Receipt', 'intercase_n_3__Record Service Entry Sheet_Record Service Entry Sheet_Record Invoice Receipt', 'intercase_n_3__Record Service Entry Sheet_Record Service Entry Sheet_Record Service Entry Sheet', 'intercase_n_3__Record Service Entry Sheet_Record Service Entry Sheet_Remove Payment Block', 'intercase_n_3__Record Service Entry Sheet_Record Service Entry Sheet_Vendor creates debit memo', 'intercase_n_3__Record Service Entry Sheet_Record Service Entry Sheet_Vendor creates invoice', 'intercase_n_3__Record Service Entry Sheet_Remove Payment Block_Clear Invoice', 'intercase_n_3__Record Service Entry Sheet_Remove Payment Block_Record Invoice Receipt', 'intercase_n_3__Record Service Entry Sheet_Remove Payment Block_Remove Payment Block', 'intercase_n_3__Record Service Entry Sheet_Remove Payment Block_Vendor creates invoice', 'intercase_n_3__Record Service Entry Sheet_SRM: Complete_SRM: Awaiting Approval', 'intercase_n_3__Record Service Entry Sheet_SRM: Created_Cancel Goods Receipt', 'intercase_n_3__Record Service Entry Sheet_SRM: Created_SRM: Complete', 'intercase_n_3__Record Service Entry Sheet_SRM: Ordered_SRM: In Transfer to Execution Syst.', 'intercase_n_3__Record Service Entry Sheet_Vendor creates debit memo_Cancel Invoice Receipt', 'intercase_n_3__Record Service Entry Sheet_Vendor creates debit memo_Clear Invoice', 'intercase_n_3__Record Service Entry Sheet_Vendor creates debit memo_Record Invoice Receipt', 'intercase_n_3__Record Service Entry Sheet_Vendor creates debit memo_Vendor creates invoice', 'intercase_n_3__Record Service Entry Sheet_Vendor creates invoice_Cancel Goods Receipt', 'intercase_n_3__Record Service Entry Sheet_Vendor creates invoice_Cancel Invoice Receipt', 'intercase_n_3__Record Service Entry Sheet_Vendor creates invoice_Change Price', 'intercase_n_3__Record Service Entry Sheet_Vendor creates invoice_Clear Invoice', 'intercase_n_3__Record Service Entry Sheet_Vendor creates invoice_Record Goods Receipt', 'intercase_n_3__Record Service Entry Sheet_Vendor creates invoice_Record Invoice Receipt', 'intercase_n_3__Record Service Entry Sheet_Vendor creates invoice_Record Service Entry Sheet', 'intercase_n_3__Record Service Entry Sheet_Vendor creates invoice_Remove Payment Block', 'intercase_n_3__Record Service Entry Sheet_Vendor creates invoice_SRM: Created', 'intercase_n_3__Record Service Entry Sheet_Vendor creates invoice_Vendor creates debit memo', 'intercase_n_3__Record Service Entry Sheet_Vendor creates invoice_Vendor creates invoice', 'intercase_n_3__Record Subsequent Invoice_Cancel Invoice Receipt_Cancel Subsequent Invoice', 'intercase_n_3__Record Subsequent Invoice_Cancel Invoice Receipt_Record Invoice Receipt', 'intercase_n_3__Record Subsequent Invoice_Cancel Subsequent Invoice_Clear Invoice', 'intercase_n_3__Record Subsequent Invoice_Cancel Subsequent Invoice_Remove Payment Block', 'intercase_n_3__Record Subsequent Invoice_Change Price_Remove Payment Block', 'intercase_n_3__Record Subsequent Invoice_Clear Invoice_Cancel Invoice Receipt', 'intercase_n_3__Record Subsequent Invoice_Clear Invoice_Cancel Subsequent Invoice', 'intercase_n_3__Record Subsequent Invoice_Clear Invoice_Clear Invoice', 'intercase_n_3__Record Subsequent Invoice_Clear Invoice_Record Subsequent Invoice', 'intercase_n_3__Record Subsequent Invoice_Clear Invoice_Remove Payment Block', 'intercase_n_3__Record Subsequent Invoice_Clear Invoice_Vendor creates invoice', 'intercase_n_3__Record Subsequent Invoice_Record Goods Receipt_Record Service Entry Sheet', 'intercase_n_3__Record Subsequent Invoice_Record Subsequent Invoice_Record Subsequent Invoice', 'intercase_n_3__Record Subsequent Invoice_Record Subsequent Invoice_Remove Payment Block', 'intercase_n_3__Record Subsequent Invoice_Remove Payment Block_Clear Invoice', 'intercase_n_3__Record Subsequent Invoice_Remove Payment Block_Record Service Entry Sheet', 'intercase_n_3__Record Subsequent Invoice_Remove Payment Block_Remove Payment Block', 'intercase_n_3__Record Subsequent Invoice_Vendor creates invoice_Clear Invoice', 'intercase_n_3__Record Subsequent Invoice_Vendor creates invoice_Record Goods Receipt', 'intercase_n_3__Release Purchase Order', 'intercase_n_3__Release Purchase Order_Change Approval for Purchase Order', 'intercase_n_3__Release Purchase Order_Change Approval for Purchase Order_Cancel Goods Receipt', 'intercase_n_3__Release Purchase Order_Change Approval for Purchase Order_Change Approval for Purchase Order', 'intercase_n_3__Release Purchase Order_Change Approval for Purchase Order_Change Price', 'intercase_n_3__Release Purchase Order_Change Approval for Purchase Order_Change Quantity', 'intercase_n_3__Release Purchase Order_Change Approval for Purchase Order_Change Storage Location', 'intercase_n_3__Release Purchase Order_Change Approval for Purchase Order_Clear Invoice', 'intercase_n_3__Release Purchase Order_Change Approval for Purchase Order_Create Purchase Order Item', 'intercase_n_3__Release Purchase Order_Change Approval for Purchase Order_Delete Purchase Order Item', 'intercase_n_3__Release Purchase Order_Change Approval for Purchase Order_Receive Order Confirmation', 'intercase_n_3__Release Purchase Order_Change Approval for Purchase Order_Record Goods Receipt', 'intercase_n_3__Release Purchase Order_Change Approval for Purchase Order_Record Invoice Receipt', 'intercase_n_3__Release Purchase Order_Change Approval for Purchase Order_Release Purchase Order', 'intercase_n_3__Release Purchase Order_Change Approval for Purchase Order_Remove Payment Block', 'intercase_n_3__Release Purchase Order_Change Approval for Purchase Order_Vendor creates invoice', 'intercase_n_3__Release Purchase Order_Change Price_Change Approval for Purchase Order', 'intercase_n_3__Release Purchase Order_Change Price_Change Price', 'intercase_n_3__Release Purchase Order_Change Price_Change Quantity', 'intercase_n_3__Release Purchase Order_Change Price_Remove Payment Block', 'intercase_n_3__Release Purchase Order_Change Quantity_Change Approval for Purchase Order', 'intercase_n_3__Release Purchase Order_Change Quantity_Change Quantity', 'intercase_n_3__Release Purchase Order_Change Quantity_Vendor creates invoice', 'intercase_n_3__Release Purchase Order_Clear Invoice_Change Approval for Purchase Order', 'intercase_n_3__Release Purchase Order_Create Purchase Order Item', 'intercase_n_3__Release Purchase Order_Create Purchase Order Item_Change Approval for Purchase Order', 'intercase_n_3__Release Purchase Order_Create Purchase Order Item_Change Quantity', 'intercase_n_3__Release Purchase Order_Delete Purchase Order Item_Change Approval for Purchase Order', 'intercase_n_3__Release Purchase Order_Remove Payment Block_Change Price', 'intercase_n_3__Release Purchase Order_Vendor creates invoice_Change Approval for Purchase Order', 'intercase_n_3__Release Purchase Order_Vendor creates invoice_Change Quantity', 'intercase_n_3__Release Purchase Requisition_Create Purchase Order Item_Change Price', 'intercase_n_3__Release Purchase Requisition_Create Purchase Order Item_Change Quantity', 'intercase_n_3__Release Purchase Requisition_Create Purchase Order Item_Receive Order Confirmation', 'intercase_n_3__Release Purchase Requisition_Create Purchase Order Item_Record Goods Receipt', 'intercase_n_3__Release Purchase Requisition_Create Purchase Order Item_Vendor creates invoice', 'intercase_n_3__Remove Payment Block_Cancel Goods Receipt_Change Delivery Indicator', 'intercase_n_3__Remove Payment Block_Cancel Goods Receipt_Change Quantity', 'intercase_n_3__Remove Payment Block_Cancel Goods Receipt_Clear Invoice', 'intercase_n_3__Remove Payment Block_Cancel Goods Receipt_Record Goods Receipt', 'intercase_n_3__Remove Payment Block_Cancel Goods Receipt_Vendor creates debit memo', 'intercase_n_3__Remove Payment Block_Cancel Invoice Receipt_Cancel Invoice Receipt', 'intercase_n_3__Remove Payment Block_Cancel Invoice Receipt_Clear Invoice', 'intercase_n_3__Remove Payment Block_Cancel Invoice Receipt_Record Invoice Receipt', 'intercase_n_3__Remove Payment Block_Cancel Invoice Receipt_Record Subsequent Invoice', 'intercase_n_3__Remove Payment Block_Cancel Invoice Receipt_Remove Payment Block', 'intercase_n_3__Remove Payment Block_Cancel Invoice Receipt_Vendor creates debit memo', 'intercase_n_3__Remove Payment Block_Cancel Invoice Receipt_Vendor creates invoice', 'intercase_n_3__Remove Payment Block_Cancel Subsequent Invoice_Cancel Invoice Receipt', 'intercase_n_3__Remove Payment Block_Cancel Subsequent Invoice_Clear Invoice', 'intercase_n_3__Remove Payment Block_Cancel Subsequent Invoice_Vendor creates debit memo', 'intercase_n_3__Remove Payment Block_Change Approval for Purchase Order_Change Approval for Purchase Order', 'intercase_n_3__Remove Payment Block_Change Approval for Purchase Order_Release Purchase Order', 'intercase_n_3__Remove Payment Block_Change Delivery Indicator_Cancel Invoice Receipt', 'intercase_n_3__Remove Payment Block_Change Delivery Indicator_Change Delivery Indicator', 'intercase_n_3__Remove Payment Block_Change Delivery Indicator_Change Quantity', 'intercase_n_3__Remove Payment Block_Change Delivery Indicator_Clear Invoice', 'intercase_n_3__Remove Payment Block_Change Delivery Indicator_Record Goods Receipt', 'intercase_n_3__Remove Payment Block_Change Delivery Indicator_Record Invoice Receipt', 'intercase_n_3__Remove Payment Block_Change Delivery Indicator_Vendor creates debit memo', 'intercase_n_3__Remove Payment Block_Change Delivery Indicator_Vendor creates invoice', 'intercase_n_3__Remove Payment Block_Change Price_Cancel Invoice Receipt', 'intercase_n_3__Remove Payment Block_Change Price_Change Price', 'intercase_n_3__Remove Payment Block_Change Price_Change Quantity', 'intercase_n_3__Remove Payment Block_Change Price_Clear Invoice', 'intercase_n_3__Remove Payment Block_Change Price_Record Invoice Receipt', 'intercase_n_3__Remove Payment Block_Change Price_Record Subsequent Invoice', 'intercase_n_3__Remove Payment Block_Change Price_Remove Payment Block', 'intercase_n_3__Remove Payment Block_Change Price_Vendor creates debit memo', 'intercase_n_3__Remove Payment Block_Change Price_Vendor creates invoice', 'intercase_n_3__Remove Payment Block_Change Quantity_Cancel Invoice Receipt', 'intercase_n_3__Remove Payment Block_Change Quantity_Change Delivery Indicator', 'intercase_n_3__Remove Payment Block_Change Quantity_Change Price', 'intercase_n_3__Remove Payment Block_Change Quantity_Change Quantity', 'intercase_n_3__Remove Payment Block_Change Quantity_Clear Invoice', 'intercase_n_3__Remove Payment Block_Change Quantity_Record Goods Receipt', 'intercase_n_3__Remove Payment Block_Change Quantity_Vendor creates debit memo', 'intercase_n_3__Remove Payment Block_Change Quantity_Vendor creates invoice', 'intercase_n_3__Remove Payment Block_Clear Invoice_Block Purchase Order Item', 'intercase_n_3__Remove Payment Block_Clear Invoice_Cancel Goods Receipt', 'intercase_n_3__Remove Payment Block_Clear Invoice_Cancel Invoice Receipt', 'intercase_n_3__Remove Payment Block_Clear Invoice_Cancel Subsequent Invoice', 'intercase_n_3__Remove Payment Block_Clear Invoice_Change Delivery Indicator', 'intercase_n_3__Remove Payment Block_Clear Invoice_Change Price', 'intercase_n_3__Remove Payment Block_Clear Invoice_Change Quantity', 'intercase_n_3__Remove Payment Block_Clear Invoice_Clear Invoice', 'intercase_n_3__Remove Payment Block_Clear Invoice_Record Goods Receipt', 'intercase_n_3__Remove Payment Block_Clear Invoice_Record Invoice Receipt', 'intercase_n_3__Remove Payment Block_Clear Invoice_Record Service Entry Sheet', 'intercase_n_3__Remove Payment Block_Clear Invoice_Record Subsequent Invoice', 'intercase_n_3__Remove Payment Block_Clear Invoice_Remove Payment Block', 'intercase_n_3__Remove Payment Block_Clear Invoice_SRM: Created', 'intercase_n_3__Remove Payment Block_Clear Invoice_Vendor creates debit memo', 'intercase_n_3__Remove Payment Block_Clear Invoice_Vendor creates invoice', 'intercase_n_3__Remove Payment Block_Record Goods Receipt_Cancel Goods Receipt', 'intercase_n_3__Remove Payment Block_Record Goods Receipt_Change Delivery Indicator', 'intercase_n_3__Remove Payment Block_Record Goods Receipt_Change Quantity', 'intercase_n_3__Remove Payment Block_Record Goods Receipt_Clear Invoice', 'intercase_n_3__Remove Payment Block_Record Goods Receipt_Record Goods Receipt', 'intercase_n_3__Remove Payment Block_Record Goods Receipt_Record Invoice Receipt', 'intercase_n_3__Remove Payment Block_Record Goods Receipt_Record Service Entry Sheet', 'intercase_n_3__Remove Payment Block_Record Goods Receipt_Remove Payment Block', 'intercase_n_3__Remove Payment Block_Record Goods Receipt_Vendor creates invoice', 'intercase_n_3__Remove Payment Block_Record Invoice Receipt_Cancel Invoice Receipt', 'intercase_n_3__Remove Payment Block_Record Invoice Receipt_Cancel Subsequent Invoice', 'intercase_n_3__Remove Payment Block_Record Invoice Receipt_Change Delivery Indicator', 'intercase_n_3__Remove Payment Block_Record Invoice Receipt_Change Price', 'intercase_n_3__Remove Payment Block_Record Invoice Receipt_Clear Invoice', 'intercase_n_3__Remove Payment Block_Record Invoice Receipt_Record Goods Receipt', 'intercase_n_3__Remove Payment Block_Record Invoice Receipt_Record Invoice Receipt', 'intercase_n_3__Remove Payment Block_Record Invoice Receipt_Record Service Entry Sheet', 'intercase_n_3__Remove Payment Block_Record Invoice Receipt_Remove Payment Block', 'intercase_n_3__Remove Payment Block_Record Invoice Receipt_Vendor creates invoice', 'intercase_n_3__Remove Payment Block_Record Service Entry Sheet_Change Price', 'intercase_n_3__Remove Payment Block_Record Service Entry Sheet_Record Goods Receipt', 'intercase_n_3__Remove Payment Block_Record Service Entry Sheet_Record Service Entry Sheet', 'intercase_n_3__Remove Payment Block_Record Subsequent Invoice_Change Price', 'intercase_n_3__Remove Payment Block_Record Subsequent Invoice_Clear Invoice', 'intercase_n_3__Remove Payment Block_Record Subsequent Invoice_Remove Payment Block', 'intercase_n_3__Remove Payment Block_Record Subsequent Invoice_Vendor creates invoice', 'intercase_n_3__Remove Payment Block_Release Purchase Order_Change Approval for Purchase Order', 'intercase_n_3__Remove Payment Block_Remove Payment Block_Cancel Invoice Receipt', 'intercase_n_3__Remove Payment Block_Remove Payment Block_Cancel Subsequent Invoice', 'intercase_n_3__Remove Payment Block_Remove Payment Block_Change Delivery Indicator', 'intercase_n_3__Remove Payment Block_Remove Payment Block_Change Price', 'intercase_n_3__Remove Payment Block_Remove Payment Block_Clear Invoice', 'intercase_n_3__Remove Payment Block_Remove Payment Block_Record Goods Receipt', 'intercase_n_3__Remove Payment Block_Remove Payment Block_Record Invoice Receipt', 'intercase_n_3__Remove Payment Block_Remove Payment Block_Remove Payment Block', 'intercase_n_3__Remove Payment Block_Remove Payment Block_Vendor creates invoice', 'intercase_n_3__Remove Payment Block_Set Payment Block_Change Price', 'intercase_n_3__Remove Payment Block_Set Payment Block_Clear Invoice', 'intercase_n_3__Remove Payment Block_Vendor creates debit memo_Cancel Invoice Receipt', 'intercase_n_3__Remove Payment Block_Vendor creates debit memo_Cancel Subsequent Invoice', 'intercase_n_3__Remove Payment Block_Vendor creates debit memo_Clear Invoice', 'intercase_n_3__Remove Payment Block_Vendor creates debit memo_Record Goods Receipt', 'intercase_n_3__Remove Payment Block_Vendor creates debit memo_Vendor creates invoice', 'intercase_n_3__Remove Payment Block_Vendor creates invoice_Cancel Goods Receipt', 'intercase_n_3__Remove Payment Block_Vendor creates invoice_Cancel Invoice Receipt', 'intercase_n_3__Remove Payment Block_Vendor creates invoice_Change Delivery Indicator', 'intercase_n_3__Remove Payment Block_Vendor creates invoice_Change Price', 'intercase_n_3__Remove Payment Block_Vendor creates invoice_Change Quantity', 'intercase_n_3__Remove Payment Block_Vendor creates invoice_Clear Invoice', 'intercase_n_3__Remove Payment Block_Vendor creates invoice_Record Goods Receipt', 'intercase_n_3__Remove Payment Block_Vendor creates invoice_Record Invoice Receipt', 'intercase_n_3__Remove Payment Block_Vendor creates invoice_Record Subsequent Invoice', 'intercase_n_3__Remove Payment Block_Vendor creates invoice_Remove Payment Block', 'intercase_n_3__Remove Payment Block_Vendor creates invoice_Vendor creates debit memo', 'intercase_n_3__Remove Payment Block_Vendor creates invoice_Vendor creates invoice', 'intercase_n_3__SRM: Awaiting Approval_SRM: Document Completed_Cancel Goods Receipt', 'intercase_n_3__SRM: Awaiting Approval_SRM: Document Completed_Clear Invoice', 'intercase_n_3__SRM: Awaiting Approval_SRM: Document Completed_Create Purchase Order Item', 'intercase_n_3__SRM: Awaiting Approval_SRM: Document Completed_Record Invoice Receipt', 'intercase_n_3__SRM: Awaiting Approval_SRM: Document Completed_SRM: Change was Transmitted', 'intercase_n_3__SRM: Awaiting Approval_SRM: Document Completed_SRM: Created', 'intercase_n_3__SRM: Awaiting Approval_SRM: Document Completed_SRM: In Transfer to Execution Syst.', 'intercase_n_3__SRM: Awaiting Approval_SRM: Document Completed_SRM: Ordered', 'intercase_n_3__SRM: Awaiting Approval_SRM: Document Completed_Vendor creates invoice', 'intercase_n_3__SRM: Awaiting Approval_SRM: In Transfer to Execution Syst._SRM: Ordered', 'intercase_n_3__SRM: Awaiting Approval_SRM: Ordered_SRM: Document Completed', 'intercase_n_3__SRM: Change was Transmitted_Create Purchase Order Item_Record Goods Receipt', 'intercase_n_3__SRM: Change was Transmitted_Create Purchase Order Item_Record Invoice Receipt', 'intercase_n_3__SRM: Change was Transmitted_Create Purchase Order Item_Record Service Entry Sheet', 'intercase_n_3__SRM: Change was Transmitted_Create Purchase Order Item_SRM: Created', 'intercase_n_3__SRM: Change was Transmitted_Create Purchase Order Item_SRM: In Transfer to Execution Syst.', 'intercase_n_3__SRM: Change was Transmitted_Create Purchase Order Item_Vendor creates debit memo', 'intercase_n_3__SRM: Change was Transmitted_Create Purchase Order Item_Vendor creates invoice', 'intercase_n_3__SRM: Change was Transmitted_Record Goods Receipt_Record Invoice Receipt', 'intercase_n_3__SRM: Change was Transmitted_Record Goods Receipt_Record Service Entry Sheet', 'intercase_n_3__SRM: Change was Transmitted_Record Goods Receipt_Vendor creates debit memo', 'intercase_n_3__SRM: Change was Transmitted_Record Goods Receipt_Vendor creates invoice', 'intercase_n_3__SRM: Change was Transmitted_Record Invoice Receipt_Record Goods Receipt', 'intercase_n_3__SRM: Change was Transmitted_Record Invoice Receipt_Vendor creates invoice', 'intercase_n_3__SRM: Change was Transmitted_Record Service Entry Sheet_Record Goods Receipt', 'intercase_n_3__SRM: Change was Transmitted_SRM: Complete_SRM: Awaiting Approval', 'intercase_n_3__SRM: Change was Transmitted_SRM: Created_SRM: Complete', 'intercase_n_3__SRM: Change was Transmitted_SRM: Created_SRM: Created', 'intercase_n_3__SRM: Change was Transmitted_SRM: In Transfer to Execution Syst._Create Purchase Order Item', 'intercase_n_3__SRM: Change was Transmitted_SRM: Ordered_Create Purchase Order Item', 'intercase_n_3__SRM: Change was Transmitted_SRM: Ordered_SRM: In Transfer to Execution Syst.', 'intercase_n_3__SRM: Change was Transmitted_SRM: Ordered_Vendor creates invoice', 'intercase_n_3__SRM: Change was Transmitted_Vendor creates debit memo_Vendor creates invoice', 'intercase_n_3__SRM: Change was Transmitted_Vendor creates invoice_Record Goods Receipt', 'intercase_n_3__SRM: Change was Transmitted_Vendor creates invoice_Record Invoice Receipt', 'intercase_n_3__SRM: Change was Transmitted_Vendor creates invoice_Record Service Entry Sheet', 'intercase_n_3__SRM: Change was Transmitted_Vendor creates invoice_SRM: Created', 'intercase_n_3__SRM: Change was Transmitted_Vendor creates invoice_SRM: In Transfer to Execution Syst.', 'intercase_n_3__SRM: Change was Transmitted_Vendor creates invoice_Vendor creates debit memo', 'intercase_n_3__SRM: Change was Transmitted_Vendor creates invoice_Vendor creates invoice', 'intercase_n_3__SRM: Complete_SRM: Awaiting Approval_SRM: Document Completed', 'intercase_n_3__SRM: Complete_SRM: Awaiting Approval_SRM: In Transfer to Execution Syst.', 'intercase_n_3__SRM: Complete_SRM: Awaiting Approval_SRM: Ordered', 'intercase_n_3__SRM: Created', 'intercase_n_3__SRM: Created_Cancel Goods Receipt_SRM: Complete', 'intercase_n_3__SRM: Created_Clear Invoice_SRM: Complete', 'intercase_n_3__SRM: Created_Create Purchase Order Item', 'intercase_n_3__SRM: Created_Create Purchase Order Item_SRM: Complete', 'intercase_n_3__SRM: Created_Record Goods Receipt_Record Service Entry Sheet', 'intercase_n_3__SRM: Created_Record Goods Receipt_SRM: Complete', 'intercase_n_3__SRM: Created_Record Invoice Receipt_SRM: Complete', 'intercase_n_3__SRM: Created_Record Service Entry Sheet_Record Goods Receipt', 'intercase_n_3__SRM: Created_SRM: Complete', 'intercase_n_3__SRM: Created_SRM: Complete_SRM: Awaiting Approval', 'intercase_n_3__SRM: Created_SRM: Created', 'intercase_n_3__SRM: Created_SRM: Created_SRM: Complete', 'intercase_n_3__SRM: Created_SRM: In Transfer to Execution Syst._SRM: Complete', 'intercase_n_3__SRM: Created_SRM: Incomplete', 'intercase_n_3__SRM: Created_SRM: Incomplete_SRM: Held', 'intercase_n_3__SRM: Created_SRM: Ordered_SRM: In Transfer to Execution Syst.', 'intercase_n_3__SRM: Deleted_Change Delivery Indicator_Change Final Invoice Indicator', 'intercase_n_3__SRM: Deleted_Change Delivery Indicator_SRM: Created', 'intercase_n_3__SRM: Deleted_Change Price_Change Quantity', 'intercase_n_3__SRM: Deleted_Change Price_Clear Invoice', 'intercase_n_3__SRM: Deleted_Change Price_Record Goods Receipt', 'intercase_n_3__SRM: Deleted_Change Price_Record Invoice Receipt', 'intercase_n_3__SRM: Deleted_Change Price_Record Service Entry Sheet', 'intercase_n_3__SRM: Deleted_Change Price_SRM: Complete', 'intercase_n_3__SRM: Deleted_Change Price_SRM: Created', 'intercase_n_3__SRM: Deleted_Change Price_SRM: In Transfer to Execution Syst.', 'intercase_n_3__SRM: Deleted_Change Price_SRM: Transaction Completed', 'intercase_n_3__SRM: Deleted_Change Price_Vendor creates invoice', 'intercase_n_3__SRM: Deleted_Change Quantity_Record Goods Receipt', 'intercase_n_3__SRM: Deleted_Clear Invoice_SRM: Created', 'intercase_n_3__SRM: Deleted_Clear Invoice_Vendor creates invoice', 'intercase_n_3__SRM: Deleted_Delete Purchase Order Item_Change Price', 'intercase_n_3__SRM: Deleted_Delete Purchase Order Item_SRM: In Transfer to Execution Syst.', 'intercase_n_3__SRM: Deleted_Record Goods Receipt_Record Service Entry Sheet', 'intercase_n_3__SRM: Deleted_Record Invoice Receipt_Clear Invoice', 'intercase_n_3__SRM: Deleted_Record Invoice Receipt_Record Goods Receipt', 'intercase_n_3__SRM: Deleted_Record Invoice Receipt_Record Invoice Receipt', 'intercase_n_3__SRM: Deleted_Record Service Entry Sheet_Record Goods Receipt', 'intercase_n_3__SRM: Deleted_SRM: Complete_SRM: Awaiting Approval', 'intercase_n_3__SRM: Deleted_SRM: Created_SRM: Complete', 'intercase_n_3__SRM: Deleted_SRM: In Transfer to Execution Syst._Change Price', 'intercase_n_3__SRM: Deleted_SRM: In Transfer to Execution Syst._SRM: In Transfer to Execution Syst.', 'intercase_n_3__SRM: Deleted_Vendor creates invoice_Clear Invoice', 'intercase_n_3__SRM: Deleted_Vendor creates invoice_Record Goods Receipt', 'intercase_n_3__SRM: Deleted_Vendor creates invoice_SRM: Created', 'intercase_n_3__SRM: Document Completed_Cancel Goods Receipt_SRM: Ordered', 'intercase_n_3__SRM: Document Completed_Clear Invoice_Clear Invoice', 'intercase_n_3__SRM: Document Completed_Clear Invoice_SRM: In Transfer to Execution Syst.', 'intercase_n_3__SRM: Document Completed_Create Purchase Order Item_SRM: In Transfer to Execution Syst.', 'intercase_n_3__SRM: Document Completed_Create Purchase Order Item_SRM: Ordered', 'intercase_n_3__SRM: Document Completed_Record Invoice Receipt_Record Invoice Receipt', 'intercase_n_3__SRM: Document Completed_Record Invoice Receipt_SRM: In Transfer to Execution Syst.', 'intercase_n_3__SRM: Document Completed_Record Invoice Receipt_SRM: Ordered', 'intercase_n_3__SRM: Document Completed_SRM: Change was Transmitted_Create Purchase Order Item', 'intercase_n_3__SRM: Document Completed_SRM: Change was Transmitted_SRM: In Transfer to Execution Syst.', 'intercase_n_3__SRM: Document Completed_SRM: Change was Transmitted_SRM: Ordered', 'intercase_n_3__SRM: Document Completed_SRM: Created_SRM: Ordered', 'intercase_n_3__SRM: Document Completed_SRM: In Transfer to Execution Syst._Create Purchase Order Item', 'intercase_n_3__SRM: Document Completed_SRM: In Transfer to Execution Syst._SRM: Change was Transmitted', 'intercase_n_3__SRM: Document Completed_SRM: In Transfer to Execution Syst._SRM: In Transfer to Execution Syst.', 'intercase_n_3__SRM: Document Completed_SRM: In Transfer to Execution Syst._SRM: Ordered', 'intercase_n_3__SRM: Document Completed_SRM: Ordered_Create Purchase Order Item', 'intercase_n_3__SRM: Document Completed_SRM: Ordered_SRM: Change was Transmitted', 'intercase_n_3__SRM: Document Completed_SRM: Ordered_SRM: Deleted', 'intercase_n_3__SRM: Document Completed_SRM: Ordered_SRM: In Transfer to Execution Syst.', 'intercase_n_3__SRM: Document Completed_Vendor creates invoice_Cancel Invoice Receipt', 'intercase_n_3__SRM: Document Completed_Vendor creates invoice_Record Goods Receipt', 'intercase_n_3__SRM: Document Completed_Vendor creates invoice_SRM: In Transfer to Execution Syst.', 'intercase_n_3__SRM: Held_SRM: Complete_SRM: Awaiting Approval', 'intercase_n_3__SRM: In Transfer to Execution Syst._Cancel Goods Receipt_Record Goods Receipt', 'intercase_n_3__SRM: In Transfer to Execution Syst._Change Price_Record Goods Receipt', 'intercase_n_3__SRM: In Transfer to Execution Syst._Change Price_Record Invoice Receipt', 'intercase_n_3__SRM: In Transfer to Execution Syst._Change Price_Vendor creates invoice', 'intercase_n_3__SRM: In Transfer to Execution Syst._Create Purchase Order Item_Record Goods Receipt', 'intercase_n_3__SRM: In Transfer to Execution Syst._Create Purchase Order Item_SRM: Change was Transmitted', 'intercase_n_3__SRM: In Transfer to Execution Syst._Create Purchase Order Item_SRM: Created', 'intercase_n_3__SRM: In Transfer to Execution Syst._Create Purchase Order Item_SRM: Ordered', 'intercase_n_3__SRM: In Transfer to Execution Syst._Create Purchase Order Item_Vendor creates debit memo', 'intercase_n_3__SRM: In Transfer to Execution Syst._Create Purchase Order Item_Vendor creates invoice', 'intercase_n_3__SRM: In Transfer to Execution Syst._Record Goods Receipt_Record Invoice Receipt', 'intercase_n_3__SRM: In Transfer to Execution Syst._Record Goods Receipt_Record Service Entry Sheet', 'intercase_n_3__SRM: In Transfer to Execution Syst._Record Goods Receipt_Vendor creates invoice', 'intercase_n_3__SRM: In Transfer to Execution Syst._Record Service Entry Sheet_Record Goods Receipt', 'intercase_n_3__SRM: In Transfer to Execution Syst._SRM: Change was Transmitted_Create Purchase Order Item', 'intercase_n_3__SRM: In Transfer to Execution Syst._SRM: Change was Transmitted_Record Goods Receipt', 'intercase_n_3__SRM: In Transfer to Execution Syst._SRM: Change was Transmitted_Record Service Entry Sheet', 'intercase_n_3__SRM: In Transfer to Execution Syst._SRM: Change was Transmitted_SRM: Created', 'intercase_n_3__SRM: In Transfer to Execution Syst._SRM: Change was Transmitted_SRM: Ordered', 'intercase_n_3__SRM: In Transfer to Execution Syst._SRM: Change was Transmitted_Vendor creates invoice', 'intercase_n_3__SRM: In Transfer to Execution Syst._SRM: Complete_SRM: Awaiting Approval', 'intercase_n_3__SRM: In Transfer to Execution Syst._SRM: Created_SRM: Complete', 'intercase_n_3__SRM: In Transfer to Execution Syst._SRM: Deleted_Change Delivery Indicator', 'intercase_n_3__SRM: In Transfer to Execution Syst._SRM: Deleted_Change Price', 'intercase_n_3__SRM: In Transfer to Execution Syst._SRM: Deleted_Change Quantity', 'intercase_n_3__SRM: In Transfer to Execution Syst._SRM: Deleted_Delete Purchase Order Item', 'intercase_n_3__SRM: In Transfer to Execution Syst._SRM: Deleted_Record Goods Receipt', 'intercase_n_3__SRM: In Transfer to Execution Syst._SRM: Deleted_Record Invoice Receipt', 'intercase_n_3__SRM: In Transfer to Execution Syst._SRM: Deleted_SRM: Complete', 'intercase_n_3__SRM: In Transfer to Execution Syst._SRM: Deleted_Vendor creates invoice', 'intercase_n_3__SRM: In Transfer to Execution Syst._SRM: In Transfer to Execution Syst._Create Purchase Order Item', 'intercase_n_3__SRM: In Transfer to Execution Syst._SRM: In Transfer to Execution Syst._Record Goods Receipt', 'intercase_n_3__SRM: In Transfer to Execution Syst._SRM: In Transfer to Execution Syst._Record Invoice Receipt', 'intercase_n_3__SRM: In Transfer to Execution Syst._SRM: In Transfer to Execution Syst._SRM: Ordered', 'intercase_n_3__SRM: In Transfer to Execution Syst._SRM: Ordered_Create Purchase Order Item', 'intercase_n_3__SRM: In Transfer to Execution Syst._SRM: Ordered_SRM: Change was Transmitted', 'intercase_n_3__SRM: In Transfer to Execution Syst._SRM: Ordered_SRM: Deleted', 'intercase_n_3__SRM: In Transfer to Execution Syst._SRM: Ordered_SRM: Document Completed', 'intercase_n_3__SRM: In Transfer to Execution Syst._Vendor creates invoice_Record Goods Receipt', 'intercase_n_3__SRM: Incomplete_SRM: Held_SRM: Complete', 'intercase_n_3__SRM: Ordered_Create Purchase Order Item_Record Goods Receipt', 'intercase_n_3__SRM: Ordered_Create Purchase Order Item_SRM: Change was Transmitted', 'intercase_n_3__SRM: Ordered_Create Purchase Order Item_SRM: In Transfer to Execution Syst.', 'intercase_n_3__SRM: Ordered_Create Purchase Order Item_Vendor creates invoice', 'intercase_n_3__SRM: Ordered_SRM: Change was Transmitted_Create Purchase Order Item', 'intercase_n_3__SRM: Ordered_SRM: Change was Transmitted_Record Goods Receipt', 'intercase_n_3__SRM: Ordered_SRM: Change was Transmitted_Record Invoice Receipt', 'intercase_n_3__SRM: Ordered_SRM: Change was Transmitted_SRM: In Transfer to Execution Syst.', 'intercase_n_3__SRM: Ordered_SRM: Change was Transmitted_Vendor creates invoice', 'intercase_n_3__SRM: Ordered_SRM: Deleted_Change Delivery Indicator', 'intercase_n_3__SRM: Ordered_SRM: Deleted_Change Final Invoice Indicator', 'intercase_n_3__SRM: Ordered_SRM: Deleted_Change Price', 'intercase_n_3__SRM: Ordered_SRM: Deleted_Clear Invoice', 'intercase_n_3__SRM: Ordered_SRM: Deleted_Delete Purchase Order Item', 'intercase_n_3__SRM: Ordered_SRM: Deleted_Record Goods Receipt', 'intercase_n_3__SRM: Ordered_SRM: Deleted_Record Invoice Receipt', 'intercase_n_3__SRM: Ordered_SRM: Deleted_Record Service Entry Sheet', 'intercase_n_3__SRM: Ordered_SRM: Deleted_SRM: Complete', 'intercase_n_3__SRM: Ordered_SRM: Deleted_SRM: Created', 'intercase_n_3__SRM: Ordered_SRM: Deleted_SRM: In Transfer to Execution Syst.', 'intercase_n_3__SRM: Ordered_SRM: Deleted_Vendor creates invoice', 'intercase_n_3__SRM: Ordered_SRM: Document Completed_SRM: Change was Transmitted', 'intercase_n_3__SRM: Ordered_SRM: In Transfer to Execution Syst._Create Purchase Order Item', 'intercase_n_3__SRM: Ordered_SRM: In Transfer to Execution Syst._SRM: Change was Transmitted', 'intercase_n_3__SRM: Ordered_SRM: In Transfer to Execution Syst._SRM: Deleted', 'intercase_n_3__SRM: Ordered_Vendor creates invoice_Record Goods Receipt', 'intercase_n_3__SRM: Transaction Completed_Change Delivery Indicator_Record Invoice Receipt', 'intercase_n_3__Set Payment Block_Cancel Invoice Receipt_Clear Invoice', 'intercase_n_3__Set Payment Block_Cancel Subsequent Invoice_Remove Payment Block', 'intercase_n_3__Set Payment Block_Change Approval for Purchase Order_Change Approval for Purchase Order', 'intercase_n_3__Set Payment Block_Change Price_Change Quantity', 'intercase_n_3__Set Payment Block_Clear Invoice_Set Payment Block', 'intercase_n_3__Set Payment Block_Clear Invoice_Vendor creates invoice', 'intercase_n_3__Update Order Confirmation_Change Price_Change Quantity', 'intercase_n_3__Update Order Confirmation_Change Quantity_Change Delivery Indicator', 'intercase_n_3__Update Order Confirmation_Change Quantity_Record Goods Receipt', 'intercase_n_3__Update Order Confirmation_Change Quantity_Record Invoice Receipt', 'intercase_n_3__Update Order Confirmation_Change Quantity_Vendor creates invoice', 'intercase_n_3__Update Order Confirmation_Delete Purchase Order Item_Vendor creates invoice', 'intercase_n_3__Update Order Confirmation_Receive Order Confirmation_Record Goods Receipt', 'intercase_n_3__Update Order Confirmation_Record Goods Receipt_Change Delivery Indicator', 'intercase_n_3__Update Order Confirmation_Record Goods Receipt_Clear Invoice', 'intercase_n_3__Update Order Confirmation_Record Goods Receipt_Record Invoice Receipt', 'intercase_n_3__Update Order Confirmation_Record Goods Receipt_Remove Payment Block', 'intercase_n_3__Update Order Confirmation_Record Goods Receipt_Update Order Confirmation', 'intercase_n_3__Update Order Confirmation_Record Goods Receipt_Vendor creates debit memo', 'intercase_n_3__Update Order Confirmation_Record Goods Receipt_Vendor creates invoice', 'intercase_n_3__Update Order Confirmation_Record Invoice Receipt_Record Goods Receipt', 'intercase_n_3__Update Order Confirmation_Record Invoice Receipt_Remove Payment Block', 'intercase_n_3__Update Order Confirmation_Record Invoice Receipt_Vendor creates invoice', 'intercase_n_3__Update Order Confirmation_Update Order Confirmation_Change Quantity', 'intercase_n_3__Update Order Confirmation_Update Order Confirmation_Record Goods Receipt', 'intercase_n_3__Update Order Confirmation_Update Order Confirmation_Update Order Confirmation', 'intercase_n_3__Update Order Confirmation_Update Order Confirmation_Vendor creates invoice', 'intercase_n_3__Update Order Confirmation_Vendor creates debit memo_Vendor creates invoice', 'intercase_n_3__Update Order Confirmation_Vendor creates invoice_Change Price', 'intercase_n_3__Update Order Confirmation_Vendor creates invoice_Change Quantity', 'intercase_n_3__Update Order Confirmation_Vendor creates invoice_Record Goods Receipt', 'intercase_n_3__Update Order Confirmation_Vendor creates invoice_Record Invoice Receipt', 'intercase_n_3__Update Order Confirmation_Vendor creates invoice_Update Order Confirmation', 'intercase_n_3__Update Order Confirmation_Vendor creates invoice_Vendor creates invoice', 'intercase_n_3__Vendor creates debit memo', 'intercase_n_3__Vendor creates debit memo_Cancel Goods Receipt_Cancel Invoice Receipt', 'intercase_n_3__Vendor creates debit memo_Cancel Goods Receipt_Change Delivery Indicator', 'intercase_n_3__Vendor creates debit memo_Cancel Goods Receipt_Record Goods Receipt', 'intercase_n_3__Vendor creates debit memo_Cancel Invoice Receipt_Cancel Goods Receipt', 'intercase_n_3__Vendor creates debit memo_Cancel Invoice Receipt_Cancel Invoice Receipt', 'intercase_n_3__Vendor creates debit memo_Cancel Invoice Receipt_Change Delivery Indicator', 'intercase_n_3__Vendor creates debit memo_Cancel Invoice Receipt_Change Price', 'intercase_n_3__Vendor creates debit memo_Cancel Invoice Receipt_Change Quantity', 'intercase_n_3__Vendor creates debit memo_Cancel Invoice Receipt_Clear Invoice', 'intercase_n_3__Vendor creates debit memo_Cancel Invoice Receipt_Delete Purchase Order Item', 'intercase_n_3__Vendor creates debit memo_Cancel Invoice Receipt_Record Goods Receipt', 'intercase_n_3__Vendor creates debit memo_Cancel Invoice Receipt_Record Invoice Receipt', 'intercase_n_3__Vendor creates debit memo_Cancel Invoice Receipt_Remove Payment Block', 'intercase_n_3__Vendor creates debit memo_Cancel Invoice Receipt_Vendor creates debit memo', 'intercase_n_3__Vendor creates debit memo_Cancel Invoice Receipt_Vendor creates invoice', 'intercase_n_3__Vendor creates debit memo_Cancel Subsequent Invoice_Cancel Subsequent Invoice', 'intercase_n_3__Vendor creates debit memo_Cancel Subsequent Invoice_Change Price', 'intercase_n_3__Vendor creates debit memo_Cancel Subsequent Invoice_Clear Invoice', 'intercase_n_3__Vendor creates debit memo_Cancel Subsequent Invoice_Record Invoice Receipt', 'intercase_n_3__Vendor creates debit memo_Cancel Subsequent Invoice_Remove Payment Block', 'intercase_n_3__Vendor creates debit memo_Cancel Subsequent Invoice_Vendor creates debit memo', 'intercase_n_3__Vendor creates debit memo_Cancel Subsequent Invoice_Vendor creates invoice', 'intercase_n_3__Vendor creates debit memo_Change Delivery Indicator_Cancel Invoice Receipt', 'intercase_n_3__Vendor creates debit memo_Change Delivery Indicator_Cancel Subsequent Invoice', 'intercase_n_3__Vendor creates debit memo_Change Delivery Indicator_Change Delivery Indicator', 'intercase_n_3__Vendor creates debit memo_Change Delivery Indicator_Clear Invoice', 'intercase_n_3__Vendor creates debit memo_Change Delivery Indicator_Record Goods Receipt', 'intercase_n_3__Vendor creates debit memo_Change Delivery Indicator_Record Invoice Receipt', 'intercase_n_3__Vendor creates debit memo_Change Price_Cancel Invoice Receipt', 'intercase_n_3__Vendor creates debit memo_Change Price_Change Quantity', 'intercase_n_3__Vendor creates debit memo_Change Price_Record Goods Receipt', 'intercase_n_3__Vendor creates debit memo_Change Price_Record Invoice Receipt', 'intercase_n_3__Vendor creates debit memo_Change Price_Remove Payment Block', 'intercase_n_3__Vendor creates debit memo_Change Quantity_Cancel Invoice Receipt', 'intercase_n_3__Vendor creates debit memo_Change Quantity_Change Delivery Indicator', 'intercase_n_3__Vendor creates debit memo_Change Quantity_Change Quantity', 'intercase_n_3__Vendor creates debit memo_Change Quantity_Record Goods Receipt', 'intercase_n_3__Vendor creates debit memo_Change Quantity_Record Invoice Receipt', 'intercase_n_3__Vendor creates debit memo_Change Quantity_Vendor creates debit memo', 'intercase_n_3__Vendor creates debit memo_Change Quantity_Vendor creates invoice', 'intercase_n_3__Vendor creates debit memo_Clear Invoice_Cancel Invoice Receipt', 'intercase_n_3__Vendor creates debit memo_Clear Invoice_Cancel Subsequent Invoice', 'intercase_n_3__Vendor creates debit memo_Clear Invoice_Clear Invoice', 'intercase_n_3__Vendor creates debit memo_Clear Invoice_Record Goods Receipt', 'intercase_n_3__Vendor creates debit memo_Clear Invoice_Record Invoice Receipt', 'intercase_n_3__Vendor creates debit memo_Clear Invoice_Remove Payment Block', 'intercase_n_3__Vendor creates debit memo_Clear Invoice_SRM: Created', 'intercase_n_3__Vendor creates debit memo_Clear Invoice_Vendor creates debit memo', 'intercase_n_3__Vendor creates debit memo_Clear Invoice_Vendor creates invoice', 'intercase_n_3__Vendor creates debit memo_Create Purchase Order Item', 'intercase_n_3__Vendor creates debit memo_Create Purchase Order Item_Cancel Goods Receipt', 'intercase_n_3__Vendor creates debit memo_Create Purchase Order Item_Change Approval for Purchase Order', 'intercase_n_3__Vendor creates debit memo_Create Purchase Order Item_Change Price', 'intercase_n_3__Vendor creates debit memo_Create Purchase Order Item_Change Quantity', 'intercase_n_3__Vendor creates debit memo_Create Purchase Order Item_Receive Order Confirmation', 'intercase_n_3__Vendor creates debit memo_Create Purchase Order Item_Record Goods Receipt', 'intercase_n_3__Vendor creates debit memo_Create Purchase Order Item_Vendor creates invoice', 'intercase_n_3__Vendor creates debit memo_Create Purchase Requisition Item_Create Purchase Order Item', 'intercase_n_3__Vendor creates debit memo_Delete Purchase Order Item_Reactivate Purchase Order Item', 'intercase_n_3__Vendor creates debit memo_Reactivate Purchase Order Item_Change Quantity', 'intercase_n_3__Vendor creates debit memo_Reactivate Purchase Order Item_Record Goods Receipt', 'intercase_n_3__Vendor creates debit memo_Reactivate Purchase Order Item_Record Invoice Receipt', 'intercase_n_3__Vendor creates debit memo_Record Goods Receipt_Cancel Invoice Receipt', 'intercase_n_3__Vendor creates debit memo_Record Goods Receipt_Cancel Subsequent Invoice', 'intercase_n_3__Vendor creates debit memo_Record Goods Receipt_Change Delivery Indicator', 'intercase_n_3__Vendor creates debit memo_Record Goods Receipt_Change Price', 'intercase_n_3__Vendor creates debit memo_Record Goods Receipt_Change Quantity', 'intercase_n_3__Vendor creates debit memo_Record Goods Receipt_Clear Invoice', 'intercase_n_3__Vendor creates debit memo_Record Goods Receipt_Record Goods Receipt', 'intercase_n_3__Vendor creates debit memo_Record Goods Receipt_Record Invoice Receipt', 'intercase_n_3__Vendor creates debit memo_Record Goods Receipt_Record Service Entry Sheet', 'intercase_n_3__Vendor creates debit memo_Record Goods Receipt_Remove Payment Block', 'intercase_n_3__Vendor creates debit memo_Record Goods Receipt_Vendor creates debit memo', 'intercase_n_3__Vendor creates debit memo_Record Goods Receipt_Vendor creates invoice', 'intercase_n_3__Vendor creates debit memo_Record Invoice Receipt_Block Purchase Order Item', 'intercase_n_3__Vendor creates debit memo_Record Invoice Receipt_Cancel Goods Receipt', 'intercase_n_3__Vendor creates debit memo_Record Invoice Receipt_Cancel Invoice Receipt', 'intercase_n_3__Vendor creates debit memo_Record Invoice Receipt_Cancel Subsequent Invoice', 'intercase_n_3__Vendor creates debit memo_Record Invoice Receipt_Change Delivery Indicator', 'intercase_n_3__Vendor creates debit memo_Record Invoice Receipt_Change Price', 'intercase_n_3__Vendor creates debit memo_Record Invoice Receipt_Change Quantity', 'intercase_n_3__Vendor creates debit memo_Record Invoice Receipt_Clear Invoice', 'intercase_n_3__Vendor creates debit memo_Record Invoice Receipt_Record Goods Receipt', 'intercase_n_3__Vendor creates debit memo_Record Invoice Receipt_Record Invoice Receipt', 'intercase_n_3__Vendor creates debit memo_Record Invoice Receipt_Remove Payment Block', 'intercase_n_3__Vendor creates debit memo_Record Invoice Receipt_Set Payment Block', 'intercase_n_3__Vendor creates debit memo_Record Invoice Receipt_Vendor creates debit memo', 'intercase_n_3__Vendor creates debit memo_Record Invoice Receipt_Vendor creates invoice', 'intercase_n_3__Vendor creates debit memo_Record Service Entry Sheet_Record Goods Receipt', 'intercase_n_3__Vendor creates debit memo_Record Service Entry Sheet_Record Service Entry Sheet', 'intercase_n_3__Vendor creates debit memo_Record Subsequent Invoice_Cancel Invoice Receipt', 'intercase_n_3__Vendor creates debit memo_Record Subsequent Invoice_Cancel Subsequent Invoice', 'intercase_n_3__Vendor creates debit memo_Record Subsequent Invoice_Clear Invoice', 'intercase_n_3__Vendor creates debit memo_Remove Payment Block_Cancel Invoice Receipt', 'intercase_n_3__Vendor creates debit memo_Remove Payment Block_Cancel Subsequent Invoice', 'intercase_n_3__Vendor creates debit memo_Remove Payment Block_Change Delivery Indicator', 'intercase_n_3__Vendor creates debit memo_Remove Payment Block_Change Quantity', 'intercase_n_3__Vendor creates debit memo_Remove Payment Block_Clear Invoice', 'intercase_n_3__Vendor creates debit memo_Remove Payment Block_Record Goods Receipt', 'intercase_n_3__Vendor creates debit memo_Remove Payment Block_Record Invoice Receipt', 'intercase_n_3__Vendor creates debit memo_Remove Payment Block_Remove Payment Block', 'intercase_n_3__Vendor creates debit memo_Remove Payment Block_Vendor creates invoice', 'intercase_n_3__Vendor creates debit memo_SRM: Created', 'intercase_n_3__Vendor creates debit memo_SRM: Created_SRM: Complete', 'intercase_n_3__Vendor creates debit memo_Vendor creates debit memo_Cancel Invoice Receipt', 'intercase_n_3__Vendor creates debit memo_Vendor creates debit memo_Cancel Subsequent Invoice', 'intercase_n_3__Vendor creates debit memo_Vendor creates debit memo_Clear Invoice', 'intercase_n_3__Vendor creates debit memo_Vendor creates debit memo_Record Invoice Receipt', 'intercase_n_3__Vendor creates debit memo_Vendor creates debit memo_Remove Payment Block', 'intercase_n_3__Vendor creates debit memo_Vendor creates invoice', 'intercase_n_3__Vendor creates debit memo_Vendor creates invoice_Cancel Goods Receipt', 'intercase_n_3__Vendor creates debit memo_Vendor creates invoice_Cancel Invoice Receipt', 'intercase_n_3__Vendor creates debit memo_Vendor creates invoice_Cancel Subsequent Invoice', 'intercase_n_3__Vendor creates debit memo_Vendor creates invoice_Change Approval for Purchase Order', 'intercase_n_3__Vendor creates debit memo_Vendor creates invoice_Change Delivery Indicator', 'intercase_n_3__Vendor creates debit memo_Vendor creates invoice_Change Price', 'intercase_n_3__Vendor creates debit memo_Vendor creates invoice_Change Quantity', 'intercase_n_3__Vendor creates debit memo_Vendor creates invoice_Clear Invoice', 'intercase_n_3__Vendor creates debit memo_Vendor creates invoice_Create Purchase Order Item', 'intercase_n_3__Vendor creates debit memo_Vendor creates invoice_Create Purchase Requisition Item', 'intercase_n_3__Vendor creates debit memo_Vendor creates invoice_Receive Order Confirmation', 'intercase_n_3__Vendor creates debit memo_Vendor creates invoice_Record Goods Receipt', 'intercase_n_3__Vendor creates debit memo_Vendor creates invoice_Record Invoice Receipt', 'intercase_n_3__Vendor creates debit memo_Vendor creates invoice_Record Service Entry Sheet', 'intercase_n_3__Vendor creates debit memo_Vendor creates invoice_Record Subsequent Invoice', 'intercase_n_3__Vendor creates debit memo_Vendor creates invoice_Remove Payment Block', 'intercase_n_3__Vendor creates debit memo_Vendor creates invoice_SRM: Created', 'intercase_n_3__Vendor creates debit memo_Vendor creates invoice_Update Order Confirmation', 'intercase_n_3__Vendor creates debit memo_Vendor creates invoice_Vendor creates debit memo', 'intercase_n_3__Vendor creates debit memo_Vendor creates invoice_Vendor creates invoice', 'intercase_n_3__Vendor creates invoice', 'intercase_n_3__Vendor creates invoice_Block Purchase Order Item_Change Quantity', 'intercase_n_3__Vendor creates invoice_Block Purchase Order Item_Reactivate Purchase Order Item', 'intercase_n_3__Vendor creates invoice_Cancel Goods Receipt_Cancel Goods Receipt', 'intercase_n_3__Vendor creates invoice_Cancel Goods Receipt_Change Delivery Indicator', 'intercase_n_3__Vendor creates invoice_Cancel Goods Receipt_Change Price', 'intercase_n_3__Vendor creates invoice_Cancel Goods Receipt_Change Quantity', 'intercase_n_3__Vendor creates invoice_Cancel Goods Receipt_Record Goods Receipt', 'intercase_n_3__Vendor creates invoice_Cancel Goods Receipt_Record Invoice Receipt', 'intercase_n_3__Vendor creates invoice_Cancel Goods Receipt_Record Service Entry Sheet', 'intercase_n_3__Vendor creates invoice_Cancel Goods Receipt_Vendor creates debit memo', 'intercase_n_3__Vendor creates invoice_Cancel Invoice Receipt_Cancel Invoice Receipt', 'intercase_n_3__Vendor creates invoice_Cancel Invoice Receipt_Change Delivery Indicator', 'intercase_n_3__Vendor creates invoice_Cancel Invoice Receipt_Clear Invoice', 'intercase_n_3__Vendor creates invoice_Cancel Invoice Receipt_Record Goods Receipt', 'intercase_n_3__Vendor creates invoice_Cancel Invoice Receipt_Record Invoice Receipt', 'intercase_n_3__Vendor creates invoice_Cancel Invoice Receipt_Remove Payment Block', 'intercase_n_3__Vendor creates invoice_Cancel Invoice Receipt_Vendor creates invoice', 'intercase_n_3__Vendor creates invoice_Cancel Subsequent Invoice_Clear Invoice', 'intercase_n_3__Vendor creates invoice_Cancel Subsequent Invoice_Record Goods Receipt', 'intercase_n_3__Vendor creates invoice_Cancel Subsequent Invoice_Vendor creates invoice', 'intercase_n_3__Vendor creates invoice_Change Approval for Purchase Order', 'intercase_n_3__Vendor creates invoice_Change Approval for Purchase Order_Change Approval for Purchase Order', 'intercase_n_3__Vendor creates invoice_Change Approval for Purchase Order_Change Quantity', 'intercase_n_3__Vendor creates invoice_Change Approval for Purchase Order_Create Purchase Order Item', 'intercase_n_3__Vendor creates invoice_Change Approval for Purchase Order_Record Goods Receipt', 'intercase_n_3__Vendor creates invoice_Change Approval for Purchase Order_Record Invoice Receipt', 'intercase_n_3__Vendor creates invoice_Change Approval for Purchase Order_Release Purchase Order', 'intercase_n_3__Vendor creates invoice_Change Delivery Indicator_Cancel Goods Receipt', 'intercase_n_3__Vendor creates invoice_Change Delivery Indicator_Cancel Invoice Receipt', 'intercase_n_3__Vendor creates invoice_Change Delivery Indicator_Change Delivery Indicator', 'intercase_n_3__Vendor creates invoice_Change Delivery Indicator_Change Price', 'intercase_n_3__Vendor creates invoice_Change Delivery Indicator_Change Quantity', 'intercase_n_3__Vendor creates invoice_Change Delivery Indicator_Clear Invoice', 'intercase_n_3__Vendor creates invoice_Change Delivery Indicator_Record Goods Receipt', 'intercase_n_3__Vendor creates invoice_Change Delivery Indicator_Record Invoice Receipt', 'intercase_n_3__Vendor creates invoice_Change Delivery Indicator_Remove Payment Block', 'intercase_n_3__Vendor creates invoice_Change Delivery Indicator_Vendor creates debit memo', 'intercase_n_3__Vendor creates invoice_Change Delivery Indicator_Vendor creates invoice', 'intercase_n_3__Vendor creates invoice_Change Price_Cancel Goods Receipt', 'intercase_n_3__Vendor creates invoice_Change Price_Cancel Invoice Receipt', 'intercase_n_3__Vendor creates invoice_Change Price_Change Delivery Indicator', 'intercase_n_3__Vendor creates invoice_Change Price_Change Price', 'intercase_n_3__Vendor creates invoice_Change Price_Change Quantity', 'intercase_n_3__Vendor creates invoice_Change Price_Change Storage Location', 'intercase_n_3__Vendor creates invoice_Change Price_Receive Order Confirmation', 'intercase_n_3__Vendor creates invoice_Change Price_Record Goods Receipt', 'intercase_n_3__Vendor creates invoice_Change Price_Record Invoice Receipt', 'intercase_n_3__Vendor creates invoice_Change Price_Record Service Entry Sheet', 'intercase_n_3__Vendor creates invoice_Change Price_Remove Payment Block', 'intercase_n_3__Vendor creates invoice_Change Price_Vendor creates debit memo', 'intercase_n_3__Vendor creates invoice_Change Price_Vendor creates invoice', 'intercase_n_3__Vendor creates invoice_Change Quantity_Block Purchase Order Item', 'intercase_n_3__Vendor creates invoice_Change Quantity_Cancel Goods Receipt', 'intercase_n_3__Vendor creates invoice_Change Quantity_Change Approval for Purchase Order', 'intercase_n_3__Vendor creates invoice_Change Quantity_Change Delivery Indicator', 'intercase_n_3__Vendor creates invoice_Change Quantity_Change Price', 'intercase_n_3__Vendor creates invoice_Change Quantity_Change Quantity', 'intercase_n_3__Vendor creates invoice_Change Quantity_Change Storage Location', 'intercase_n_3__Vendor creates invoice_Change Quantity_Clear Invoice', 'intercase_n_3__Vendor creates invoice_Change Quantity_Delete Purchase Order Item', 'intercase_n_3__Vendor creates invoice_Change Quantity_Record Goods Receipt', 'intercase_n_3__Vendor creates invoice_Change Quantity_Record Invoice Receipt', 'intercase_n_3__Vendor creates invoice_Change Quantity_Release Purchase Order', 'intercase_n_3__Vendor creates invoice_Change Quantity_Remove Payment Block', 'intercase_n_3__Vendor creates invoice_Change Quantity_Vendor creates debit memo', 'intercase_n_3__Vendor creates invoice_Change Quantity_Vendor creates invoice', 'intercase_n_3__Vendor creates invoice_Change Storage Location_Change Price', 'intercase_n_3__Vendor creates invoice_Change Storage Location_Change Storage Location', 'intercase_n_3__Vendor creates invoice_Change Storage Location_Delete Purchase Order Item', 'intercase_n_3__Vendor creates invoice_Change Storage Location_Record Goods Receipt', 'intercase_n_3__Vendor creates invoice_Change Storage Location_Record Invoice Receipt', 'intercase_n_3__Vendor creates invoice_Change Storage Location_Vendor creates invoice', 'intercase_n_3__Vendor creates invoice_Clear Invoice_Cancel Goods Receipt', 'intercase_n_3__Vendor creates invoice_Clear Invoice_Cancel Invoice Receipt', 'intercase_n_3__Vendor creates invoice_Clear Invoice_Change Price', 'intercase_n_3__Vendor creates invoice_Clear Invoice_Change Quantity', 'intercase_n_3__Vendor creates invoice_Clear Invoice_Clear Invoice', 'intercase_n_3__Vendor creates invoice_Clear Invoice_Record Goods Receipt', 'intercase_n_3__Vendor creates invoice_Clear Invoice_Record Invoice Receipt', 'intercase_n_3__Vendor creates invoice_Clear Invoice_Record Service Entry Sheet', 'intercase_n_3__Vendor creates invoice_Clear Invoice_Record Subsequent Invoice', 'intercase_n_3__Vendor creates invoice_Clear Invoice_Remove Payment Block', 'intercase_n_3__Vendor creates invoice_Clear Invoice_SRM: Created', 'intercase_n_3__Vendor creates invoice_Clear Invoice_SRM: In Transfer to Execution Syst.', 'intercase_n_3__Vendor creates invoice_Clear Invoice_Vendor creates debit memo', 'intercase_n_3__Vendor creates invoice_Clear Invoice_Vendor creates invoice', 'intercase_n_3__Vendor creates invoice_Create Purchase Order Item', 'intercase_n_3__Vendor creates invoice_Create Purchase Order Item_Cancel Goods Receipt', 'intercase_n_3__Vendor creates invoice_Create Purchase Order Item_Change Approval for Purchase Order', 'intercase_n_3__Vendor creates invoice_Create Purchase Order Item_Change Delivery Indicator', 'intercase_n_3__Vendor creates invoice_Create Purchase Order Item_Change Price', 'intercase_n_3__Vendor creates invoice_Create Purchase Order Item_Change Quantity', 'intercase_n_3__Vendor creates invoice_Create Purchase Order Item_Change Storage Location', 'intercase_n_3__Vendor creates invoice_Create Purchase Order Item_Delete Purchase Order Item', 'intercase_n_3__Vendor creates invoice_Create Purchase Order Item_Receive Order Confirmation', 'intercase_n_3__Vendor creates invoice_Create Purchase Order Item_Record Goods Receipt', 'intercase_n_3__Vendor creates invoice_Create Purchase Order Item_Record Invoice Receipt', 'intercase_n_3__Vendor creates invoice_Create Purchase Order Item_Record Service Entry Sheet', 'intercase_n_3__Vendor creates invoice_Create Purchase Order Item_Release Purchase Order', 'intercase_n_3__Vendor creates invoice_Create Purchase Order Item_Vendor creates debit memo', 'intercase_n_3__Vendor creates invoice_Create Purchase Order Item_Vendor creates invoice', 'intercase_n_3__Vendor creates invoice_Create Purchase Requisition Item', 'intercase_n_3__Vendor creates invoice_Create Purchase Requisition Item_Create Purchase Order Item', 'intercase_n_3__Vendor creates invoice_Delete Purchase Order Item_Change Quantity', 'intercase_n_3__Vendor creates invoice_Delete Purchase Order Item_Reactivate Purchase Order Item', 'intercase_n_3__Vendor creates invoice_Reactivate Purchase Order Item_Block Purchase Order Item', 'intercase_n_3__Vendor creates invoice_Reactivate Purchase Order Item_Change Quantity', 'intercase_n_3__Vendor creates invoice_Reactivate Purchase Order Item_Record Goods Receipt', 'intercase_n_3__Vendor creates invoice_Reactivate Purchase Order Item_Record Invoice Receipt', 'intercase_n_3__Vendor creates invoice_Receive Order Confirmation_Change Price', 'intercase_n_3__Vendor creates invoice_Receive Order Confirmation_Change Quantity', 'intercase_n_3__Vendor creates invoice_Receive Order Confirmation_Record Goods Receipt', 'intercase_n_3__Vendor creates invoice_Receive Order Confirmation_Record Invoice Receipt', 'intercase_n_3__Vendor creates invoice_Receive Order Confirmation_Update Order Confirmation', 'intercase_n_3__Vendor creates invoice_Receive Order Confirmation_Vendor creates invoice', 'intercase_n_3__Vendor creates invoice_Record Goods Receipt_Block Purchase Order Item', 'intercase_n_3__Vendor creates invoice_Record Goods Receipt_Cancel Goods Receipt', 'intercase_n_3__Vendor creates invoice_Record Goods Receipt_Cancel Invoice Receipt', 'intercase_n_3__Vendor creates invoice_Record Goods Receipt_Cancel Subsequent Invoice', 'intercase_n_3__Vendor creates invoice_Record Goods Receipt_Change Approval for Purchase Order', 'intercase_n_3__Vendor creates invoice_Record Goods Receipt_Change Delivery Indicator', 'intercase_n_3__Vendor creates invoice_Record Goods Receipt_Change Price', 'intercase_n_3__Vendor creates invoice_Record Goods Receipt_Change Quantity', 'intercase_n_3__Vendor creates invoice_Record Goods Receipt_Change Storage Location', 'intercase_n_3__Vendor creates invoice_Record Goods Receipt_Clear Invoice', 'intercase_n_3__Vendor creates invoice_Record Goods Receipt_Receive Order Confirmation', 'intercase_n_3__Vendor creates invoice_Record Goods Receipt_Record Goods Receipt', 'intercase_n_3__Vendor creates invoice_Record Goods Receipt_Record Invoice Receipt', 'intercase_n_3__Vendor creates invoice_Record Goods Receipt_Record Service Entry Sheet', 'intercase_n_3__Vendor creates invoice_Record Goods Receipt_Release Purchase Order', 'intercase_n_3__Vendor creates invoice_Record Goods Receipt_Remove Payment Block', 'intercase_n_3__Vendor creates invoice_Record Goods Receipt_Update Order Confirmation', 'intercase_n_3__Vendor creates invoice_Record Goods Receipt_Vendor creates debit memo', 'intercase_n_3__Vendor creates invoice_Record Goods Receipt_Vendor creates invoice', 'intercase_n_3__Vendor creates invoice_Record Invoice Receipt_Block Purchase Order Item', 'intercase_n_3__Vendor creates invoice_Record Invoice Receipt_Cancel Goods Receipt', 'intercase_n_3__Vendor creates invoice_Record Invoice Receipt_Cancel Invoice Receipt', 'intercase_n_3__Vendor creates invoice_Record Invoice Receipt_Cancel Subsequent Invoice', 'intercase_n_3__Vendor creates invoice_Record Invoice Receipt_Change Approval for Purchase Order', 'intercase_n_3__Vendor creates invoice_Record Invoice Receipt_Change Delivery Indicator', 'intercase_n_3__Vendor creates invoice_Record Invoice Receipt_Change Price', 'intercase_n_3__Vendor creates invoice_Record Invoice Receipt_Change Quantity', 'intercase_n_3__Vendor creates invoice_Record Invoice Receipt_Change Storage Location', 'intercase_n_3__Vendor creates invoice_Record Invoice Receipt_Clear Invoice', 'intercase_n_3__Vendor creates invoice_Record Invoice Receipt_Delete Purchase Order Item', 'intercase_n_3__Vendor creates invoice_Record Invoice Receipt_Receive Order Confirmation', 'intercase_n_3__Vendor creates invoice_Record Invoice Receipt_Record Goods Receipt', 'intercase_n_3__Vendor creates invoice_Record Invoice Receipt_Record Invoice Receipt', 'intercase_n_3__Vendor creates invoice_Record Invoice Receipt_Record Service Entry Sheet', 'intercase_n_3__Vendor creates invoice_Record Invoice Receipt_Record Subsequent Invoice', 'intercase_n_3__Vendor creates invoice_Record Invoice Receipt_Release Purchase Order', 'intercase_n_3__Vendor creates invoice_Record Invoice Receipt_Remove Payment Block', 'intercase_n_3__Vendor creates invoice_Record Invoice Receipt_SRM: Created', 'intercase_n_3__Vendor creates invoice_Record Invoice Receipt_Set Payment Block', 'intercase_n_3__Vendor creates invoice_Record Invoice Receipt_Update Order Confirmation', 'intercase_n_3__Vendor creates invoice_Record Invoice Receipt_Vendor creates debit memo', 'intercase_n_3__Vendor creates invoice_Record Invoice Receipt_Vendor creates invoice', 'intercase_n_3__Vendor creates invoice_Record Service Entry Sheet_Change Price', 'intercase_n_3__Vendor creates invoice_Record Service Entry Sheet_Clear Invoice', 'intercase_n_3__Vendor creates invoice_Record Service Entry Sheet_Record Goods Receipt', 'intercase_n_3__Vendor creates invoice_Record Service Entry Sheet_Record Invoice Receipt', 'intercase_n_3__Vendor creates invoice_Record Service Entry Sheet_Record Service Entry Sheet', 'intercase_n_3__Vendor creates invoice_Record Service Entry Sheet_Vendor creates invoice', 'intercase_n_3__Vendor creates invoice_Record Subsequent Invoice_Clear Invoice', 'intercase_n_3__Vendor creates invoice_Record Subsequent Invoice_Record Goods Receipt', 'intercase_n_3__Vendor creates invoice_Record Subsequent Invoice_Remove Payment Block', 'intercase_n_3__Vendor creates invoice_Release Purchase Order_Change Approval for Purchase Order', 'intercase_n_3__Vendor creates invoice_Release Purchase Order_Change Price', 'intercase_n_3__Vendor creates invoice_Release Purchase Order_Change Quantity', 'intercase_n_3__Vendor creates invoice_Release Purchase Order_Create Purchase Order Item', 'intercase_n_3__Vendor creates invoice_Remove Payment Block_Cancel Invoice Receipt', 'intercase_n_3__Vendor creates invoice_Remove Payment Block_Change Quantity', 'intercase_n_3__Vendor creates invoice_Remove Payment Block_Clear Invoice', 'intercase_n_3__Vendor creates invoice_Remove Payment Block_Record Goods Receipt', 'intercase_n_3__Vendor creates invoice_Remove Payment Block_Record Invoice Receipt', 'intercase_n_3__Vendor creates invoice_Remove Payment Block_Record Service Entry Sheet', 'intercase_n_3__Vendor creates invoice_Remove Payment Block_Record Subsequent Invoice', 'intercase_n_3__Vendor creates invoice_Remove Payment Block_Remove Payment Block', 'intercase_n_3__Vendor creates invoice_Remove Payment Block_Vendor creates debit memo', 'intercase_n_3__Vendor creates invoice_Remove Payment Block_Vendor creates invoice', 'intercase_n_3__Vendor creates invoice_SRM: Created', 'intercase_n_3__Vendor creates invoice_SRM: Created_Record Goods Receipt', 'intercase_n_3__Vendor creates invoice_SRM: Created_Record Invoice Receipt', 'intercase_n_3__Vendor creates invoice_SRM: Created_Record Service Entry Sheet', 'intercase_n_3__Vendor creates invoice_SRM: Created_SRM: Complete', 'intercase_n_3__Vendor creates invoice_SRM: Created_SRM: Created', 'intercase_n_3__Vendor creates invoice_SRM: In Transfer to Execution Syst._Change Price', 'intercase_n_3__Vendor creates invoice_SRM: In Transfer to Execution Syst._Create Purchase Order Item', 'intercase_n_3__Vendor creates invoice_SRM: In Transfer to Execution Syst._Record Goods Receipt', 'intercase_n_3__Vendor creates invoice_SRM: In Transfer to Execution Syst._Record Service Entry Sheet', 'intercase_n_3__Vendor creates invoice_SRM: In Transfer to Execution Syst._SRM: Created', 'intercase_n_3__Vendor creates invoice_SRM: In Transfer to Execution Syst._SRM: In Transfer to Execution Syst.', 'intercase_n_3__Vendor creates invoice_SRM: In Transfer to Execution Syst._SRM: Ordered', 'intercase_n_3__Vendor creates invoice_Set Payment Block_Change Approval for Purchase Order', 'intercase_n_3__Vendor creates invoice_Update Order Confirmation_Change Quantity', 'intercase_n_3__Vendor creates invoice_Update Order Confirmation_Receive Order Confirmation', 'intercase_n_3__Vendor creates invoice_Update Order Confirmation_Record Goods Receipt', 'intercase_n_3__Vendor creates invoice_Update Order Confirmation_Record Invoice Receipt', 'intercase_n_3__Vendor creates invoice_Update Order Confirmation_Update Order Confirmation', 'intercase_n_3__Vendor creates invoice_Vendor creates debit memo', 'intercase_n_3__Vendor creates invoice_Vendor creates debit memo_Cancel Goods Receipt', 'intercase_n_3__Vendor creates invoice_Vendor creates debit memo_Cancel Invoice Receipt', 'intercase_n_3__Vendor creates invoice_Vendor creates debit memo_Cancel Subsequent Invoice', 'intercase_n_3__Vendor creates invoice_Vendor creates debit memo_Change Delivery Indicator', 'intercase_n_3__Vendor creates invoice_Vendor creates debit memo_Change Price', 'intercase_n_3__Vendor creates invoice_Vendor creates debit memo_Change Quantity', 'intercase_n_3__Vendor creates invoice_Vendor creates debit memo_Clear Invoice', 'intercase_n_3__Vendor creates invoice_Vendor creates debit memo_Create Purchase Order Item', 'intercase_n_3__Vendor creates invoice_Vendor creates debit memo_Create Purchase Requisition Item', 'intercase_n_3__Vendor creates invoice_Vendor creates debit memo_Delete Purchase Order Item', 'intercase_n_3__Vendor creates invoice_Vendor creates debit memo_Reactivate Purchase Order Item', 'intercase_n_3__Vendor creates invoice_Vendor creates debit memo_Record Goods Receipt', 'intercase_n_3__Vendor creates invoice_Vendor creates debit memo_Record Invoice Receipt', 'intercase_n_3__Vendor creates invoice_Vendor creates debit memo_Record Service Entry Sheet', 'intercase_n_3__Vendor creates invoice_Vendor creates debit memo_Record Subsequent Invoice', 'intercase_n_3__Vendor creates invoice_Vendor creates debit memo_Remove Payment Block', 'intercase_n_3__Vendor creates invoice_Vendor creates debit memo_SRM: Created', 'intercase_n_3__Vendor creates invoice_Vendor creates debit memo_Vendor creates debit memo', 'intercase_n_3__Vendor creates invoice_Vendor creates debit memo_Vendor creates invoice', 'intercase_n_3__Vendor creates invoice_Vendor creates invoice', 'intercase_n_3__Vendor creates invoice_Vendor creates invoice_Block Purchase Order Item', 'intercase_n_3__Vendor creates invoice_Vendor creates invoice_Cancel Goods Receipt', 'intercase_n_3__Vendor creates invoice_Vendor creates invoice_Cancel Invoice Receipt', 'intercase_n_3__Vendor creates invoice_Vendor creates invoice_Change Approval for Purchase Order', 'intercase_n_3__Vendor creates invoice_Vendor creates invoice_Change Price', 'intercase_n_3__Vendor creates invoice_Vendor creates invoice_Change Quantity', 'intercase_n_3__Vendor creates invoice_Vendor creates invoice_Clear Invoice', 'intercase_n_3__Vendor creates invoice_Vendor creates invoice_Create Purchase Order Item', 'intercase_n_3__Vendor creates invoice_Vendor creates invoice_Record Goods Receipt', 'intercase_n_3__Vendor creates invoice_Vendor creates invoice_Record Invoice Receipt', 'intercase_n_3__Vendor creates invoice_Vendor creates invoice_Record Service Entry Sheet', 'intercase_n_3__Vendor creates invoice_Vendor creates invoice_Remove Payment Block', 'intercase_n_3__Vendor creates invoice_Vendor creates invoice_SRM: Created', 'intercase_n_3__Vendor creates invoice_Vendor creates invoice_SRM: In Transfer to Execution Syst.', 'intercase_n_3__Vendor creates invoice_Vendor creates invoice_Vendor creates debit memo', 'intercase_n_3__Vendor creates invoice_Vendor creates invoice_Vendor creates invoice']

In [3]:
N = 1000
import conduct_evaluation
get_pscores = lambda likelihoods : [pscore(likelihoods[1][i], likelihoods[2][k][3]).compute()[0] for i, k in enumerate(list(likelihoods[0].keys()))]

In [4]:
drbart_model = DRBART(parser_dir = '../../../models/standard_numerical_stable/'+model_name+'/crsdar_ii3/',
                     strict_parser=False)
evaluator = conduct_evaluation.ConductEvaluation(drbart_model, SampleOutcomes_DRBART_Normal_A_R_S_D_AC_RC_II,
                                                   {
                                                        'activity_key' : 'concept:name_start',
                                                        'resource_key' : 'org:resource_start',
                                                        'known_resources' : known_resources,
                                                        'known_activities' : known_activities,
                                                        'inter_instance_column_names' : ii3
                                                    },
                                     test_event_log, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods = evaluator.sample_cases(False, True, store_case_data=False)

  0%|                                                 | 0/49819 [00:00<?, ?it/s]

  0%|                                                 | 0/49819 [00:20<?, ?it/s]

  0%|                             | 1/49819 [3:48:08<189424:30:35, 13688.39s/it]

  1%|▎                                | 401/49819 [8:45:13<938:55:27, 68.40s/it]

 30%|█████████▎                     | 14901/49819 [10:43:48<17:15:31,  1.78s/it]

...

 98%|████████████████████████████████████████████████████████████████████████████████████████▍ | 48976/49819 [00:55<00:00, 1270.64it/s]

 99%|█████████████████████████████████████████████████████████████████████████████████████████▏| 49351/49819 [00:55<00:00, 1693.69it/s]

100%|█████████████████████████████████████████████████████████████████████████████████████████▊| 49726/49819 [00:55<00:00, 1767.96it/s]

100%|███████████████████████████████████████████████████████████████████████████████████████████| 49819/49819 [00:55<00:00, 895.12it/s]

Error in _get_kde_from_samples: [<class 'decimal.DivisionByZero'>]
trying with eps
Error in _get_kde_from_samples: [<class 'decimal.DivisionByZero'>]
trying with eps
Error in _get_kde_from_samples: [<class 'decimal.DivisionByZero'>]
trying with eps
Error in _get_kde_from_samples: [<class 'decimal.DivisionByZero'>]
trying with eps
Error in _get_kde_from_samples: [<class 'decimal.DivisionByZero'>]
trying with eps
Error in _get_kde_from_samples: [<class 'decimal.DivisionByZero'>]
trying with eps
Error in _get_kde_from_samples: [<class 'decimal.DivisionByZero'>]
trying with eps
Error in _get_kde_from_samples: [<class 'decimal.DivisionByZero'>]
trying with eps
Error in _get_kde_from_samples: [<class 'decimal.DivisionByZero'>]
trying with eps
Error in _get_kde_from_samples: [<class 'decimal.DivisionByZero'>]
trying with eps
Error in _get_kde_from_samples: [<class 'decimal.DivisionByZero'>]
trying with eps
Error in _get_kde_from_samples: [<class 'decimal.DivisionByZero'>]
trying with eps
Erro

In [5]:
np.mean([v.ln() for v in likelihoods[0].values()])

Decimal('-Infinity')

In [6]:
np.mean(get_pscores(likelihoods))

np.float64(10281266.13529974)